# OFV-kontroller - samlet Databricks-notebook

Python/Databricks-port av `OFV_RefreshInfo.bas`. Kjorer de tre
kontrollene og samler input og alle resultater i EN Excel-arbeidsbok
(+ en interaktiv HTML-rapport), i stedet for tre separate VBA-ark i samme
arbeidsbok.

## Hva kontrollene gjor
- **Kontroll solgte biler**: for hver bil hentes hele OFV-
  transaksjonshistorikken. Den transaksjonen som ligger **naermest**
  bokfort dato (for eller etter, likegyldig) brukes som
  kontrollgrunnlag. Har OFV ingen transaksjoner for bilen i det hele
  tatt, brukes forstegangsregistreringsdato fra Statens vegvesen
  (SVV) i stedet. Er "OrgNrSelger" fylt ut i input, flagges det
  ekstra ("Selvhandel") hvis samme selskap star som bade selger og
  kjoper i den matchede transaksjonen.
- **Varekjop bruktbil**: tre lister - Innkjop (i perioden), IB (kjopt
  forrige periode) og UB (fortsatt pa lager). For hver bil i Innkjop
  og IB beregnes "lagerperioden" (siste kjop av oppgitt
  juridisk enhet, og forste videresalg etter det kjopet) - OK sa
  lenge lagerperioden overlapper kontrollperioden. I tillegg hentes
  full historikk for biler OFV sier er kjopt av selskapet i perioden,
  men som IKKE star i Innkjop-listen ("mangler i bokforing").
- **Kontroll Demobil**: for hver bil sjekkes om bilens NYESTE
  registrerte OFV-transaksjon fortsatt har oppgitt juridisk enhet som
  kjoper (OK), eller om bilen er videreselgt (Avvik).

## Mappestruktur per selskap
Selskapsnavn settes via widgeten `SELSKAP` og styrer alle stier:
- `RåData/<Selskap>/` - ra kildefiler
- `Input/<Selskap>/` - ferdig formaterte inputfiler
- `Output/<Selskap>/` - resultater (Excel + HTML)

## Input - tre separate Excel-filer
Hver kontroll har sin EGEN inputfil, med faste filnavn i
`Input/<Selskap>/` (se Celle 1). Hver fil har to faner:

- **"Konfig"**: to kolonner, `Felt` og `Verdi` - en rad per
  innstilling (orgnr/datoer).
- **"Biler"**: en rad per bil/kjoretoy, med kolonnen `Identifikator`
  (regnr ELLER VIN - bare en av dem per rad, koden kjenner dem
  automatisk fra hverandre pa lengde: VIN er alltid 17 tegn) pluss en
  kontroll-spesifikk kolonne.

| Fil | Konfig-felt | Biler-kolonner |
|---|---|---|
| `kontroll_solgte_biler_input.xlsx` | `OrgNrSelger` (valgfri) | `Identifikator`, `BokfortDato` |
| `kontroll_varekjop_bruktbil_input.xlsx` | `OrgNrKjoper`, `DatoFra`, `DatoTil` | `Identifikator`, `Liste` (`Innkjop`/`IB`/`UB`) |
| `kontroll_demobil_input.xlsx` | `OrgNr`, `DatoFra` (valgfri), `DatoTil` (valgfri) | `Identifikator`, `BokfortInnDato` |

Datoer kan skrives som ekte Excel-datoer, som tekst `dd.mm.aaaa` eller
som ISO-tekst `aaaa-mm-dd`. Mangler en inputfil (01_Verifisering_Input
fant ingen rafil for kontrollen), hoppes den kontrollen over.

## Output
EN samlet arbeidsbok `OFV_Kontroller_<tidsstempel>.xlsx` i
`OUTPUT_DIR`, med fanene:

1. **Input** - oversikt over hvilken kildefil/konfigurasjon/liste med
   biler som ble brukt for hver av de tre kontrollene.
2. **Resultat solgte biler** / **Kontroll solgte biler**
3. **Resultat Varekjop** / **Kontroll Varekjop Bruktbil**
4. **Resultat Demobil** / **Kontroll Demobil**

I tillegg en selvstendig HTML-rapport (`OFV_Kontroller_<tidsstempel>.html`,
virker uten nett) i BDO-presentasjonsstil: forst presentasjonen "Kontroll
solgte biler" med faste eksempeldata, deretter rapportsidene **Oversikt**,
**Per forhandler**, **Solgte biler**, **Varekjop bruktbil**, **Demobil**
og **Analyse**, og til slutt **Sporsmal**. Alle tabeller kan sorteres,
sokes i og filtreres pa hver kolonne, og filtrert utvalg kan lastes ned
som CSV. Via tannhjulet kan flere datasett fra Kontroll solgte biler
limes inn eller lastes opp - de vises sammen med Databricks-kjoringen.
Excel-leseren og fontene bygges inn fra `Ressurser/` (se Celle 1).

Fargene er BDOs egen profilpalett (se Celle 1) i stedet for Excels
innebygde gront/gult/rodt - BDO bla er forbeholdt logoen og brukes
derfor ikke her.

## Forbedringer i forhold til VBA-versjonen
- JSON bygges/parses med `json`/`requests` i stedet for en
  handrullet tegn-for-tegn-parser.
- Retry bruker ekte eksponentiell backoff (+ jitter) i stedet for
  VBA sin lineaere "ventetid * forsoksnummer".
- SelgerType/KjoperType regnes direkte i Python (samme regel som
  Excel-formelen i VBA-versjonen), ikke som en live Excel-formel.
- API-nokler hentes fra Databricks secrets, aldri hardkodet.
- Samme regnr/VIN slas bare opp en gang per kjoring (hurtigbuffer) -
  resultatet er identisk, men med farre API-kall.
- Orgnr normaliseres (`normalize_orgnr`) for de sammenlignes, sa
  `933749312.0` fra Excel matcher `933749312` fra OFV.

In [ ]:
%pip install openpyxl -q

In [ ]:
# ==================================================================
# CELLE 1: KONFIGURASJON
# ==================================================================

import base64
import html
import json
import os
import random
import re
import time
from collections import Counter
from datetime import date, datetime, timedelta

import pandas as pd
import requests
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# ------------------------------------------------------------------
# Selskapsnavn - styrer alle undermapper
# ------------------------------------------------------------------
dbutils.widgets.text("SELSKAP", "", "Selskapsnavn (mappenavn)")
SELSKAP = dbutils.widgets.get("SELSKAP").strip()
if not SELSKAP:
    raise ValueError("SELSKAP-widgeten ma fylles ut (f.eks. 'Toyota Bilia Oslo AS').")

print(f"Selskap: {SELSKAP}")

# ------------------------------------------------------------------
# Workspace-stier - per selskap
# ------------------------------------------------------------------
_BASE = "/Workspace/Users/stefan.luidold@bdo.no/OFV API"

INPUT_DIR = os.path.join(_BASE, "Input", SELSKAP)
OUTPUT_DIR = os.path.join(_BASE, "Output", SELSKAP)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input:   {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")

INPUT_FILE_SOLGTE_BILER = os.path.join(INPUT_DIR, "kontroll_solgte_biler_input.xlsx")
INPUT_FILE_VAREKJOP = os.path.join(INPUT_DIR, "kontroll_varekjop_bruktbil_input.xlsx")
INPUT_FILE_DEMOBIL = os.path.join(INPUT_DIR, "kontroll_demobil_input.xlsx")

# Felles ressurser for HTML-rapporten (Excel-leser og fonter som bygges
# inn i filen). Last opp filene fra repoets OFV_Databricks/ressurser hit -
# mangler de, forsoker Celle 5 a laste dem ned og lagre dem her.
RESSURS_DIR = os.path.join(_BASE, "Ressurser")

# Hvilke kontroller som kjores styres av hvilke inputfiler som finnes -
# 01_Verifisering_Input hopper over kontroller uten matchende rafil, og
# da hoppes de over her ogsa (i stedet for at hele notebooken krasjer).
KJOR_KONTROLL = {
    "solgte_biler": os.path.exists(INPUT_FILE_SOLGTE_BILER),
    "varekjop_bruktbil": os.path.exists(INPUT_FILE_VAREKJOP),
    "demobil": os.path.exists(INPUT_FILE_DEMOBIL),
}

print("\nInputfiler:")
for _navn, _sti in [("solgte_biler", INPUT_FILE_SOLGTE_BILER),
                    ("varekjop_bruktbil", INPUT_FILE_VAREKJOP),
                    ("demobil", INPUT_FILE_DEMOBIL)]:
    print(f"  {'OK     ' if KJOR_KONTROLL[_navn] else 'MANGLER'}  {_navn}: {os.path.basename(_sti)}")

if not any(KJOR_KONTROLL.values()):
    raise FileNotFoundError(
        f"Fant ingen inputfiler i {INPUT_DIR}. Kjor 01_Verifisering_Input "
        f"for selskapet '{SELSKAP}' forst."
    )

_TIDSSTEMPEL = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_XLSX = os.path.join(OUTPUT_DIR, f"OFV_Kontroller_{_TIDSSTEMPEL}.xlsx")
OUTPUT_HTML = os.path.join(OUTPUT_DIR, f"OFV_Kontroller_{_TIDSSTEMPEL}.html")

# ------------------------------------------------------------------
# API-endepunkter og retry-oppsett
# ------------------------------------------------------------------

OFV_URL = "https://api.ofv.no/transactions/v1/"
SVV_URL = "https://akfell-datautlevering.atlas.vegvesen.no/enkeltoppslag/kjoretoydata"

MAX_RETRIES = 4
RETRY_BASE_S = 2.0       # eksponentiell backoff: ca 2, 4, 8, 16 sek (+ jitter)
RETRY_JITTER_S = 0.5
API_PAUSE_S = 0.15       # pause mellom hvert kall, for a vaere en god API-borger
HTTP_TIMEOUT_S = 30

RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

# ------------------------------------------------------------------
# Secrets - API-nokler hentes ALDRI hardkodet
# ------------------------------------------------------------------

SECRET_SCOPE = "stefan-api-keys"

try:
    OFV_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key="ofv-api-key")
except Exception as exc:
    raise RuntimeError(
        f"Fant ikke OFV API-nokkel i secret scope '{SECRET_SCOPE}' "
        f"(key 'ofv-api-key'). Opprett secreten med "
        f"'databricks secrets put --scope {SECRET_SCOPE} --key ofv-api-key'. "
        f"({exc})"
    )

try:
    SVV_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key="vegvesen-api-key")
except Exception:
    # SVV er kun en reserve for forstegangsregistrering (Kontroll solgte
    # biler) nar OFV ikke har noen transaksjoner for kjoretoyet - mangler
    # nokkelen, hoppes SVV-oppslaget bare over, akkurat som i VBA.
    SVV_API_KEY = None

print(f"\nSVV-reserve: {'aktiv' if SVV_API_KEY else 'av (mangler vegvesen-api-key)'}")

# ------------------------------------------------------------------
# BDOs profilpalett (se cloud.brandmaster.com/point/no/bdobc) - BDO
# Bla (#22409A) er forbeholdt logoen og brukes IKKE i rapporten.
# ------------------------------------------------------------------

BDO = {
    "red": "E81A3B",
    "charcoal": "333333",
    "slate": "5B6E7F",
    "burgundy": "98002E",
    "pale_charcoal": "E7E7E7",
    "jade": "009966",
    "ocean": "008FD2",
    "gold": "D67900",
    "white": "FFFFFF",
}

# Semantisk bruk - samme roller som fargekodene i VBA-versjonen
FARGE_HEADER_BG = BDO["charcoal"]
FARGE_HEADER_FONT = BDO["white"]
FARGE_SEKSJON_BG = BDO["pale_charcoal"]
FARGE_SEKSJON_FONT = BDO["charcoal"]
FARGE_OK_BG = BDO["jade"]
FARGE_OK_FONT = BDO["white"]
FARGE_AVVIK_BG = BDO["red"]
FARGE_AVVIK_FONT = BDO["white"]
FARGE_VARSEL_BG = BDO["gold"]
FARGE_VARSEL_FONT = BDO["white"]
FARGE_GRUPPE_BG = "F4F4F4"
FARGE_KANT = BDO["slate"]

# Grenser for fargekoding av "Dager avvik" i Kontroll solgte biler:
AVVIK_GRONN_MAX = 2
AVVIK_GUL_MAX = 14

# Felt som regnes som "bil-nivaa"
CAR_LEVEL_FIELDS = {
    "ChassisNumber", "MakeName", "ModelName", "FuelGroup",
    "IsLeased", "IsUsedImported", "FirstRegistrationDate",
}

# 24-kolonners feltkart for Resultat-tabellene
FIELD_MAP = [
    ("Input", "Input", False),
    ("Kilde", "Kilde", False),
    ("RegNo", "RegNo", False),
    ("ChassisNumber", "Chassisnummer", False),
    ("MakeName", "Merke", False),
    ("ModelName", "Modell", False),
    ("RegistrationType", "RegistreringsType", False),
    ("FuelGroup", "Drivstoffgruppe", False),
    ("IsLeased", "Leaset", False),
    ("IsUsedImported", "Bruktimportert", False),
    ("FirstRegistrationDate", "ForstegangsRegistrering", True),
    ("TransactionNumber", "TransaksjonsNummer", False),
    ("TransactionDate", "Eierskiftedato", True),
    ("SelgerType", "SelgerType", False),
    ("KjoperType", "KjoperType", False),
    ("FromOwnerType", "SelgerEierType", False),
    ("FromOwnerCompanyName", "SelgerEierFirma", False),
    ("FromOwnerOrgNo", "SelgerOrgNr", False),
    ("FromOwnerCounty", "SelgerEierFylke", False),
    ("ToOwnerType", "KjoperEierType", False),
    ("ToOwnerCompanyName", "KjoperEierFirma", False),
    ("ToOwnerOrgNo", "KjoperOrgNr", False),
    ("ToOwnerCounty", "KjoperEierFylke", False),
    ("Status", "Status", False),
]

In [ ]:
# ==================================================================
# CELLE 2: HJELPEFUNKSJONER
# ==================================================================

# ------------------------------------------------------------------
# Tekst, dato og identifikatorer
# ------------------------------------------------------------------


def _er_tom(value) -> bool:
    """None, NaN/NaT og tomme strenger regnes som tomme."""
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass
    return isinstance(value, str) and not value.strip()


def normalize_identifier(value) -> str:
    if _er_tom(value):
        return ""
    return str(value).strip().upper().replace(" ", "")


def normalize_orgnr(value) -> str:
    """Organisasjonsnummer som ren sifferstreng.

    pandas leser Konfig-kolonnen "Verdi" som float nar de andre cellene i
    kolonnen er tomme (f.eks. Demobil uten DatoFra/DatoTil), slik at
    933749312 blir 933749312.0. Uten denne normaliseringen ble orgnr
    sammenlignet som "933749312.0" mot OFV sitt "933749312", og ALLE
    demobiler ble feilaktig flagget som avvik. Haandterer ogsa NaN (tom
    celle -> ""), mellomrom ("933 749 312") og "NO ... MVA"."""
    if _er_tom(value):
        return ""
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    text = str(value).strip()
    if re.fullmatch(r"\d+\.0+", text):
        text = text.split(".")[0]
    return re.sub(r"\D", "", text)


def is_vin(text: str) -> bool:
    """VIN (chassisnummer) er alltid noyaktig 17 tegn per ISO 3779."""
    return len(text) == 17


def split_identifikator(raw) -> tuple:
    """Deler en raw Identifikator-celle inn i (regno, vin)."""
    verdi = normalize_identifier(raw)
    if not verdi:
        return "", ""
    if is_vin(verdi):
        return "", verdi
    return verdi, ""


def build_vehicle_key(vin: str, regno: str) -> str:
    if vin:
        return f"VIN|{vin}"
    if regno:
        return f"REG|{regno}"
    return ""


def parse_bokfort_dato(raw):
    """Tolker en dato skrevet inn av en bruker (Excel-celle)."""
    if _er_tom(raw):
        return None
    if isinstance(raw, pd.Timestamp):
        return raw.date()
    if isinstance(raw, datetime):
        return raw.date()
    if isinstance(raw, date):
        return raw

    text = str(raw).strip()
    if not text:
        return None

    # ISO-format (aaaa-mm-dd), f.eks. fra en tekstcelle eller widget
    iso = re.fullmatch(r"(\d{4})-(\d{1,2})-(\d{1,2})(?:[ T].*)?", text)
    if iso:
        try:
            return date(int(iso.group(1)), int(iso.group(2)), int(iso.group(3)))
        except ValueError:
            return None

    text = text.replace("/", ".").replace("-", ".")
    parts = text.split(".")
    if len(parts) != 3:
        return None

    try:
        dag, maned, ar = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return None

    if ar < 100:
        ar += 2000

    if not (1 <= maned <= 12 and 1 <= dag <= 31 and 1900 <= ar <= 2100):
        return None

    try:
        return date(ar, maned, dag)
    except ValueError:
        return None


def date_from_iso(iso_value):
    """Parser OFV/SVV sine ISO-formatterte datoer."""
    if not iso_value:
        return None
    s = str(iso_value)
    if len(s) < 10:
        return None
    try:
        return date(int(s[0:4]), int(s[5:7]), int(s[8:10]))
    except ValueError:
        return None


def find_any(obj, key):
    """Sok rekursivt etter forste forekomst av `key` i et nestet dict/list-tre."""
    if isinstance(obj, dict):
        if key in obj and obj[key] not in (None, ""):
            return obj[key]
        for v in obj.values():
            found = find_any(v, key)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = find_any(item, key)
            if found is not None:
                return found
    return None


def compute_owner_label(owner_type, company_name) -> str:
    owner_type = owner_type or ""
    company_name = company_name or ""
    if not owner_type and not company_name:
        return ""
    if owner_type == "Privat":
        return "Privat"
    return company_name or owner_type


# ------------------------------------------------------------------
# HTTP - OFV og SVV, med eksponentiell backoff
# ------------------------------------------------------------------

_SESSION = requests.Session()


def _backoff_sleep(attempt: int) -> None:
    wait = RETRY_BASE_S * (2 ** (attempt - 1)) + random.uniform(0, RETRY_JITTER_S)
    time.sleep(wait)


def post_ofv_with_retries(api_key: str, body: dict):
    """POST mot OFV med eksponentiell backoff-retry pa 429/5xx."""
    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        API_TELLER["ofv_http"] += 1
        try:
            resp = _SESSION.post(
                OFV_URL,
                headers={
                    "Content-Type": "application/json",
                    "Cache-Control": "no-cache",
                    "Ocp-Apim-Subscription-Key": api_key,
                },
                json=body,
                timeout=HTTP_TIMEOUT_S,
            )
        except requests.RequestException as exc:
            last_error = str(exc)
            _backoff_sleep(attempt)
            continue

        if resp.status_code == 200:
            return resp.json(), "OK"
        if resp.status_code == 401:
            return None, "Feil: OFV 401 - kontroller API-nokkelen"
        if resp.status_code == 403:
            return None, "Feil: OFV 403 - tilgang eller kvote"
        if resp.status_code == 404:
            return None, "Feil: OFV-endepunktet ble ikke funnet"
        if resp.status_code in RETRYABLE_STATUS_CODES:
            last_error = f"{resp.status_code}: {resp.text[:200]}"
            _backoff_sleep(attempt)
            continue
        return None, f"Feil: OFV HTTP {resp.status_code}"
    return None, f"Feil: OFV - {last_error}"


def get_svv_response(api_key: str, filter_name: str, identifier: str):
    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        API_TELLER["svv_http"] += 1
        try:
            resp = _SESSION.get(
                SVV_URL,
                params={filter_name: identifier},
                headers={
                    "SVV-Authorization": f"Apikey {api_key}",
                    "Accept": "application/json",
                },
                timeout=HTTP_TIMEOUT_S,
            )
        except requests.RequestException as exc:
            last_error = str(exc)
            _backoff_sleep(attempt)
            continue

        if resp.status_code == 200:
            return resp.json(), "Statens vegvesen"
        if resp.status_code == 401:
            return None, "Feil: Vegvesenet 401 - kontroller API-nokkel"
        if resp.status_code == 403:
            return None, "Feil: Vegvesenet 403 - tilgang eller kvote"
        if resp.status_code == 404:
            return None, "Statens vegvesen - ingen treff"
        if resp.status_code in RETRYABLE_STATUS_CODES:
            last_error = f"{resp.status_code}: {resp.text[:200]}"
            _backoff_sleep(attempt)
            continue
        return None, f"Feil: Vegvesenet HTTP {resp.status_code}"
    return None, f"Feil: Vegvesenet - {last_error}"


def fetch_vehicle_info_from_svv(api_key: str, regno: str, vin: str) -> dict:
    """Kalles kun nar OFV ikke har noen transaksjoner for kjoretoyet.
    Buffres per kjoretoy pa samme mate som OFV-oppslagene."""
    key = (regno, vin)
    if key in _SVV_CACHE:
        return dict(_SVV_CACHE[key])
    result = _fetch_vehicle_info_from_svv_uncached(api_key, regno, vin)
    if not str(result.get("Status") or "").startswith("Feil:"):
        _SVV_CACHE[key] = dict(result)
    return result


def _fetch_vehicle_info_from_svv_uncached(api_key: str, regno: str, vin: str) -> dict:
    result = {"Status": "Statens vegvesen - ingen dato"}
    data = None
    status = None
    if regno:
        data, status = get_svv_response(api_key, "kjennemerke", regno)
    if data is None and vin:
        data, status = get_svv_response(api_key, "understellsnummer", vin)
    if data is None:
        result["Status"] = status or "Statens vegvesen - ingen dato"
        return result
    iso_date = (
        find_any(data, "registrertForstegangNorgeDato")
        or find_any(data, "registrertForstegangDato")
    )
    d = date_from_iso(iso_date) if iso_date else None
    if d:
        result["FirstRegistrationDate"] = d
        result["Status"] = "Statens vegvesen"
    return result


# ------------------------------------------------------------------
# OFV-transaksjoner - bygging av rader + de to API-kallene
# ------------------------------------------------------------------


def build_transaction_row(identifier, tx: dict, use_vin: bool,
                           original_regno: str, original_vin: str) -> dict:
    from_side = tx.get("from") or {}
    to_side = tx.get("to") or {}
    from_owner = from_side.get("owner") or {}
    to_owner = to_side.get("owner") or {}
    from_company = from_owner.get("companyInfo") or {}
    to_company = to_owner.get("companyInfo") or {}
    from_type = from_owner.get("type") or ""
    from_company_name = from_company.get("name") or ""
    to_type = to_owner.get("type") or ""
    to_company_name = to_company.get("name") or ""

    return {
        "Input": identifier,
        "Kilde": "VIN" if use_vin else "Regnr",
        "RegNo": tx.get("regNo") or original_regno,
        "ChassisNumber": tx.get("chassisNumber") or original_vin,
        "MakeName": tx.get("makeName"),
        "ModelName": tx.get("modelName"),
        "RegistrationType": tx.get("registrationType"),
        "FuelGroup": tx.get("fuelGroup"),
        "IsLeased": tx.get("isLeased"),
        "IsUsedImported": tx.get("isUsedImported"),
        "FirstRegistrationDate": date_from_iso(tx.get("firstRegistrationDate")),
        "TransactionNumber": tx.get("transactionNumber"),
        "TransactionDate": date_from_iso(tx.get("transactionDate")),
        "SelgerType": compute_owner_label(from_type, from_company_name),
        "KjoperType": compute_owner_label(to_type, to_company_name),
        "FromOwnerType": from_type,
        "FromOwnerCompanyName": from_company_name,
        "FromOwnerOrgNo": from_company.get("organizationNumber"),
        "FromOwnerCounty": from_owner.get("countyName"),
        "ToOwnerType": to_type,
        "ToOwnerCompanyName": to_company_name,
        "ToOwnerOrgNo": to_company.get("organizationNumber"),
        "ToOwnerCounty": to_owner.get("countyName"),
        "Status": "OK",
    }


def build_empty_transaction_row(identifier, use_vin: bool,
                                 original_regno: str, original_vin: str,
                                 status_text: str) -> dict:
    return {
        "Input": identifier,
        "Kilde": "VIN" if use_vin else "Regnr",
        "RegNo": original_regno,
        "ChassisNumber": original_vin,
        "Status": status_text,
    }


# Hurtigbuffer per kjoring: samme regnr/VIN star ofte flere ganger i
# input (f.eks. en bil som er solgt flere ganger), og OFV-historikken er
# den samme hver gang. Buffret tommes ved starten av hver kjoring (Celle
# 3), og feilede kall buffres aldri - de forsokes pa nytt neste gang.
_OFV_CACHE = {}
_SVV_CACHE = {}
API_TELLER = Counter()


def fetch_ofv_transactions(api_key: str, identifier: str, use_vin: bool,
                            original_regno: str, original_vin: str) -> list:
    """Som _fetch_ofv_transactions_uncached, men med hurtigbuffer per
    kjoretoy. Returnerer en NY liste med KOPIER av radene hver gang, sa
    resultatet er identisk med et ferskt API-kall."""
    key = ("VIN" if use_vin else "REG", normalize_identifier(identifier),
           original_regno, original_vin)
    if key in _OFV_CACHE:
        API_TELLER["ofv_buffer"] += 1
        return [dict(r) for r in _OFV_CACHE[key]]
    rows = _fetch_ofv_transactions_uncached(
        api_key, identifier, use_vin, original_regno, original_vin)
    API_TELLER["ofv_kjoretoy"] += 1
    if not any(str(r.get("Status") or "").startswith("Feil:") for r in rows):
        _OFV_CACHE[key] = [dict(r) for r in rows]
    return rows


def _fetch_ofv_transactions_uncached(api_key: str, identifier: str, use_vin: bool,
                                      original_regno: str, original_vin: str) -> list:
    """Henter ALLE transaksjoner for kjoretoyet - ett kall per kjoretoy."""
    filter_key = "chassisNumber" if use_vin else "regNo"
    rows = []
    cursor = None
    while True:
        pagination = {"first": 1000}
        if cursor:
            pagination["cursor"] = cursor
        body = {
            "filters": {filter_key: identifier},
            "pagination": pagination,
            "sorting": {"orderBy": "transactionDate", "orderDirection": "DESC"},
        }
        data, status = post_ofv_with_retries(api_key, body)
        if status != "OK":
            rows.append(build_empty_transaction_row(
                identifier, use_vin, original_regno, original_vin, status))
            return rows
        for tx in (data or {}).get("transactions", []):
            rows.append(build_transaction_row(
                identifier, tx, use_vin, original_regno, original_vin))
        pg = (data or {}).get("pagination", {})
        if pg.get("hasNextPage"):
            cursor = pg.get("endCursor")
            if not cursor:
                break
            time.sleep(API_PAUSE_S)
        else:
            break
    if not rows:
        rows.append(build_empty_transaction_row(
            identifier, use_vin, original_regno, original_vin,
            "Ingen eierskifter funnet"))
    return rows


def fetch_ofv_transactions_by_buyer_org(api_key: str, org_no: str,
                                         date_fra: date, date_til: date) -> list:
    """Henter ALLE OFV-transaksjoner der gitt orgnr star som KJOPER."""
    rows = []
    cursor = None
    while True:
        pagination = {"first": 1000}
        if cursor:
            pagination["cursor"] = cursor
        body = {
            "filters": {
                "toOrganizationNumber": org_no,
                "transactionDateFrom": date_fra.strftime("%Y-%m-%d"),
                "transactionDateTo": date_til.strftime("%Y-%m-%d"),
            },
            "pagination": pagination,
            "sorting": {"orderBy": "transactionDate", "orderDirection": "DESC"},
        }
        data, status = post_ofv_with_retries(api_key, body)
        if status != "OK":
            raise RuntimeError(f"FetchOFVTransactionsByBuyerOrg: {status}")
        for tx in (data or {}).get("transactions", []):
            rows.append(build_transaction_row("", tx, False, "", ""))
        pg = (data or {}).get("pagination", {})
        if pg.get("hasNextPage"):
            cursor = pg.get("endCursor")
            if not cursor:
                break
            time.sleep(API_PAUSE_S)
        else:
            break
    return rows


# ------------------------------------------------------------------
# Input-lesere
# ------------------------------------------------------------------


def read_konfig(path: str) -> dict:
    df = pd.read_excel(path, sheet_name="Konfig")
    konfig = {}
    for _, row in df.iterrows():
        felt = str(row.get("Felt", "")).strip()
        if felt and felt.lower() != "nan":
            verdi = row.get("Verdi")
            konfig[felt] = None if _er_tom(verdi) else verdi
    return konfig


def read_biler(path: str, extra_col: str) -> list:
    df = pd.read_excel(path, sheet_name="Biler")
    out = []
    for _, row in df.iterrows():
        raw_ident = row.get("Identifikator")
        if raw_ident is None or (isinstance(raw_ident, float) and pd.isna(raw_ident)):
            continue
        regno, vin = split_identifikator(raw_ident)
        if not regno and not vin:
            continue
        out.append({
            "regno": regno,
            "vin": vin,
            "identifikator": vin or regno,
            extra_col: row.get(extra_col) if extra_col in df.columns else None,
        })
    return out


# ------------------------------------------------------------------
# Resultat-tabell - delt av alle tre kontroller
# ------------------------------------------------------------------


def build_resultat_rows(all_rows: list) -> list:
    records = []
    forrige_input = None
    for row in all_rows:
        denne_input = row.get("Input")
        er_forste = (denne_input != forrige_input)
        forrige_input = denne_input
        record = {"_ErForsteIGruppe": er_forste, "_Input": denne_input}
        for key, header, _is_date in FIELD_MAP:
            if key in ("SelgerType", "KjoperType"):
                record[header] = row.get(key)
            elif not er_forste and key in CAR_LEVEL_FIELDS:
                record[header] = None
            else:
                record[header] = row.get(key)
        records.append(record)
    return records


# ------------------------------------------------------------------
# Excel-stiler (BDO-palett)
# ------------------------------------------------------------------

_THIN_BORDER_SIDE = Side(style="thin", color=FARGE_KANT)
THIN_BORDER = Border(
    left=_THIN_BORDER_SIDE, right=_THIN_BORDER_SIDE,
    top=_THIN_BORDER_SIDE, bottom=_THIN_BORDER_SIDE,
)


def set_title_bar(ws, row: int, col_span: int, text: str) -> None:
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=col_span)
    cell = ws.cell(row=row, column=1, value=text)
    cell.font = Font(bold=True, size=14, color=FARGE_HEADER_FONT)
    cell.fill = PatternFill("solid", fgColor=FARGE_HEADER_BG)
    cell.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 26


def set_explanation_row(ws, row: int, col_span: int, text: str) -> None:
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=col_span)
    cell = ws.cell(row=row, column=1, value=text)
    cell.font = Font(italic=True)
    cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.row_dimensions[row].height = 30


def set_section_bar(ws, row: int, col_span: int, text: str) -> None:
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=col_span)
    cell = ws.cell(row=row, column=1, value=text)
    cell.font = Font(bold=True, size=12, color=FARGE_SEKSJON_FONT)
    cell.fill = PatternFill("solid", fgColor=FARGE_SEKSJON_BG)
    cell.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 20


def set_kpi_box(ws, row: int, col_from: int, col_to: int, label: str,
                 value, bg_hex: str = None, font_hex: str = None,
                 big: bool = False) -> None:
    ws.merge_cells(start_row=row, start_column=col_from, end_row=row, end_column=col_to)
    cell = ws.cell(row=row, column=col_from, value=label)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", vertical="center")
    if bg_hex:
        cell.fill = PatternFill("solid", fgColor=bg_hex)
        cell.font = Font(bold=True, color=font_hex or "000000")
    ws.merge_cells(start_row=row + 1, start_column=col_from, end_row=row + 1, end_column=col_to)
    vcell = ws.cell(row=row + 1, column=col_from, value=value)
    vcell.font = Font(bold=True, size=16 if big else 12, color=font_hex or "000000")
    vcell.alignment = Alignment(horizontal="center", vertical="center")
    if bg_hex:
        vcell.fill = PatternFill("solid", fgColor=bg_hex)


def style_table_header(ws, row: int, headers: list) -> None:
    for i, header in enumerate(headers, start=1):
        cell = ws.cell(row=row, column=i, value=header)
        cell.font = Font(bold=True, color=FARGE_HEADER_FONT)
        cell.fill = PatternFill("solid", fgColor=FARGE_HEADER_BG)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[row].height = 30


def color_status_cell(cell, status_text: str) -> None:
    status_text = status_text or ""
    if status_text.startswith("OK"):
        cell.fill = PatternFill("solid", fgColor=FARGE_OK_BG)
        cell.font = Font(color=FARGE_OK_FONT, bold=True)
    elif status_text.startswith("Avvik"):
        cell.fill = PatternFill("solid", fgColor=FARGE_AVVIK_BG)
        cell.font = Font(color=FARGE_AVVIK_FONT, bold=True)


def set_column_widths(ws, widths: dict) -> None:
    for col_letter, width in widths.items():
        ws.column_dimensions[col_letter].width = width


def set_date_format(ws, col_letter: str, first_row: int, last_row: int) -> None:
    if last_row < first_row:
        return
    for row in range(first_row, last_row + 1):
        ws[f"{col_letter}{row}"].number_format = "dd.mm.yyyy"

In [ ]:
# ==================================================================
# CELLE 3: HOVEDLOGIKK
# ==================================================================

# ------------------------------------------------------------------
# Kontroll 1: Kontroll solgte biler
# ------------------------------------------------------------------


def build_kontroll_row(regno: str, vin: str, bokfort_raw, tx_rows: list,
                        svv_info, seller_org_no: str) -> dict:
    bokfort_date = parse_bokfort_dato(bokfort_raw)
    seller_norm = normalize_orgnr(seller_org_no)

    model_name = ""
    chassis_no = vin
    regno_resolved = regno
    first_reg_date = None
    has_any_ok = False
    error_status = ""

    nearest_row = None
    nearest_diff = None

    for r in tx_rows:
        status = r.get("Status")
        if status == "OK":
            has_any_ok = True
            if not model_name:
                model_name = r.get("ModelName") or ""
            if r.get("ChassisNumber"):
                chassis_no = r["ChassisNumber"]
            if r.get("RegNo"):
                regno_resolved = r["RegNo"]
            if first_reg_date is None and r.get("FirstRegistrationDate"):
                first_reg_date = r["FirstRegistrationDate"]
            tx_date = r.get("TransactionDate")
            if tx_date and bokfort_date:
                diff = abs((tx_date - bokfort_date).days)
                if nearest_diff is None or diff < nearest_diff:
                    nearest_diff = diff
                    nearest_row = r
        elif isinstance(status, str) and status.startswith("Feil:"):
            if not error_status:
                error_status = status

    if not has_any_ok and svv_info and svv_info.get("FirstRegistrationDate"):
        first_reg_date = svv_info["FirstRegistrationDate"]

    result = {
        "RegnrInput": regno_resolved,
        "Chassisnummer": chassis_no,
        "Modell": model_name,
        "BokfortDato": bokfort_date,
        "Forstegangsregistrert": first_reg_date,
        "KontrollTransaksjonDato": None,
        "RegistreringsType": "",
        "DagerAvvik": None,
        "Selger": "",
        "Kjoper": "",
        "Kilde": "Ingen",
        "Selvhandel": "",
    }

    if bokfort_date is None:
        if has_any_ok or first_reg_date is not None:
            result["ApiTreff"] = "Mangler bokfort dato"
        elif error_status:
            result["ApiTreff"] = error_status
        else:
            result["ApiTreff"] = "Ingen treff"
        result["Kontrollert"] = "Nei"

    elif nearest_row is not None:
        result["KontrollTransaksjonDato"] = nearest_row["TransactionDate"]
        result["RegistreringsType"] = nearest_row.get("RegistrationType") or ""
        result["DagerAvvik"] = nearest_diff
        result["ApiTreff"] = "Treff OFV eierskifte"
        result["Kontrollert"] = "Ja"
        result["Kilde"] = "OFV"
        result["Selger"] = compute_owner_label(
            nearest_row.get("FromOwnerType"), nearest_row.get("FromOwnerCompanyName"))
        result["Kjoper"] = compute_owner_label(
            nearest_row.get("ToOwnerType"), nearest_row.get("ToOwnerCompanyName"))
        if seller_norm:
            fra_org = normalize_orgnr(nearest_row.get("FromOwnerOrgNo"))
            til_org = normalize_orgnr(nearest_row.get("ToOwnerOrgNo"))
            result["Selvhandel"] = "Ja" if (
                fra_org == seller_norm and til_org == seller_norm) else "Nei"

    elif first_reg_date is not None:
        result["KontrollTransaksjonDato"] = first_reg_date
        result["RegistreringsType"] = "Forstegangsregistrering (SVV)"
        result["DagerAvvik"] = abs((first_reg_date - bokfort_date).days)
        result["ApiTreff"] = "Treff SVV forstegangsregistrering"
        result["Kontrollert"] = "Ja"
        result["Kilde"] = "SVV"

    else:
        if svv_info is None:
            result["ApiTreff"] = error_status or "Ingen treff"
        else:
            result["ApiTreff"] = svv_info.get("Status", "Ingen treff")
        result["Kontrollert"] = "Nei"

    return result


def run_kontroll_solgte_biler() -> dict:
    print("=== Kontroll solgte biler ===")
    konfig = read_konfig(INPUT_FILE_SOLGTE_BILER)
    seller_org_no = normalize_orgnr(konfig.get("OrgNrSelger"))
    biler = read_biler(INPUT_FILE_SOLGTE_BILER, "BokfortDato")

    all_rows = []
    kontroll_rows = []
    total = len(biler)
    svv_treff = 0
    svv_feil = 0
    ofv_feil = 0
    ofv_uten_treff = 0

    for i, bil in enumerate(biler, start=1):
        regno, vin = bil["regno"], bil["vin"]
        ident = bil["identifikator"]
        use_vin = bool(vin)
        print(f"  OFV: Eierskifter {ident}  ({i}/{total})")
        tx_rows = fetch_ofv_transactions(OFV_API_KEY, ident, use_vin, regno, vin)
        all_rows.extend(tx_rows)
        has_any_ok = any(r.get("Status") == "OK" for r in tx_rows)
        if not has_any_ok:
            if any(isinstance(r.get("Status"), str) and r["Status"].startswith("Feil:")
                   for r in tx_rows):
                ofv_feil += 1
            else:
                ofv_uten_treff += 1
        svv_info = None
        if not has_any_ok and SVV_API_KEY:
            print(f"  SVV: Forstegangsregistrering {ident}  ({i}/{total})")
            svv_info = fetch_vehicle_info_from_svv(SVV_API_KEY, regno, vin)
            if svv_info.get("FirstRegistrationDate"):
                svv_treff += 1
            if isinstance(svv_info.get("Status"), str) and svv_info["Status"].startswith("Feil:"):
                svv_feil += 1
        kontroll_rows.append(
            build_kontroll_row(regno, vin, bil["BokfortDato"], tx_rows, svv_info, seller_org_no))
        time.sleep(API_PAUSE_S)

    kontroll_rows_sortert = sorted(
        kontroll_rows,
        key=lambda r: r["DagerAvvik"] if isinstance(r["DagerAvvik"], (int, float)) else -1,
        reverse=True,
    )

    return {
        "kontroll": "Kontroll solgte biler",
        "kildefil": INPUT_FILE_SOLGTE_BILER,
        "orgnr": seller_org_no,
        "dato_fra": None,
        "dato_til": None,
        "antall_biler": total,
        "biler_manifest": biler,
        "all_rows": all_rows,
        "kontroll_rows": kontroll_rows_sortert,
        "stats": {
            "Antall biler": total,
            "OFV-treff": sum(1 for r in kontroll_rows if r["Kilde"] == "OFV"),
            "Uten OFV-treff": ofv_uten_treff,
            "OFV-feil": ofv_feil,
            "SVV-forstegangsregistreringer": svv_treff,
            "SVV-feil": svv_feil,
        },
    }


# ------------------------------------------------------------------
# Kontroll 2: Varekjop bruktbil
# ------------------------------------------------------------------


def finn_lagerperiode(tx_rows: list, buyer_org_no: str, kjop_grense: date):
    buyer_norm = normalize_orgnr(buyer_org_no)
    chassis_no = ""
    model_name = ""
    regno_resolved = ""
    kjopt_row = None
    solgt_row = None

    for r in tx_rows:
        if r.get("Status") != "OK":
            continue
        if not model_name:
            model_name = r.get("ModelName") or ""
        if r.get("ChassisNumber"):
            chassis_no = r["ChassisNumber"]
        if r.get("RegNo"):
            regno_resolved = r["RegNo"]
        tx_date = r.get("TransactionDate")
        if not tx_date:
            continue
        if (normalize_orgnr(r.get("ToOwnerOrgNo")) == buyer_norm
                and tx_date <= kjop_grense):
            if kjopt_row is None or tx_date > kjopt_row["TransactionDate"]:
                kjopt_row = r

    if kjopt_row is not None:
        for r in tx_rows:
            if r.get("Status") != "OK":
                continue
            tx_date = r.get("TransactionDate")
            if not tx_date:
                continue
            if (normalize_orgnr(r.get("FromOwnerOrgNo")) == buyer_norm
                    and tx_date > kjopt_row["TransactionDate"]):
                if solgt_row is None or tx_date < solgt_row["TransactionDate"]:
                    solgt_row = r

    return kjopt_row, solgt_row, chassis_no, model_name, regno_resolved


def lager_status_tekst(kjopt_row, solgt_row, dato_fra: date) -> str:
    if kjopt_row is None:
        return "Avvik: fant ikke opprinnelig kjop hos forhandleren"
    if solgt_row is None:
        return "OK - bilen var pa lager i perioden"
    if solgt_row["TransactionDate"] >= dato_fra:
        return "OK - bilen var pa lager i perioden"
    return "Avvik: solgt for perioden startet"


def build_innkjop_eller_ib_row(regno: str, vin: str, tx_rows: list, buyer_org_no: str,
                                dato_fra: date, dato_til: date, kjop_grense: date,
                                flagg_etikett: str, flagg_verdi: str) -> dict:
    kjopt_row, solgt_row, chassis_no, model_name, regno_resolved = finn_lagerperiode(
        tx_rows, buyer_org_no, kjop_grense)
    result = {
        "RegnrInput": regno_resolved or regno,
        "Chassisnummer": chassis_no or vin,
        "Modell": model_name,
        "FlaggEtikett": flagg_etikett,
        "FlaggVerdi": flagg_verdi,
        "FraDato": kjopt_row["TransactionDate"] if kjopt_row else None,
        "KjoptFra": compute_owner_label(
            kjopt_row.get("FromOwnerType"), kjopt_row.get("FromOwnerCompanyName")
        ) if kjopt_row else "",
        "TilDato": solgt_row["TransactionDate"] if solgt_row else None,
        "Kjoper": compute_owner_label(
            solgt_row.get("ToOwnerType"), solgt_row.get("ToOwnerCompanyName")
        ) if solgt_row else "",
    }
    result["Status"] = lager_status_tekst(kjopt_row, solgt_row, dato_fra)
    return result


def build_manglende_bokforing_row(ofv_row: dict, tx_rows: list, buyer_org_no: str,
                                   dato_til: date) -> dict:
    kjopt_row, solgt_row, chassis_no, model_name, regno_resolved = finn_lagerperiode(
        tx_rows, buyer_org_no, dato_til)
    if kjopt_row is None:
        kjopt_row = ofv_row
    result = {
        "RegnrInput": regno_resolved or ofv_row.get("RegNo") or "",
        "Chassisnummer": chassis_no or ofv_row.get("ChassisNumber") or "",
        "Modell": model_name or ofv_row.get("ModelName") or "",
        "FlaggEtikett": "I OFV-liste",
        "FlaggVerdi": "Ja",
        "FraDato": kjopt_row.get("TransactionDate"),
        "KjoptFra": compute_owner_label(
            kjopt_row.get("FromOwnerType"), kjopt_row.get("FromOwnerCompanyName")),
        "TilDato": solgt_row["TransactionDate"] if solgt_row else None,
        "Kjoper": compute_owner_label(
            solgt_row.get("ToOwnerType"), solgt_row.get("ToOwnerCompanyName")
        ) if solgt_row else "",
        "Status": "Avvik: OFV viser kjop, mangler i bokforing",
    }
    return result


def build_ub_row(regno: str, vin: str, tx_rows: list, er_i_ib: bool,
                  er_i_innkjop: bool) -> dict:
    chassis_no, model_name, regno_resolved = vin, "", regno
    for r in tx_rows:
        if r.get("Status") != "OK":
            continue
        if not model_name:
            model_name = r.get("ModelName") or ""
        if r.get("ChassisNumber"):
            chassis_no = r["ChassisNumber"]
        if r.get("RegNo"):
            regno_resolved = r["RegNo"]
    return {
        "RegnrInput": regno_resolved,
        "Chassisnummer": chassis_no,
        "Modell": model_name,
        "FunnetIIB": "Ja" if er_i_ib else "Nei",
        "FunnetIInnkjop": "Ja" if er_i_innkjop else "Nei",
        "Status": "OK - forklart" if (er_i_ib or er_i_innkjop) else "Avvik: uforklart lagerbeholdning",
    }


def run_kontroll_varekjop_bruktbil() -> dict:
    print("=== Varekjop bruktbil ===")
    konfig = read_konfig(INPUT_FILE_VAREKJOP)
    buyer_org_no = normalize_orgnr(konfig.get("OrgNrKjoper"))
    dato_fra = parse_bokfort_dato(konfig.get("DatoFra"))
    dato_til = parse_bokfort_dato(konfig.get("DatoTil"))
    if not buyer_org_no or not dato_fra or not dato_til:
        raise ValueError(
            "Varekjop bruktbil: OrgNrKjoper/DatoFra/DatoTil ma vaere fylt ut "
            "i Konfig-fanen."
        )
    biler = read_biler(INPUT_FILE_VAREKJOP, "Liste")
    queue_innkjop, queue_ib, queue_ub = {}, {}, {}
    for bil in biler:
        key = build_vehicle_key(bil["vin"], bil["regno"])
        if not key:
            continue
        liste = str(bil.get("Liste") or "").strip().lower()
        if liste == "innkjop":
            queue_innkjop[key] = bil
        elif liste == "ib":
            queue_ib[key] = bil
        elif liste == "ub":
            queue_ub[key] = bil
    print(f"  Innkjop: {len(queue_innkjop)}, IB: {len(queue_ib)}, UB: {len(queue_ub)}")

    # Steg 1: alle biler OFV sier er kjopt av orgnr i perioden
    print(f"  OFV: henter kjopsliste for org {buyer_org_no} ({dato_fra} - {dato_til}) ...")
    ofv_liste = fetch_ofv_transactions_by_buyer_org(OFV_API_KEY, buyer_org_no, dato_fra, dato_til)
    print(f"  OFV: {len(ofv_liste)} transaksjoner funnet i perioden.")
    ofv_regnr = {normalize_identifier(r["RegNo"]) for r in ofv_liste if r.get("RegNo")}
    ofv_vin = {normalize_identifier(r["ChassisNumber"]) for r in ofv_liste if r.get("ChassisNumber")}

    # Steg 2: full transaksjonshistorikk for hver bil i Innkjop + IB
    queue_api = {**queue_innkjop, **queue_ib}
    vehicle_rows_by_key = {}
    all_rows = []
    total = len(queue_api)
    for i, (key, bil) in enumerate(queue_api.items(), start=1):
        regno, vin = bil["regno"], bil["vin"]
        ident = bil["identifikator"]
        print(f"  OFV: Eierskifter {ident}  ({i}/{total})")
        tx_rows = fetch_ofv_transactions(OFV_API_KEY, ident, bool(vin), regno, vin)
        vehicle_rows_by_key[key] = tx_rows
        all_rows.extend(tx_rows)
        time.sleep(API_PAUSE_S)

    # Seksjon A: Innkjop i perioden
    seksjon_a = []
    for key, bil in queue_innkjop.items():
        regno, vin = bil["regno"], bil["vin"]
        tx_rows = vehicle_rows_by_key.get(key, [])
        er_i_ofv_liste = (
            (regno and normalize_identifier(regno) in ofv_regnr)
            or (vin and normalize_identifier(vin) in ofv_vin)
        )
        seksjon_a.append(build_innkjop_eller_ib_row(
            regno, vin, tx_rows, buyer_org_no, dato_fra, dato_til, dato_til,
            "I OFV-liste", "Ja" if er_i_ofv_liste else "Nei",
        ))

    # Biler OFV sier er kjopt i perioden, men IKKE i Innkjop-listen
    for ofv_row in ofv_liste:
        reg_n = normalize_identifier(ofv_row.get("RegNo"))
        vin_n = normalize_identifier(ofv_row.get("ChassisNumber"))
        funnet = any(
            (reg_n and reg_n == normalize_identifier(b["regno"]))
            or (vin_n and vin_n == normalize_identifier(b["vin"]))
            for b in queue_innkjop.values()
        )
        if funnet:
            continue
        key = build_vehicle_key(vin_n, reg_n)
        tx_rows = vehicle_rows_by_key.get(key)
        if tx_rows is None:
            ident = vin_n or reg_n
            print(f"  OFV: Eierskifter (mangler i bokforing) {ident}")
            tx_rows = fetch_ofv_transactions(OFV_API_KEY, ident, bool(vin_n), reg_n, vin_n)
            vehicle_rows_by_key[key] = tx_rows
            all_rows.extend(tx_rows)
            time.sleep(API_PAUSE_S)
        seksjon_a.append(build_manglende_bokforing_row(ofv_row, tx_rows, buyer_org_no, dato_til))

    # Seksjon B: IB (kjopt forrige periode)
    seksjon_b = []
    kjop_grense_ib = dato_fra - timedelta(days=1)
    for key, bil in queue_ib.items():
        regno, vin = bil["regno"], bil["vin"]
        tx_rows = vehicle_rows_by_key.get(key, [])
        er_pa_ub = key in queue_ub
        seksjon_b.append(build_innkjop_eller_ib_row(
            regno, vin, tx_rows, buyer_org_no, dato_fra, dato_til, kjop_grense_ib,
            "Pa UB-liste", "Ja" if er_pa_ub else "Nei",
        ))

    # Seksjon C: UB (fortsatt pa lager)
    seksjon_c = []
    for key, bil in queue_ub.items():
        regno, vin = bil["regno"], bil["vin"]
        tx_rows = vehicle_rows_by_key.get(key, [])
        seksjon_c.append(build_ub_row(
            regno, vin, tx_rows, key in queue_ib, key in queue_innkjop))

    return {
        "kontroll": "Varekjop bruktbil",
        "kildefil": INPUT_FILE_VAREKJOP,
        "orgnr": buyer_org_no,
        "dato_fra": dato_fra,
        "dato_til": dato_til,
        "antall_biler": len(biler),
        "biler_manifest": biler,
        "all_rows": all_rows,
        "seksjon_a": seksjon_a,
        "seksjon_b": seksjon_b,
        "seksjon_c": seksjon_c,
        "stats": {
            "Innkjop (antall)": len(queue_innkjop),
            "IB (antall)": len(queue_ib),
            "UB (antall)": len(queue_ub),
            "Innkjop - OK": sum(1 for r in seksjon_a if r["Status"].startswith("OK")),
            "Innkjop - Avvik": sum(1 for r in seksjon_a if r["Status"].startswith("Avvik")
                                   and "mangler i bokforing" not in r["Status"]),
            "OFV-kjop uten bokforing": sum(
                1 for r in seksjon_a if "mangler i bokforing" in r["Status"]),
        },
    }


# ------------------------------------------------------------------
# Kontroll 3: Kontroll Demobil
# ------------------------------------------------------------------


def build_demobil_row(regno: str, vin: str, bokfort_inn_raw, tx_rows: list,
                       company_org_no: str) -> dict:
    bokfort_date = parse_bokfort_dato(bokfort_inn_raw)
    company_norm = normalize_orgnr(company_org_no)
    model_name = ""
    chassis_no = vin
    regno_resolved = regno
    nyeste_row = None
    company_navn = ""

    for r in tx_rows:
        if r.get("Status") != "OK":
            continue
        if not model_name:
            model_name = r.get("ModelName") or ""
        if r.get("ChassisNumber"):
            chassis_no = r["ChassisNumber"]
        if r.get("RegNo"):
            regno_resolved = r["RegNo"]
        if not company_navn and normalize_orgnr(r.get("ToOwnerOrgNo")) == company_norm:
            company_navn = r.get("ToOwnerCompanyName") or ""
        tx_date = r.get("TransactionDate")
        if tx_date and (nyeste_row is None or tx_date > nyeste_row["TransactionDate"]):
            nyeste_row = r

    result = {
        "RegnrInput": regno_resolved,
        "Chassisnummer": chassis_no,
        "Modell": model_name,
        "BokfortInnDato": bokfort_date,
        "SisteTransaksjonsDato": None,
        "SisteKjoper": "",
        "EidAvEnhet": "",
    }

    if nyeste_row is None:
        result["Status"] = "Ingen treff - ingen OFV-transaksjoner funnet"
    else:
        result["SisteTransaksjonsDato"] = nyeste_row["TransactionDate"]
        result["SisteKjoper"] = compute_owner_label(
            nyeste_row.get("ToOwnerType"), nyeste_row.get("ToOwnerCompanyName"))
        if normalize_orgnr(nyeste_row.get("ToOwnerOrgNo")) == company_norm:
            navn = company_navn or nyeste_row.get("ToOwnerCompanyName") or company_org_no
            result["EidAvEnhet"] = "Ja"
            result["Status"] = f"OK - fortsatt eid av {navn}"
        else:
            result["EidAvEnhet"] = "Nei"
            result["Status"] = f"Avvik - siste eierskifte er til {result['SisteKjoper']}"

    return result


def run_kontroll_demobil() -> dict:
    print("=== Kontroll Demobil ===")
    konfig = read_konfig(INPUT_FILE_DEMOBIL)
    company_org_no = normalize_orgnr(konfig.get("OrgNr"))
    dato_fra = parse_bokfort_dato(konfig.get("DatoFra"))
    dato_til = parse_bokfort_dato(konfig.get("DatoTil"))
    if not company_org_no:
        raise ValueError("Kontroll Demobil: OrgNr ma vaere fylt ut i Konfig-fanen.")
    biler = read_biler(INPUT_FILE_DEMOBIL, "BokfortInnDato")
    all_rows = []
    kontroll_rows = []
    total = len(biler)
    for i, bil in enumerate(biler, start=1):
        regno, vin = bil["regno"], bil["vin"]
        ident = bil["identifikator"]
        print(f"  OFV: Eierskifter {ident}  ({i}/{total})")
        tx_rows = fetch_ofv_transactions(OFV_API_KEY, ident, bool(vin), regno, vin)
        all_rows.extend(tx_rows)
        kontroll_rows.append(
            build_demobil_row(regno, vin, bil["BokfortInnDato"], tx_rows, company_org_no))
        time.sleep(API_PAUSE_S)

    return {
        "kontroll": "Kontroll Demobil",
        "kildefil": INPUT_FILE_DEMOBIL,
        "orgnr": company_org_no,
        "dato_fra": dato_fra,
        "dato_til": dato_til,
        "antall_biler": total,
        "biler_manifest": biler,
        "all_rows": all_rows,
        "kontroll_rows": kontroll_rows,
        "stats": {
            "Antall biler": total,
            "Fortsatt hos enhet": sum(
                1 for r in kontroll_rows if r["Status"].startswith("OK")),
            "Avvik (videresolgt)": sum(
                1 for r in kontroll_rows if r["Status"].startswith("Avvik")),
            "Ingen OFV-treff": sum(
                1 for r in kontroll_rows if r["Status"].startswith("Ingen treff")),
        },
    }


# ------------------------------------------------------------------
# Kjor alle tre kontrollene
# ------------------------------------------------------------------

_OFV_CACHE.clear()
_SVV_CACHE.clear()
API_TELLER.clear()

_t0 = time.time()
TIDSBRUK_PER_KONTROLL = {}


def _kjor(navn: str, funksjon):
    """Kjorer en kontroll hvis inputfilen finnes, ellers None."""
    if not KJOR_KONTROLL.get(navn):
        print(f"=== {navn}: hoppet over (inputfil mangler) ===")
        return None
    t = time.time()
    resultat = funksjon()
    TIDSBRUK_PER_KONTROLL[resultat["kontroll"]] = time.time() - t
    print(f"  Ferdig pa {time.time() - t:.1f} s\n")
    return resultat


resultat_solgte_biler = _kjor("solgte_biler", run_kontroll_solgte_biler)
resultat_varekjop = _kjor("varekjop_bruktbil", run_kontroll_varekjop_bruktbil)
resultat_demobil = _kjor("demobil", run_kontroll_demobil)

_elapsed_s = time.time() - _t0

print(f"OFV: {API_TELLER['ofv_kjoretoy']} kjoretoy slatt opp "
      f"({API_TELLER['ofv_buffer']} gjenbrukt fra buffer, "
      f"{API_TELLER['ofv_http']} HTTP-kall totalt). "
      f"SVV: {API_TELLER['svv_http']} HTTP-kall.")

In [ ]:
# ==================================================================
# CELLE 4: OUTPUT - samlet Excel-arbeidsbok
# ==================================================================

# ------------------------------------------------------------------
# Felles
# ------------------------------------------------------------------


def finn_orgnavn(all_rows: list, org_no: str) -> str:
    """Firmanavnet OFV oppgir for orgnr (forste treff som kjoper/selger)."""
    org_norm = normalize_orgnr(org_no)
    if not org_norm:
        return ""
    for side in ("ToOwner", "FromOwner"):
        for row in all_rows:
            if normalize_orgnr(row.get(f"{side}OrgNo")) == org_norm:
                navn = row.get(f"{side}CompanyName") or ""
                if navn:
                    return navn
    return ""


# ------------------------------------------------------------------
# "Input"-fane
# ------------------------------------------------------------------


def skriv_input_fane(wb: Workbook, bundles: list) -> None:
    ws = wb.create_sheet("Input")
    set_title_bar(ws, 1, 6, "Input - oversikt over kildefiler og konfigurasjon")
    ws.row_dimensions[1].height = 26
    set_section_bar(ws, 3, 6, "Konfigurasjon brukt per kontroll")
    headers = ["Kontroll", "Kildefil", "Orgnr", "Dato fra", "Dato til", "Antall biler"]
    style_table_header(ws, 4, headers)
    r = 5
    for b in bundles:
        ws.cell(row=r, column=1, value=b["kontroll"])
        ws.cell(row=r, column=2, value=b["kildefil"])
        ws.cell(row=r, column=3, value=b["orgnr"])
        if b["dato_fra"]:
            ws.cell(row=r, column=4, value=b["dato_fra"]).number_format = "dd.mm.yyyy"
        if b["dato_til"]:
            ws.cell(row=r, column=5, value=b["dato_til"]).number_format = "dd.mm.yyyy"
        ws.cell(row=r, column=6, value=b["antall_biler"])
        r += 1

    r += 2
    set_section_bar(ws, r, 6, "Biler lest per kontroll")
    r += 1
    headers2 = ["Kontroll", "Regnr/VIN", "Bokfort dato", "Liste-type", "Bokfort inn dato"]
    style_table_header(ws, r, headers2)
    r += 1

    for b in bundles:
        for bil in b["biler_manifest"]:
            ws.cell(row=r, column=1, value=b["kontroll"])
            ws.cell(row=r, column=2, value=bil["identifikator"])
            if "BokfortDato" in bil and bil["BokfortDato"] is not None:
                d = parse_bokfort_dato(bil["BokfortDato"])
                if d:
                    ws.cell(row=r, column=3, value=d).number_format = "dd.mm.yyyy"
            if "Liste" in bil and bil["Liste"]:
                ws.cell(row=r, column=4, value=str(bil["Liste"]))
            if "BokfortInnDato" in bil and bil["BokfortInnDato"] is not None:
                d = parse_bokfort_dato(bil["BokfortInnDato"])
                if d:
                    ws.cell(row=r, column=5, value=d).number_format = "dd.mm.yyyy"
            r += 1

    set_column_widths(ws, {"A": 26, "B": 45, "C": 16, "D": 18, "E": 16, "F": 14})


# ------------------------------------------------------------------
# "Resultat ..." - full transaksjonshistorikk
# ------------------------------------------------------------------


def skriv_resultat_fane(wb: Workbook, sheet_name: str, all_rows: list) -> None:
    ws = wb.create_sheet(sheet_name[:31])
    records = build_resultat_rows(all_rows)
    headers = [header for _key, header, _is_date in FIELD_MAP]
    style_table_header(ws, 1, headers)

    for r_idx, rec in enumerate(records, start=2):
        for c_idx, (_key, header, is_date) in enumerate(FIELD_MAP, start=1):
            value = rec.get(header)
            cell = ws.cell(row=r_idx, column=c_idx, value=value)
            if is_date and value:
                cell.number_format = "dd.mm.yyyy"
        if rec["_ErForsteIGruppe"]:
            for c_idx in range(1, len(headers) + 1):
                ws.cell(row=r_idx, column=c_idx).fill = PatternFill("solid", fgColor=FARGE_GRUPPE_BG)
                ws.cell(row=r_idx, column=c_idx).font = Font(bold=True)

    widths = {get_column_letter(i): 18 for i in range(1, len(headers) + 1)}
    widths.update({"A": 14, "D": 20, "P": 25, "Q": 25, "U": 25})
    set_column_widths(ws, widths)
    ws.freeze_panes = "A2"
    if records:
        ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{len(records) + 1}"


# ------------------------------------------------------------------
# "Kontroll solgte biler"
# ------------------------------------------------------------------


def skriv_kontroll_solgte_biler(wb: Workbook, bundle: dict) -> None:
    ws = wb.create_sheet("Kontroll solgte biler"[:31])
    rows = bundle["kontroll_rows"]
    set_title_bar(ws, 1, 12, "Kontroll solgte biler")
    set_explanation_row(
        ws, 2, 12,
        "Kontrollregel: Bokfort dato sjekkes mot den OFV-transaksjonen som "
        "ligger NAERMEST bokfort dato i tid (for eller etter). Dager avvik "
        "er antall dager mellom denne datoen og bokfort dato. Har OFV ingen "
        "transaksjoner for kjoretoyet, brukes forstegangsregistrering fra "
        "Statens vegvesen (SVV) i stedet. Er OrgNrSelger fylt ut i input, "
        "flagges Selvhandel - det tvinger Dager avvik rod uansett faktisk "
        "avvik. Sortert med storst avvik forst."
    )

    bucket0 = sum(1 for r in rows if r["Kontrollert"] == "Ja"
                  and isinstance(r["DagerAvvik"], (int, float))
                  and r["Selvhandel"] != "Ja" and r["DagerAvvik"] <= AVVIK_GRONN_MAX)
    bucket1 = sum(1 for r in rows if r["Kontrollert"] == "Ja"
                  and isinstance(r["DagerAvvik"], (int, float))
                  and r["Selvhandel"] != "Ja"
                  and AVVIK_GRONN_MAX < r["DagerAvvik"] <= AVVIK_GUL_MAX)
    bucket2 = sum(1 for r in rows if r["Kontrollert"] == "Ja"
                  and (r["Selvhandel"] == "Ja"
                       or not isinstance(r["DagerAvvik"], (int, float))
                       or r["DagerAvvik"] > AVVIK_GUL_MAX))

    set_kpi_box(ws, 4, 1, 2, "Inputbiler", len(rows), big=True)
    set_kpi_box(ws, 4, 3, 4, "0-2 dager", bucket0, FARGE_OK_BG, FARGE_OK_FONT, big=True)
    set_kpi_box(ws, 4, 5, 6, "3-14 dager", bucket1, FARGE_VARSEL_BG, FARGE_VARSEL_FONT, big=True)
    set_kpi_box(ws, 4, 7, 8, "15+ dager / selvhandel", bucket2, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)

    header_row = 7
    headers = ["Kilde", "API-treff", "Regnr", "Chassisnummer", "Modell",
               "Forstegangsregistrert", "Bokfort dato", "Transaksjonsdato",
               "RegistreringsType", "Dager avvik", "Selger", "Kjoper"]
    style_table_header(ws, header_row, headers)

    r = header_row + 1
    for row in rows:
        ws.cell(row=r, column=1, value=row["Kilde"])
        ws.cell(row=r, column=2, value=row["ApiTreff"])
        ws.cell(row=r, column=3, value=row["RegnrInput"])
        ws.cell(row=r, column=4, value=row["Chassisnummer"])
        ws.cell(row=r, column=5, value=row["Modell"])
        if row["Forstegangsregistrert"]:
            ws.cell(row=r, column=6, value=row["Forstegangsregistrert"]).number_format = "dd.mm.yyyy"
        if row["BokfortDato"]:
            ws.cell(row=r, column=7, value=row["BokfortDato"]).number_format = "dd.mm.yyyy"
        if row["KontrollTransaksjonDato"]:
            ws.cell(row=r, column=8, value=row["KontrollTransaksjonDato"]).number_format = "dd.mm.yyyy"
        ws.cell(row=r, column=9, value=row["RegistreringsType"])

        dager = row["DagerAvvik"]
        if isinstance(dager, (int, float)):
            avvik_cell = ws.cell(row=r, column=10, value=dager)
            if row["Selvhandel"] == "Ja" or dager > AVVIK_GUL_MAX:
                avvik_cell.fill = PatternFill("solid", fgColor=FARGE_AVVIK_BG)
                avvik_cell.font = Font(color=FARGE_AVVIK_FONT, bold=True)
            elif dager <= AVVIK_GRONN_MAX:
                avvik_cell.fill = PatternFill("solid", fgColor=FARGE_OK_BG)
                avvik_cell.font = Font(color=FARGE_OK_FONT, bold=True)
            else:
                avvik_cell.fill = PatternFill("solid", fgColor=FARGE_VARSEL_BG)
                avvik_cell.font = Font(color=FARGE_VARSEL_FONT, bold=True)

        ws.cell(row=r, column=11, value=row["Selger"])
        ws.cell(row=r, column=12, value=row["Kjoper"])
        r += 1

    set_column_widths(ws, {
        "A": 10, "B": 24, "C": 14, "D": 22, "E": 18, "F": 16, "G": 16,
        "H": 16, "I": 24, "J": 12, "K": 25, "L": 25,
    })
    ws.freeze_panes = ws.cell(row=header_row + 1, column=1)
    if rows:
        ws.auto_filter.ref = f"A{header_row}:L{header_row + len(rows)}"


# ------------------------------------------------------------------
# "Kontroll Varekjop Bruktbil"
# ------------------------------------------------------------------


def _skriv_bil_rad(ws, r: int, row: dict) -> None:
    ws.cell(row=r, column=1, value=row["RegnrInput"])
    ws.cell(row=r, column=2, value=row["Chassisnummer"])
    ws.cell(row=r, column=3, value=row["Modell"])
    ws.cell(row=r, column=4, value=row["FlaggVerdi"])
    if row["FraDato"]:
        ws.cell(row=r, column=5, value=row["FraDato"]).number_format = "dd.mm.yyyy"
    if row["TilDato"]:
        ws.cell(row=r, column=6, value=row["TilDato"]).number_format = "dd.mm.yyyy"
    ws.cell(row=r, column=7, value=row["KjoptFra"])
    if row["TilDato"]:
        ws.cell(row=r, column=8, value=row["Kjoper"])
    else:
        ws.cell(row=r, column=8, value="(fortsatt pa lager)")
    status_cell = ws.cell(row=r, column=9, value=row["Status"])
    color_status_cell(status_cell, row["Status"])


def skriv_kontroll_varekjop(wb: Workbook, bundle: dict) -> None:
    ws = wb.create_sheet("Kontroll Varekjop Bruktbil"[:31])
    seksjon_a, seksjon_b, seksjon_c = bundle["seksjon_a"], bundle["seksjon_b"], bundle["seksjon_c"]
    buyer_org_no = bundle["orgnr"]

    buyer_org_name = finn_orgnavn(bundle["all_rows"], buyer_org_no)

    set_title_bar(ws, 1, 9, "Kontroll Varekjop Bruktbil")
    set_explanation_row(
        ws, 2, 9,
        f"Kontrollregel: Biler i Innkjop-listen ({bundle['dato_fra']:%d.%m.%Y} - "
        f"{bundle['dato_til']:%d.%m.%Y}) kontrolleres mot OFV sin "
        f"transaksjonshistorikk for {buyer_org_name} (orgnr {buyer_org_no}): var "
        "bilen pa lager hos forhandleren pa et tidspunkt i perioden (kjopt av "
        "forhandleren, og eventuelt ikke solgt videre for perioden startet), og "
        "er alt OFV sier er kjopt i perioden faktisk bokfort."
    )
    set_explanation_row(
        ws, 3, 9,
        "Seksjon B og C er tillegg og vises bare hvis IB- og/eller UB-listen "
        "er fylt ut: biler kjopt forrige periode (IB) kontrolleres mot om de "
        "er solgt i denne perioden, og biler fortsatt pa lager (UB) "
        "kontrolleres mot om de kan forklares av IB eller innkjop i perioden."
    )

    r = 5
    bekreftet = sum(1 for x in seksjon_a if x["Status"].startswith("OK"))
    avvik = sum(1 for x in seksjon_a if x["Status"].startswith("Avvik") and "mangler i bokforing" not in x["Status"])
    mangler = sum(1 for x in seksjon_a if "mangler i bokforing" in x["Status"])

    set_section_bar(ws, r, 9, "A - Innkjop i perioden")
    r += 2
    set_kpi_box(ws, r - 1, 1, 2, "Innkjop", len(seksjon_a), big=True)
    set_kpi_box(ws, r - 1, 3, 4, "OK - pa lager i perioden", bekreftet, FARGE_OK_BG, FARGE_OK_FONT, big=True)
    set_kpi_box(ws, r - 1, 5, 6, "Avvik", avvik, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)
    set_kpi_box(ws, r - 1, 7, 8, "OFV-kjop uten bokforing", mangler, FARGE_VARSEL_BG, FARGE_VARSEL_FONT, big=True)
    r += 1

    set_explanation_row(ws, r, 9, f"Forhandler: {buyer_org_name}")
    r += 1
    style_table_header(ws, r, ["Regnr/VIN", "Chassisnummer", "Modell", "I OFV-liste",
                               "Pa lager fra", "Pa lager til", "Kjopt fra", "Solgt til", "Status"])
    r += 1
    for row in seksjon_a:
        _skriv_bil_rad(ws, r, row)
        r += 1

    r += 1
    if seksjon_b:
        ok_ib = sum(1 for x in seksjon_b if x["Status"].startswith("OK"))
        solgt_for = sum(1 for x in seksjon_b if x["Status"] == "Avvik: solgt for perioden startet")
        ikke_funnet = sum(1 for x in seksjon_b if x["Status"] == "Avvik: fant ikke opprinnelig kjop hos forhandleren")

        set_section_bar(ws, r, 9, "B - IB (kjopt forrige periode)")
        r += 2
        set_kpi_box(ws, r - 1, 1, 2, "IB-biler", len(seksjon_b), big=True)
        set_kpi_box(ws, r - 1, 3, 4, "OK - pa lager i perioden", ok_ib, FARGE_OK_BG, FARGE_OK_FONT, big=True)
        set_kpi_box(ws, r - 1, 5, 6, "Avvik: solgt for perioden", solgt_for, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)
        set_kpi_box(ws, r - 1, 7, 8, "Avvik: fant ikke kjop", ikke_funnet, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)
        r += 1

        set_explanation_row(ws, r, 9, f"Forhandler: {buyer_org_name}")
        r += 1
        style_table_header(ws, r, ["Regnr/VIN", "Chassisnummer", "Modell", "Pa UB-liste",
                                   "Pa lager fra", "Pa lager til", "Kjopt fra", "Solgt til", "Status"])
        r += 1
        for row in seksjon_b:
            _skriv_bil_rad(ws, r, row)
            r += 1
        r += 1

    if seksjon_c:
        forklart = sum(1 for x in seksjon_c if x["Status"] == "OK - forklart")
        uforklart = len(seksjon_c) - forklart

        set_section_bar(ws, r, 9, "C - UB (fortsatt pa lager)")
        r += 2
        set_kpi_box(ws, r - 1, 1, 3, "UB-biler", len(seksjon_c), big=True)
        set_kpi_box(ws, r - 1, 4, 5, "Forklart", forklart, FARGE_OK_BG, FARGE_OK_FONT, big=True)
        set_kpi_box(ws, r - 1, 6, 8, "Uforklart", uforklart, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)
        r += 1
        style_table_header(ws, r, ["Regnr/VIN", "Chassisnummer", "Modell", "Funnet i IB", "Funnet i innkjop", "Status"])
        r += 1
        for row in seksjon_c:
            ws.cell(row=r, column=1, value=row["RegnrInput"])
            ws.cell(row=r, column=2, value=row["Chassisnummer"])
            ws.cell(row=r, column=3, value=row["Modell"])
            ws.cell(row=r, column=4, value=row["FunnetIIB"])
            ws.cell(row=r, column=5, value=row["FunnetIInnkjop"])
            status_cell = ws.cell(row=r, column=6, value=row["Status"])
            color_status_cell(status_cell, row["Status"])
            r += 1

    set_column_widths(ws, {
        "A": 14, "B": 22, "C": 18, "D": 16, "E": 18, "F": 18, "G": 25, "H": 25, "I": 34,
    })


# ------------------------------------------------------------------
# "Kontroll Demobil"
# ------------------------------------------------------------------


def skriv_kontroll_demobil(wb: Workbook, bundle: dict) -> None:
    ws = wb.create_sheet("Kontroll Demobil"[:31])
    rows = bundle["kontroll_rows"]
    set_title_bar(ws, 1, 7, "Kontroll Demobil")
    set_explanation_row(
        ws, 2, 7,
        "Kontrollregel: For hver bil sjekkes bilens NYESTE registrerte "
        f"OFV-transaksjon (uansett dato) - kjoperen der skal vaere juridisk "
        f"enhet (orgnr {bundle['orgnr']}). Er kjoperen et annet selskap "
        "eller en privatperson, er bilen sannsynligvis solgt videre og "
        "flagges som avvik."
    )

    ok = sum(1 for r in rows if r["Status"].startswith("OK"))
    avvik = sum(1 for r in rows if r["Status"].startswith("Avvik"))
    set_kpi_box(ws, 4, 1, 2, "Inputbiler", len(rows), big=True)
    set_kpi_box(ws, 4, 3, 4, "Fortsatt hos enhet", ok, FARGE_OK_BG, FARGE_OK_FONT, big=True)
    set_kpi_box(ws, 4, 5, 6, "Avvik (videresolgt)", avvik, FARGE_AVVIK_BG, FARGE_AVVIK_FONT, big=True)

    header_row = 7
    style_table_header(ws, header_row, ["Regnr", "Chassisnummer", "Modell",
                                        "Bokfort inn dato", "Siste transaksjonsdato",
                                        "Siste kjoper", "Status"])

    r = header_row + 1
    for row in rows:
        ws.cell(row=r, column=1, value=row["RegnrInput"])
        ws.cell(row=r, column=2, value=row["Chassisnummer"])
        ws.cell(row=r, column=3, value=row["Modell"])
        if row["BokfortInnDato"]:
            ws.cell(row=r, column=4, value=row["BokfortInnDato"]).number_format = "dd.mm.yyyy"
        if row["SisteTransaksjonsDato"]:
            ws.cell(row=r, column=5, value=row["SisteTransaksjonsDato"]).number_format = "dd.mm.yyyy"
        ws.cell(row=r, column=6, value=row["SisteKjoper"])
        status_cell = ws.cell(row=r, column=7, value=row["Status"])
        color_status_cell(status_cell, row["Status"])
        r += 1

    set_column_widths(ws, {"A": 14, "B": 22, "C": 18, "D": 18, "E": 18, "F": 25, "G": 38})
    ws.freeze_panes = ws.cell(row=header_row + 1, column=1)
    if rows:
        ws.auto_filter.ref = f"A{header_row}:G{header_row + len(rows)}"


# ------------------------------------------------------------------
# Bygg hele arbeidsboken
# ------------------------------------------------------------------

_wb = Workbook()
_wb.remove(_wb.active)
_bundles = [b for b in (resultat_solgte_biler, resultat_varekjop, resultat_demobil) if b]

skriv_input_fane(_wb, _bundles)
if resultat_solgte_biler:
    skriv_resultat_fane(_wb, "Resultat solgte biler", resultat_solgte_biler["all_rows"])
    skriv_kontroll_solgte_biler(_wb, resultat_solgte_biler)
if resultat_varekjop:
    skriv_resultat_fane(_wb, "Resultat Varekjop", resultat_varekjop["all_rows"])
    skriv_kontroll_varekjop(_wb, resultat_varekjop)
if resultat_demobil:
    skriv_resultat_fane(_wb, "Resultat Demobil", resultat_demobil["all_rows"])
    skriv_kontroll_demobil(_wb, resultat_demobil)

_wb.save(OUTPUT_XLSX)
print(f"Excel-arbeidsbok lagret: {OUTPUT_XLSX}")

In [ ]:
# ==================================================================
# CELLE 5: HTML-RAPPORT
# ==================================================================
#
# En selvstendig HTML-fil (virker uten nett) i BDO-presentasjonsstil:
#   1. Presentasjonen "Kontroll solgte biler" med faste, anonymiserte
#      eksempeldata (forside, for/na, "Slik henter vi dataene").
#   2. Rapporten: Oversikt, Per forhandler, en side per kontroll og
#      Analyse. Alle tabeller kan sorteres, sokes i, filtreres pa hver
#      kolonne og eksporteres til CSV.
#   3. Tannhjulet (nede til hoyre): lim inn / last opp flere datasett
#      fra Kontroll solgte biler (opptil 20 forhandlere) - vises sammen
#      med Databricks-kjoringen. Alt skjer lokalt i nettleseren.
# Dataene legges inn som JSON og tegnes av JavaScript - Python bygger
# bare datamodellen under. Malen ligger i _HTML_MAL nederst i cellen.
# Excel-leseren (SheetJS) og fontene bygges inn fra RESSURS_DIR (se
# Celle 1) - mangler de, lastes de ned en gang og lagres der.

# ------------------------------------------------------------------
# Hjelpere for datamodellen
# ------------------------------------------------------------------


def _json_verdi(v):
    """Gjor en celleverdi JSON-vennlig (datoer som ISO, bool som Ja/Nei)."""
    if _er_tom(v):
        return None
    if isinstance(v, bool):
        return "Ja" if v else "Nei"
    if isinstance(v, datetime):
        return v.date().isoformat()
    if isinstance(v, date):
        return v.isoformat()
    if isinstance(v, (int, float)):
        return v
    return str(v)


def _kol(key: str, label: str, typ: str = "text", **ekstra) -> dict:
    """Kolonnedefinisjon. typ: text | num | date | status."""
    return {"key": key, "label": label, "type": typ, **ekstra}


def _tabell(tid: str, tittel: str, kolonner: list, rader: list,
            beskrivelse: str = "", ekstra_felt: tuple = ()) -> dict:
    keys = ([k["key"] for k in kolonner] + list(ekstra_felt)
            + [k["toneKey"] for k in kolonner if k.get("toneKey")])
    return {
        "id": tid,
        "tittel": tittel,
        "beskrivelse": beskrivelse,
        "kolonner": kolonner,
        "rader": [{k: _json_verdi(r.get(k)) for k in keys} for r in rader],
    }


# Visningsnavn i HTML (bundle["kontroll"] er intern nokkel og endres ikke)
_VISNINGSNAVN = {
    "Kontroll solgte biler": "Solgte biler",
    "Varekjop bruktbil": "Varekjøp bruktbil",
    "Kontroll Demobil": "Demobil",
}


def _dager_mellom(fra, til):
    if isinstance(fra, date) and isinstance(til, date):
        return (til - fra).days
    return None


def _fmt_dato(d) -> str:
    return d.strftime("%d.%m.%Y") if isinstance(d, date) else ""


def _periode_tekst(bundle: dict) -> str:
    if bundle.get("dato_fra") and bundle.get("dato_til"):
        return f"{_fmt_dato(bundle['dato_fra'])} - {_fmt_dato(bundle['dato_til'])}"
    return ""


# Full OFV-historikk (tilsvarer Resultat-fanene i Excel). I HTML vises
# bil-nivaa-feltene pa ALLE rader, slik at filtrering pa f.eks. Merke
# ikke mister rader.
_HISTORIKK_KOLONNER = [
    _kol("Input", "Input"),
    _kol("Kilde", "Oppslag"),
    _kol("RegNo", "Regnr"),
    _kol("ChassisNumber", "Chassisnummer"),
    _kol("MakeName", "Merke"),
    _kol("ModelName", "Modell"),
    _kol("RegistrationType", "Registreringstype"),
    _kol("FuelGroup", "Drivstoff"),
    _kol("IsLeased", "Leaset"),
    _kol("IsUsedImported", "Bruktimport"),
    _kol("FirstRegistrationDate", "Førstegangsreg.", "date"),
    _kol("TransactionNumber", "Transaksjonsnr"),
    _kol("TransactionDate", "Eierskiftedato", "date"),
    _kol("SelgerType", "Selger"),
    _kol("FromOwnerType", "Selger eiertype"),
    _kol("FromOwnerOrgNo", "Selger orgnr"),
    _kol("FromOwnerCounty", "Selger fylke"),
    _kol("KjoperType", "Kjøper"),
    _kol("ToOwnerType", "Kjøper eiertype"),
    _kol("ToOwnerOrgNo", "Kjøper orgnr"),
    _kol("ToOwnerCounty", "Kjøper fylke"),
    _kol("Status", "Status", "status"),
]


def _historikk_tabell(tid: str, bundle: dict) -> dict:
    return _tabell(
        tid, "Full OFV-transaksjonshistorikk", _HISTORIKK_KOLONNER, bundle["all_rows"],
        "Alle eierskifter OFV returnerte for bilene i kontrollen (samme innhold "
        "som Resultat-fanen i Excel), nyeste først per bil.",
    )


def _vurdering_solgt(r: dict) -> tuple:
    """(tekst, tone) - samme inndeling som KPI-boksene i Excel."""
    if r["Kontrollert"] != "Ja":
        return "Ikke kontrollert", "n"
    if r["Selvhandel"] == "Ja":
        return "Avvik - selvhandel", "r"
    dager = r["DagerAvvik"]
    if not isinstance(dager, (int, float)):
        return "Avvik", "r"
    if dager <= AVVIK_GRONN_MAX:
        return f"OK (0-{AVVIK_GRONN_MAX} dager)", "g"
    if dager <= AVVIK_GUL_MAX:
        return f"Varsel ({AVVIK_GRONN_MAX + 1}-{AVVIK_GUL_MAX} dager)", "y"
    return f"Avvik ({AVVIK_GUL_MAX + 1}+ dager)", "r"


def _status_tone(status: str) -> str:
    """g = OK, y = varsel, r = avvik, n = annet (samme koder som i HTML).
    OFV-kjop som mangler i bokforingen vises som varsel (gul), slik som
    KPI-boksen i Excel."""
    status = status or ""
    if "mangler i bokforing" in status:
        return "y"
    if status.startswith("OK"):
        return "g"
    if status.startswith("Avvik"):
        return "r"
    if status.startswith("Varsel"):
        return "y"
    return "n"


# ------------------------------------------------------------------
# En fane per kontroll
# ------------------------------------------------------------------


def _fane_solgte_biler(b) -> dict:
    """Alltid med i HTML (ogsa nar kontrollen ikke ble kjort), siden
    innlimte datasett vises her. b = None -> tom tabell."""
    tom = b is None
    if tom:
        b = {"kontroll_rows": [], "all_rows": [], "orgnr": "", "kildefil": ""}
    rader = []
    for r in b["kontroll_rows"]:
        tekst, tone = _vurdering_solgt(r)
        rader.append({**r, "Vurdering": tekst, "_tone": tone, "Forhandler": SELSKAP})

    teller = Counter(r["_tone"] for r in rader)
    orgnavn = finn_orgnavn(b["all_rows"], b["orgnr"])

    kolonner = [
        _kol("Vurdering", "Vurdering", "status", toneKey="_tone"),
        _kol("Forhandler", "Forhandler"),
        _kol("DagerAvvik", "Dager avvik", "num", toneKey="_tone"),
        _kol("RegnrInput", "Regnr"),
        _kol("Chassisnummer", "Chassisnummer"),
        _kol("Modell", "Modell"),
        _kol("BokfortDato", "Bokført dato", "date"),
        _kol("KontrollTransaksjonDato", "Transaksjonsdato", "date"),
        _kol("RegistreringsType", "Registreringstype"),
        _kol("Forstegangsregistrert", "Førstegangsreg.", "date"),
        _kol("Selger", "Selger"),
        _kol("Kjoper", "Kjøper"),
        _kol("Kilde", "Kilde"),
        _kol("ApiTreff", "API-treff"),
        _kol("Selvhandel", "Selvhandel"),
        _kol("Kontrollert", "Kontrollert"),
    ]

    return {
        "id": "solgte",
        "tittel": "Solgte biler",
        "fullTittel": "Kontroll solgte biler",
        "regel": (
            "Bokført dato sjekkes mot den OFV-transaksjonen som ligger nærmest "
            "bokført dato i tid (før eller etter). Dager avvik er antall dager mellom "
            "de to datoene. Har OFV ingen transaksjoner for bilen, brukes "
            "førstegangsregistrering fra Statens vegvesen (SVV). Er orgnr for selger "
            "oppgitt, flagges selvhandel (samme selskap som selger og kjøper) som avvik "
            f"uansett dager. Grønn: 0-{AVVIK_GRONN_MAX} dager, gul: "
            f"{AVVIK_GRONN_MAX + 1}-{AVVIK_GUL_MAX} dager, rød: {AVVIK_GUL_MAX + 1}+ dager."
        ),
        "meta": [("Databricks", "ikke kjørt (inputfil mangler)")] if tom else [
            ("Orgnr selger", b["orgnr"] or "(ikke oppgitt - ingen selvhandel-sjekk)"),
            ("Selskap", orgnavn),
            ("Kildefil", b["kildefil"]),
        ],
        "kpier": [
            {"label": "Biler i input", "verdi": len(rader), "tone": ""},
            {"label": f"OK 0-{AVVIK_GRONN_MAX} dager", "verdi": teller["g"], "tone": "g"},
            {"label": f"Varsel {AVVIK_GRONN_MAX + 1}-{AVVIK_GUL_MAX} dager", "verdi": teller["y"], "tone": "y"},
            {"label": f"Avvik {AVVIK_GUL_MAX + 1}+ dager / selvhandel", "verdi": teller["r"], "tone": "r"},
            {"label": "Ikke kontrollert", "verdi": teller["n"], "tone": "n"},
            {"label": "Treff via SVV", "verdi": sum(1 for r in rader if r["Kilde"] == "SVV"), "tone": ""},
        ],
        "tabeller": [
            _tabell("solgte-kontroll", "Kontrollresultat", kolonner, rader,
                    "Én rad per bil i input, sortert med størst avvik først.",
                    ekstra_felt=("_tone",)),
            _historikk_tabell("solgte-historikk", b),
        ],
    }


_VAREKJOP_KOLONNER = [
    _kol("Status", "Status", "status", toneKey="_tone"),
    _kol("RegnrInput", "Regnr"),
    _kol("Chassisnummer", "Chassisnummer"),
    _kol("Modell", "Modell"),
    _kol("FlaggVerdi", "FLAGG"),
    _kol("FraDato", "På lager fra", "date"),
    _kol("TilDato", "På lager til", "date"),
    _kol("Lagerdager", "Dager på lager", "num"),
    _kol("KjoptFra", "Kjøpt fra"),
    _kol("SolgtTil", "Solgt til"),
]


def _varekjop_rader(rader: list, dato_til: date) -> list:
    ut = []
    for r in rader:
        ut.append({
            **r,
            # Solgte biler: kjop -> salg. Fortsatt pa lager: kjop -> periodeslutt.
            "Lagerdager": _dager_mellom(r["FraDato"], r["TilDato"] or dato_til),
            "SolgtTil": r["Kjoper"] if r["TilDato"] else "(fortsatt på lager)",
            "_tone": _status_tone(r["Status"]),
        })
    return ut


def _fane_varekjop(b: dict) -> dict:
    a, bb, c = b["seksjon_a"], b["seksjon_b"], b["seksjon_c"]
    orgnavn = finn_orgnavn(b["all_rows"], b["orgnr"])

    def kolonner_med_flagg(etikett):
        return [dict(k, label=etikett) if k["key"] == "FlaggVerdi" else k
                for k in _VAREKJOP_KOLONNER]

    tabeller = [_tabell(
        "varekjop-a", "A - Innkjøp i perioden", kolonner_med_flagg("I OFV-kjøpsliste"),
        _varekjop_rader(a, b["dato_til"]),
        "Biler i Innkjøp-listen: var bilen på lager hos forhandleren i perioden? "
        "Inkluderer også biler OFV sier er kjøpt i perioden, men som mangler i "
        "bokføringen. Dager på lager regnes til salg, eller til periodeslutt hvis "
        "bilen fortsatt er på lager.",
    )]
    if bb:
        tabeller.append(_tabell(
            "varekjop-b", "B - IB (kjøpt forrige periode)", kolonner_med_flagg("På UB-liste"),
            _varekjop_rader(bb, b["dato_til"]),
            "Biler kjøpt før perioden startet: OK hvis bilen fortsatt var på lager "
            "når perioden startet.",
        ))
    if c:
        tabeller.append(_tabell(
            "varekjop-c", "C - UB (fortsatt på lager)",
            [_kol("Status", "Status", "status"), _kol("RegnrInput", "Regnr"),
             _kol("Chassisnummer", "Chassisnummer"), _kol("Modell", "Modell"),
             _kol("FunnetIIB", "Funnet i IB"), _kol("FunnetIInnkjop", "Funnet i innkjøp")],
            c, "Biler fortsatt på lager - forklart hvis de finnes i IB eller innkjøp.",
        ))
    tabeller.append(_historikk_tabell("varekjop-historikk", b))

    s = b["stats"]
    kpier = [
        {"label": "Innkjøp i input", "verdi": s["Innkjop (antall)"], "tone": ""},
        {"label": "OK - på lager i perioden", "verdi": s["Innkjop - OK"], "tone": "g"},
        {"label": "Avvik", "verdi": s["Innkjop - Avvik"], "tone": "r"},
        {"label": "OFV-kjøp uten bokføring", "verdi": s["OFV-kjop uten bokforing"], "tone": "y"},
    ]
    if bb:
        kpier.append({"label": "IB-biler", "verdi": len(bb), "tone": ""})
    if c:
        kpier.append({"label": "UB uforklart", "verdi": sum(
            1 for r in c if r["Status"].startswith("Avvik")), "tone": ""})

    return {
        "id": "varekjop",
        "tittel": "Varekjøp bruktbil",
        "fullTittel": "Kontroll Varekjøp bruktbil",
        "regel": (
            "For hver bil beregnes lagerperioden hos forhandleren: siste kjøp av "
            "oppgitt juridisk enhet (På lager fra) og første videresalg etter det "
            "kjøpet (På lager til). OK så lenge lagerperioden overlapper "
            "kontrollperioden. I tillegg hentes alle biler OFV sier er kjøpt av "
            "enheten i perioden - de som ikke står i Innkjøp-listen flagges som "
            "«mangler i bokføring»."
        ),
        "meta": [
            ("Orgnr kjøper", b["orgnr"]),
            ("Selskap", orgnavn),
            ("Periode", _periode_tekst(b)),
            ("Kildefil", b["kildefil"]),
        ],
        "kpier": kpier,
        "tabeller": tabeller,
    }


def _fane_demobil(b: dict) -> dict:
    rader = b["kontroll_rows"]
    orgnavn = finn_orgnavn(b["all_rows"], b["orgnr"])
    i_dag = date.today()
    rader = [{**r, "DagerSidenBokfort": _dager_mellom(r["BokfortInnDato"], i_dag),
              "DagerSidenSisteTx": _dager_mellom(r["SisteTransaksjonsDato"], i_dag)}
             for r in rader]
    s = b["stats"]
    meta = [("Orgnr", b["orgnr"]), ("Selskap", orgnavn)]
    if _periode_tekst(b):
        meta.append(("Periode", _periode_tekst(b)))
    meta.append(("Kildefil", b["kildefil"]))

    return {
        "id": "demobil",
        "tittel": "Demobil",
        "fullTittel": "Kontroll Demobil",
        "regel": (
            "En demobil skal være registrert på selskapets juridiske enhet. For hver "
            "bil sjekkes bilens NYESTE registrerte OFV-transaksjon (uansett dato): er "
            "kjøperen der oppgitt orgnr, er bilen fortsatt eid av enheten (OK). Er "
            "kjøperen et annet selskap eller en privatperson, er bilen sannsynligvis "
            "solgt videre (Avvik)."
        ),
        "meta": meta,
        "kpier": [
            {"label": "Biler i input", "verdi": s["Antall biler"], "tone": ""},
            {"label": "Fortsatt hos enhet", "verdi": s["Fortsatt hos enhet"], "tone": "g"},
            {"label": "Avvik (videresolgt)", "verdi": s["Avvik (videresolgt)"], "tone": "r"},
            {"label": "Ingen OFV-treff", "verdi": s["Ingen OFV-treff"], "tone": "n"},
        ],
        "tabeller": [
            _tabell("demobil-kontroll", "Kontrollresultat", [
                _kol("Status", "Status", "status"),
                _kol("EidAvEnhet", "Eid av enhet"),
                _kol("RegnrInput", "Regnr"),
                _kol("Chassisnummer", "Chassisnummer"),
                _kol("Modell", "Modell"),
                _kol("BokfortInnDato", "Bokført inn", "date"),
                _kol("DagerSidenBokfort", "Dager siden bokført inn", "num"),
                _kol("SisteTransaksjonsDato", "Siste eierskifte", "date"),
                _kol("DagerSidenSisteTx", "Dager siden siste eierskifte", "num"),
                _kol("SisteKjoper", "Siste kjøper"),
            ], rader, "Én rad per demobil i input."),
            _historikk_tabell("demobil-historikk", b),
        ],
    }


# ------------------------------------------------------------------
# Oversikt og Analyse
# ------------------------------------------------------------------


def _input_tabell(bundles: list) -> dict:
    rader = []
    for b in bundles:
        for bil in b["biler_manifest"]:
            rader.append({
                "Kontroll": _VISNINGSNAVN.get(b["kontroll"], b["kontroll"]),
                "Identifikator": bil["identifikator"],
                "Type": "VIN" if bil["vin"] else "Regnr",
                "BokfortDato": parse_bokfort_dato(bil.get("BokfortDato")),
                "Liste": bil.get("Liste"),
                "BokfortInnDato": parse_bokfort_dato(bil.get("BokfortInnDato")),
            })
    return _tabell("input-biler", "Biler lest fra inputfilene", [
        _kol("Kontroll", "Kontroll"),
        _kol("Identifikator", "Regnr/VIN"),
        _kol("Type", "Type"),
        _kol("BokfortDato", "Bokført dato", "date"),
        _kol("Liste", "Liste"),
        _kol("BokfortInnDato", "Bokført inn dato", "date"),
    ], rader, "Akkurat det notebooken leste fra Input-mappen, én rad per bil per kontroll.")


def _topp(teller: Counter, n: int = 10, annet: str = "Andre") -> list:
    """Topp-n kategorier, resten slatt sammen til en 'Andre'-rad."""
    data = [{"label": k, "value": v} for k, v in teller.most_common(n)]
    rest = sum(teller.values()) - sum(d["value"] for d in data)
    if rest > 0:
        data.append({"label": annet, "value": rest, "muted": True})
    return data


def _bokser(verdier: list, grenser: list) -> list:
    """Teller verdier i intervaller. grenser = [(etikett, min, maks), ...]."""
    return [{"label": etikett, "value": sum(1 for v in verdier if lo <= v <= hi)}
            for etikett, lo, hi in grenser]


def _unike_kjoretoy(bundles: list) -> dict:
    """Ett representativt OFV-rad per kjoretoy (pa tvers av kontrollene)."""
    biler = {}
    for b in bundles:
        for r in b["all_rows"]:
            if r.get("Status") != "OK":
                continue
            key = normalize_identifier(r.get("ChassisNumber")) or normalize_identifier(r.get("RegNo"))
            if key and key not in biler:
                biler[key] = r
    return biler


def _unike_transaksjoner(bundles: list) -> list:
    sett, ut = set(), []
    for b in bundles:
        for r in b["all_rows"]:
            if r.get("Status") != "OK" or not r.get("TransactionDate"):
                continue
            key = (r.get("TransactionNumber"), r.get("ChassisNumber"), r.get("TransactionDate"))
            if key not in sett:
                sett.add(key)
                ut.append(r)
    return ut


def _analyse(bundles: list) -> dict:
    """Grafer fra Databricks-dataene. Grafene for Solgte biler (og deres
    rad i "Status per kontroll") lages i nettleseren, fordi de ogsa skal
    ta med innlimte datasett."""
    grafer = []

    # 1. Status per kontroll (100 % stablet) - Solgte biler legges til i HTML
    stabel = []
    for b in bundles:
        if b["kontroll"] == "Varekjop bruktbil":
            t = Counter(_status_tone(r["Status"])
                        for r in b["seksjon_a"] + b["seksjon_b"] + b["seksjon_c"])
            seg = [("OK", t["g"], "g"), ("Mangler bokføring", t["y"], "y"), ("Avvik", t["r"], "r")]
        elif b["kontroll"] == "Kontroll Demobil":
            t = Counter(_status_tone(r["Status"]) for r in b["kontroll_rows"])
            seg = [("OK", t["g"], "g"), ("Avvik", t["r"], "r"), ("Ingen treff", t["n"], "n")]
        else:
            continue
        stabel.append({"label": _VISNINGSNAVN.get(b["kontroll"], b["kontroll"]),
                       "segments": [{"label": l, "value": v, "tone": tone} for l, v, tone in seg]})
    grafer.append({"type": "stack", "tittel": "Status per kontroll",
                   "undertittel": "Andel biler per utfall - hold musen over for antall",
                   "rows": stabel, "bred": True})

    for b in bundles:
        if b["kontroll"] == "Varekjop bruktbil":
            rows = b["seksjon_a"] + b["seksjon_b"]
            lagertid = [_dager_mellom(r["FraDato"], r["TilDato"]) for r in rows
                        if r["FraDato"] and r["TilDato"]]
            if lagertid:
                grafer.append({"type": "columns", "tittel": "Varekjøp: lagertid for solgte biler",
                               "undertittel": "Dager fra forhandlerens kjøp til videresalg",
                               "data": _bokser(lagertid, [
                                   ("0-30", 0, 30), ("31-60", 31, 60), ("61-90", 61, 90),
                                   ("91-180", 91, 180), ("181-365", 181, 365),
                                   ("> 1 år", 366, 10 ** 9)])})
            kjopt_fra = Counter(r["KjoptFra"] for r in rows if r["KjoptFra"])
            if kjopt_fra:
                grafer.append({"type": "hbar", "tittel": "Varekjøp: hvor kommer bilene fra",
                               "undertittel": "Selger i forhandlerens kjøp (topp 10)",
                               "data": _topp(kjopt_fra)})
            solgt_til = Counter(r["Kjoper"] for r in rows if r["TilDato"] and r["Kjoper"])
            if solgt_til:
                grafer.append({"type": "hbar", "tittel": "Varekjøp: hvem kjøper bilene videre",
                               "undertittel": "Kjøper ved videresalg (topp 10)",
                               "data": _topp(solgt_til)})

        if b["kontroll"] == "Kontroll Demobil":
            kjopere = Counter(r["SisteKjoper"] for r in b["kontroll_rows"]
                              if r["Status"].startswith("Avvik") and r["SisteKjoper"])
            if kjopere:
                grafer.append({"type": "hbar", "tittel": "Demobil: hvem har overtatt bilene",
                               "undertittel": "Siste kjøper for demobiler med avvik (topp 10)",
                               "data": _topp(kjopere)})
            alder = [_dager_mellom(parse_bokfort_dato(r["BokfortInnDato"]), date.today())
                     for r in b["kontroll_rows"] if r["BokfortInnDato"]]
            if alder:
                grafer.append({"type": "columns", "tittel": "Demobil: tid siden bokført inn",
                               "undertittel": "Antall demobiler etter alder i dager",
                               "data": _bokser(alder, [
                                   ("0-90", 0, 90), ("91-180", 91, 180), ("181-365", 181, 365),
                                   ("1-2 år", 366, 730), ("> 2 år", 731, 10 ** 9)])})

    # Pa tvers av kontrollene
    biler = _unike_kjoretoy(bundles)
    if biler:
        grafer.append({"type": "hbar", "tittel": "Merke",
                       "undertittel": f"Unike kjøretøy med OFV-treff (n = {len(biler)})",
                       "data": _topp(Counter((r.get("MakeName") or "Ukjent") for r in biler.values()))})
        grafer.append({"type": "hbar", "tittel": "Drivstoff",
                       "undertittel": "Unike kjøretøy med OFV-treff",
                       "data": _topp(Counter((r.get("FuelGroup") or "Ukjent") for r in biler.values()), 7)})

    tx = _unike_transaksjoner(bundles)
    if tx:
        siste = max(r["TransactionDate"] for r in tx)
        maneder = []
        y, m = siste.year, siste.month
        for _ in range(24):
            maneder.append((y, m))
            y, m = (y, m - 1) if m > 1 else (y - 1, 12)
        maneder.reverse()
        teller = Counter((r["TransactionDate"].year, r["TransactionDate"].month) for r in tx)
        grafer.append({"type": "columns", "tittel": "Eierskifter per måned",
                       "undertittel": "Unike OFV-transaksjoner for alle kontrollerte biler, siste 24 måneder",
                       "data": [{"label": f"{mm:02d}.{str(yy)[2:]}", "value": teller[(yy, mm)]}
                                for yy, mm in maneder], "bred": True})
        grafer.append({"type": "hbar", "tittel": "Kjøpertype i eierskifter",
                       "undertittel": "Eiertype på kjøpersiden, alle unike transaksjoner",
                       "data": _topp(Counter((r.get("ToOwnerType") or "Ukjent") for r in tx), 6)})

    return {"grafer": grafer, "unikeKjoretoy": len(biler), "unikeEierskifter": len(tx)}


def bygg_rapportdata(bundles: list) -> dict:
    byggere = {
        "Varekjop bruktbil": _fane_varekjop,
        "Kontroll Demobil": _fane_demobil,
    }
    solgte = next((b for b in bundles if b["kontroll"] == "Kontroll solgte biler"), None)
    faner = [_fane_solgte_biler(solgte)]
    faner += [byggere[b["kontroll"]](b) for b in bundles if b["kontroll"] in byggere]
    antall = {b["kontroll"]: b["antall_biler"] for b in bundles}
    for f in faner:
        f["meta"] = [{"label": k, "verdi": v} for k, v in f["meta"] if v]
        f["antallBiler"] = next(
            (v for k, v in antall.items() if _VISNINGSNAVN.get(k) == f["tittel"]), 0)

    return {
        "selskap": SELSKAP,
        "generert": datetime.now().strftime("%d.%m.%Y kl. %H:%M"),
        "tidsbruk": round(_elapsed_s, 1),
        "grenser": {"gronnMaks": AVVIK_GRONN_MAX, "gulMaks": AVVIK_GUL_MAX},
        "api": {
            "ofvKjoretoy": API_TELLER["ofv_kjoretoy"],
            "ofvBuffer": API_TELLER["ofv_buffer"],
            "ofvHttp": API_TELLER["ofv_http"],
            "svvHttp": API_TELLER["svv_http"],
        },
        "oversikt": [{
            "id": f["id"], "tittel": f["fullTittel"], "meta": f["meta"],
            "kpier": f["kpier"],
        } for f in faner],
        "input": _input_tabell(bundles),
        "faner": faner,
        "analyse": _analyse(bundles),
    }


# ------------------------------------------------------------------
# Innebygde ressurser (Excel-leser og fonter) - leses fra RESSURS_DIR,
# lastes ned og lagres der forste gang hvis de mangler.
# ------------------------------------------------------------------

RESSURSER = {
    "xlsx.full.min.js":
        "https://cdnjs.cloudflare.com/ajax/libs/xlsx/0.18.5/xlsx.full.min.js",
    "work-sans-latin-wght-normal.woff2":
        "https://cdn.jsdelivr.net/npm/@fontsource-variable/work-sans@5/files/work-sans-latin-wght-normal.woff2",
    "ibm-plex-mono-latin-500-normal.woff2":
        "https://cdn.jsdelivr.net/npm/@fontsource/ibm-plex-mono@5/files/ibm-plex-mono-latin-500-normal.woff2",
    "ibm-plex-mono-latin-600-normal.woff2":
        "https://cdn.jsdelivr.net/npm/@fontsource/ibm-plex-mono@5/files/ibm-plex-mono-latin-600-normal.woff2",
}


def hent_ressurs(navn: str):
    """Innholdet i RESSURS_DIR/navn, eller None. Mangler filen, lastes
    den ned fra CDN og lagres i RESSURS_DIR til neste gang."""
    sti = os.path.join(RESSURS_DIR, navn)
    if os.path.exists(sti):
        with open(sti, "rb") as f:
            return f.read()
    try:
        resp = requests.get(RESSURSER[navn], timeout=HTTP_TIMEOUT_S)
        if resp.status_code == 200 and resp.content:
            try:
                os.makedirs(RESSURS_DIR, exist_ok=True)
                with open(sti, "wb") as f:
                    f.write(resp.content)
            except OSError as exc:
                print(f"  OBS: klarte ikke a lagre {navn} i {RESSURS_DIR} ({exc})")
            return resp.content
        print(f"  OBS: {navn} - HTTP {resp.status_code} fra {RESSURSER[navn]}")
    except requests.RequestException as exc:
        print(f"  OBS: klarte ikke a laste ned {navn} ({exc})")
    return None


def _fonter_html() -> str:
    """@font-face med innebygde fonter, ellers lenke til Google Fonts."""
    filer = [("Work Sans", "work-sans-latin-wght-normal.woff2", "100 900"),
             ("IBM Plex Mono", "ibm-plex-mono-latin-500-normal.woff2", "500"),
             ("IBM Plex Mono", "ibm-plex-mono-latin-600-normal.woff2", "600")]
    css = []
    for familie, navn, vekt in filer:
        data = hent_ressurs(navn)
        if data is None:
            print("  OBS: fonter er ikke innebygd - rapporten henter dem fra Google Fonts.")
            return ('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?'
                    'family=Work+Sans:wght@400;500;600;700;800&family=IBM+Plex+Mono:'
                    'wght@500;600&display=swap">')
        b64 = base64.b64encode(data).decode("ascii")
        css.append(f'@font-face{{font-family:"{familie}";font-style:normal;font-weight:{vekt};'
                   f'font-display:swap;src:url(data:font/woff2;base64,{b64}) format("woff2")}}')
    return "<style>\n" + "\n".join(css) + "\n</style>"


def _xlsx_html() -> str:
    """SheetJS (Excel-opplasting i tannhjul-panelet) innebygd i filen."""
    data = hent_ressurs("xlsx.full.min.js")
    if data is None:
        print("  OBS: Excel-leseren er ikke innebygd - .xlsx-opplasting krever nett.")
        return f'<script src="{RESSURSER["xlsx.full.min.js"]}"></script>'
    js = data.decode("utf-8")
    if re.search(r"</script|<script", js, re.IGNORECASE):
        b64 = base64.b64encode(data).decode("ascii")
        return f'<script src="data:text/javascript;base64,{b64}"></script>'
    return "<script>\n" + js + "\n</script>"


def skriv_html_rapport(bundles: list, path: str) -> None:
    data = bygg_rapportdata(bundles)
    # "</" escapes sa JSON-innholdet aldri kan avslutte <script>-taggen.
    data_json = json.dumps(data, ensure_ascii=False).replace("</", "<\\/")
    tittel = html.escape(f"OFV-kontroller - {SELSKAP}")
    # Rekkefolgen er viktig: __DATA__ fylles inn sist, sa tekst i dataene
    # aldri kan tolkes som en plassholder.
    side = (_HTML_MAL
            .replace("__TITTEL__", tittel)
            .replace("__FONTER__", _fonter_html())
            .replace("__XLSX__", _xlsx_html()))
    side = side.replace("__DATA__", data_json)
    with open(path, "w", encoding="utf-8") as f:
        f.write(side)


# ------------------------------------------------------------------
# HTML/CSS/JS-mal - __TITTEL__ og __DATA__ fylles inn av skriv_html_rapport
# ------------------------------------------------------------------

_HTML_MAL = r"""<!DOCTYPE html>
<html lang="no">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITTEL__</title>
__FONTER__
<style>
  /* Fargetokens hentet fra BDOs opplastede fil (farger malt pa bdo.no).
     Skriften proxima-nova er BDOs lisensierte Adobe Fonts-font og kan
     ikke bygges inn her - Work Sans brukes som naermeste erstatning. */
  :root{
    --rod:#E81A3B; --blaa:#0062B8; --skifer:#5B6E7F; --tekst:#333333; --dempet:#666666;
    --kant:#E5E7EB; --flate:#F2F2F2; --flate2:#E7E7E7; --bg:#FFFFFF;
    --gronn:#009966; --gul:#D67900; --gul-lys:#FFF1DC; --gronn-lys:#E3F3EC; --rod-lys:#FDE7EB;
    --glid:cubic-bezier(.2,.7,.1,1); --myk:cubic-bezier(.4,0,.2,1);
    --kort:.2s; --mid:.4s; --lang:.7s;
    --bil:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 13'%3E%3Cpath fill-rule='evenodd' d='M1.5 8.2C1.5 6.8 2.3 6 3.6 5.7L6.6 5.1L9.6 2C10.1 1.5 10.8 1.2 11.5 1.2L16.4 1.2C17.1 1.2 17.7 1.5 18.1 2.1L20.1 4.9L21.6 5.3C22.5 5.6 23 6.3 23 7.2L23 9.3C23 9.8 22.6 10.2 22.1 10.2L20.7 10.2A2.3 2.3 0 0 0 16.1 10.2L8.1 10.2A2.3 2.3 0 0 0 3.5 10.2L2.3 10.2C1.9 10.2 1.5 9.8 1.5 9.4ZM10.8 2.5L8.9 4.8L13.5 4.8L13.5 2.5ZM14.5 2.5L14.5 4.8L18.7 4.8L17.1 2.5Z'/%3E%3Ccircle cx='5.8' cy='10.6' r='1.9'/%3E%3Ccircle cx='18.4' cy='10.6' r='1.9'/%3E%3C/svg%3E");
    color-scheme:light;
  }
  *{box-sizing:border-box}
  html,body{margin:0;background:#DADFE3}
  body{font-family:"Work Sans","Helvetica Neue",Helvetica,Arial,sans-serif;color:var(--tekst);-webkit-font-smoothing:antialiased}
  h1,h2,h3{margin:0;text-wrap:balance}
  [hidden]{display:none!important}
  button{font:inherit;color:inherit}
  .num{font-family:"IBM Plex Mono",ui-monospace,monospace;font-variant-numeric:tabular-nums}

  /* ============================================================
     DEL A - faste "scener" (1600x900, skalert, ingen scrolling),
     samme mekanikk som BDO-filen.
  ============================================================ */
  #scenehost{position:fixed;left:0;top:0;width:1600px;height:900px;transform-origin:0 0;background:var(--bg);overflow:hidden;box-shadow:0 0 0 1px var(--kant)}
  .scene-side{position:absolute;inset:0;padding:0}

  .topp{position:absolute;left:0;top:0;width:100%;height:72px;background:#fff;display:flex;align-items:center;padding:0 56px;gap:14px;border-bottom:1px solid var(--kant)}
  .bdo-logo{height:32px;width:auto;display:block}
  .topp .enhet{font-size:12px;font-weight:700;letter-spacing:.06em;text-transform:uppercase;color:var(--rod);margin-left:2px}

  .tekstblokk{position:absolute;left:56px;top:120px;width:820px}
  .eyebrow{font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:.08em;color:var(--rod);margin:0 0 10px}
  .tittel{font-weight:800;font-size:38px;line-height:1.05;letter-spacing:-.005em}
  .tittel .l2{display:block;font-weight:400;color:var(--skifer)}
  .ingress{font-size:16px;font-weight:400;color:var(--dempet);margin:12px 0 0;line-height:1.35;max-width:70ch}

  .sideliste{position:absolute;left:930px;top:128px;width:614px;list-style:none;margin:0;padding:0;display:flex;flex-direction:column;gap:11px}
  .sideliste li{display:grid;grid-template-columns:26px 1fr;font-size:17px;font-weight:400;line-height:1.28}
  .sideliste li b{font-weight:700}
  .sideliste .nr{font-weight:700;font-size:11px;padding-top:4px;color:var(--rod)}

  /* før/etter */
  .fe-rad2{position:absolute;left:56px;top:300px;width:1488px;height:470px}
  .fe-merke2{font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;color:var(--dempet)}
  .fe-rutenett2{position:absolute;left:0;top:32px;display:grid;grid-template-columns:repeat(20,40px);gap:8px 10px;width:1050px}
  .bilag2{width:40px;height:23px;background:#C9CFD5;-webkit-mask:var(--bil) center/contain no-repeat;mask:var(--bil) center/contain no-repeat;opacity:0;transform:translateY(4px);transition:opacity .5s var(--myk),transform .55s var(--myk),background .3s;transition-delay:var(--d,0s)}
  .fe-rutenett2.spill .bilag2{opacity:.45;transform:none}
  .fe-rutenett2.spill .bilag2.utvalg{opacity:1;background:var(--skifer)}
  .fe-tallboks{position:absolute;right:0;top:32px;width:382px;border-top:3px solid var(--rod);padding:20px 0 0}
  .fe-tallboks .lab{font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;color:var(--dempet)}
  .fe-tallboks .stor{font-size:84px;font-weight:800;line-height:1;margin-top:12px;letter-spacing:-.02em}
  .fe-tallboks .under{font-size:15px;margin-top:8px;color:var(--dempet);line-height:1.3}
  .fe-tallboks .skille{height:1px;background:var(--kant);margin:20px 0 16px}
  .fe-tallboks .stor2{font-size:32px;font-weight:800;line-height:1.15}
  .fe-tallboks .under2{font-size:14px;margin-top:6px;color:var(--dempet)}

  /* datakilder / ofv-kart */
  .kart-rad{position:absolute;left:56px;top:290px;width:1488px}
  .kilde3{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}
  .kilde3 .boks{border:1px solid var(--kant);background:#fff;padding:14px 18px;min-height:96px}
  .kilde3 .boks.midt{border:1.5px dashed var(--skifer)}
  .boks .sone{font-size:10.5px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;color:var(--dempet)}
  .boks h3{font-size:19px;font-weight:700;margin-top:4px;display:flex;align-items:center;gap:8px}
  .boks-logo{height:20px;width:auto;max-width:90px;object-fit:contain;display:block}
  .boks .sub{font-size:12.5px;color:var(--dempet);margin-top:3px;line-height:1.3}
  .boks .antall{font-size:12.5px;font-weight:700;margin-top:10px;display:inline-block;border:1px solid;padding:5px 10px}
  .boks .antall-vent{color:var(--dempet);border-color:var(--kant)}
  .boks .antall-ferdig{color:var(--tekst);border-color:var(--tekst)}

  .bilkortrad2{display:grid;grid-template-columns:repeat(7,200px);gap:10px;margin-top:22px;align-items:start}
  .bilkort2{border:1px solid var(--kant);background:#fff;padding:8px 10px;opacity:0;transform:translateY(6px);transition:opacity .4s var(--myk),transform .45s var(--myk),border-color .2s;transition-delay:var(--d,0s)}
  .bilkortrad2.spill .bilkort2{opacity:1;transform:none}
  .bilkort2.g{border-color:var(--gronn)} .bilkort2.y{border-color:var(--gul)} .bilkort2.r{border-color:var(--rod)}
  .bilkort2.aktiv{box-shadow:0 0 0 3px var(--gul);border-color:var(--gul)}
  .bilkort2 .hode{display:flex;justify-content:space-between;gap:4px;border-bottom:1px solid var(--kant);padding-bottom:4px}
  .bilkort2 .regnr{font-size:11.5px;font-weight:800;font-family:"IBM Plex Mono",monospace}
  .bilkort2 .modell{font-size:9.5px;color:var(--dempet);white-space:nowrap;overflow:hidden;text-overflow:ellipsis;max-width:100px}
  .bilkort2 .bokf{font-size:9.5px;color:var(--dempet);margin-top:4px}
  .histliste2{margin-top:4px;display:flex;flex-direction:column;gap:3px}
  .hist-linje{font-size:8.5px;padding:2px 5px;border-radius:4px;display:flex;justify-content:space-between;gap:4px;color:var(--dempet);opacity:0;transition:opacity .3s var(--myk)}
  .bilkortrad2.spill .hist-linje{opacity:1}
  .hist-linje .p{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
  .hist-linje.leser{background:#FFC97A;color:var(--tekst);box-shadow:inset 4px 0 0 var(--gul);font-weight:700;opacity:1;transform:scale(1.03);transform-origin:left center}
  .hist-linje.forbi{opacity:.45}
  .hist-linje.treff{font-weight:700;color:var(--tekst)}
  .hist-linje .a{font-weight:800}
  .hist-linje.treff.g{background:var(--gronn-lys)} .hist-linje.treff.g .a{color:var(--gronn)}
  .hist-linje.treff.y{background:var(--gul-lys)} .hist-linje.treff.y .a{color:var(--gul)}
  .hist-linje.treff.r{background:var(--rod-lys)} .hist-linje.treff.r .a{color:var(--rod)}

  .tellere2{display:flex;gap:14px;font-size:12px;margin-top:9px}
  .tellere2 span{display:flex;align-items:center;gap:5px}
  .tellere2 b{font-size:16px;font-weight:800;font-family:"IBM Plex Mono",monospace}
  .sw{width:9px;height:9px;border-radius:50%;flex:none}
  .sw.g{background:var(--gronn)} .sw.y{background:var(--gul)} .sw.r{background:var(--rod)}

  /* forside / spørsmål (helside) */
  .helside{position:absolute;inset:0;background-size:cover;background-position:center;background-color:#fff}
  .helside .duk{position:absolute;inset:0;background:linear-gradient(100deg,#fff 46%,rgba(255,255,255,.55) 62%,rgba(255,255,255,0) 82%)}
  .helside .innhold{position:absolute;left:64px;top:270px;width:680px}
  .helside h1{font-size:56px;font-weight:800;line-height:1.02;letter-spacing:-.01em}
  .helside h1 .l2{display:block;font-weight:400;color:var(--skifer)}
  .helside .ingress2{font-size:19px;line-height:1.35;margin:22px 0 0;max-width:520px;color:var(--dempet)}
  .stor-knapp{margin-top:30px;background:var(--rod);color:#fff;border:1px solid var(--rod);padding:13px 30px;font-size:15px;font-weight:700;text-transform:uppercase;letter-spacing:.03em;cursor:pointer}
  .helside .meta{position:absolute;left:64px;bottom:40px;font-size:13px;color:var(--dempet)}

  /* ============================================================
     Bunnlinje (utenfor den skalerte scenen - fast fysisk størrelse)
  ============================================================ */
  #bunn{position:fixed;left:0;right:0;bottom:0;height:64px;background:#fff;border-top:1px solid var(--kant);display:flex;align-items:center;justify-content:space-between;padding:0 20px;z-index:30;gap:12px}
  #bunn .merknad{font-size:11.5px;color:var(--dempet);max-width:360px;line-height:1.25;display:none}
  @media (min-width:900px){#bunn .merknad{display:block}}
  .prikknav{display:flex;align-items:center;gap:7px;flex:none}
  .prikknav button{width:9px;height:9px;border-radius:50%;border:1.5px solid #B4BCC3;background:#fff;padding:0;cursor:pointer}
  .prikknav button.passert{background:#B4BCC3;border-color:#B4BCC3}
  .prikknav button[aria-current="true"]{background:var(--rod);border-color:var(--rod);transform:scale(1.3)}
  .styr{display:flex;gap:8px;flex:none}
  .styr button{padding:9px 16px;font-size:12.5px;font-weight:700;text-transform:uppercase;letter-spacing:.03em;cursor:pointer;border:1px solid var(--tekst);background:#fff}
  .styr .frem{background:var(--rod);color:#fff;border-color:var(--rod)}
  .styr button:disabled{opacity:.3;cursor:default}

  /* ============================================================
     DEL B - frie, scrollbare sider (Oversikt per forhandler, Resultat)
  ============================================================ */
  #frihost{padding-top:72px;padding-bottom:78px;min-height:100vh;background:var(--flate)}
  .fri-side{max-width:1100px;margin:0 auto;padding:40px 20px 20px}
  .fri-topp{position:fixed;left:0;top:0;right:0;height:72px;background:#fff;display:flex;align-items:center;padding:0 20px;gap:14px;border-bottom:1px solid var(--kant);z-index:25}
  .fri-topp .bdo-logo{height:28px}
  .fri-topp .tittel2{font-weight:800;font-size:15px;margin-left:2px}

  .fri-side h2{font-size:28px;font-weight:800;letter-spacing:-.01em;color:var(--tekst)}
  .fri-side .sub{color:var(--dempet);font-size:14.5px;margin-top:8px;max-width:70ch;line-height:1.4}

  .kpirad{display:grid;grid-template-columns:repeat(auto-fit,minmax(126px,1fr));gap:12px;margin-top:22px}
  .kpi{border:1px solid var(--kant);background:#fff;padding:14px 16px}
  .kpi .lab{font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.05em;color:var(--dempet)}
  .kpi .tall{font-size:27px;font-weight:800;margin-top:6px}
  .kpi .pst{font-size:12px;color:var(--dempet);margin-top:2px}
  .kpi.g{border-top:3px solid var(--gronn)} .kpi.g .tall{color:var(--gronn)}
  .kpi.y{border-top:3px solid var(--gul)} .kpi.y .tall{color:var(--gul)}
  .kpi.r{border-top:3px solid var(--rod)} .kpi.r .tall{color:var(--rod)}
  .kpi.n{border-top:3px solid var(--skifer)} .kpi.n .tall{color:var(--skifer)}

  .filterrad{display:flex;align-items:center;gap:10px;margin-top:22px;flex-wrap:wrap;background:#fff;border:1px solid var(--kant);padding:12px 14px}
  .filterrad label{font-size:12px;font-weight:700;color:var(--dempet);text-transform:uppercase;letter-spacing:.04em}
  .filterrad select, .filterrad input[type="text"]{font:inherit;font-size:13px;padding:8px 10px;border-radius:6px;border:1px solid var(--kant);background:#fff;color:var(--tekst)}
  .bukettbrytere{display:flex;gap:6px;flex-wrap:wrap}
  .bukettknapp{border:1px solid var(--kant);background:#fff;padding:6px 11px;font-size:12.5px;font-weight:700;border-radius:99px;cursor:pointer;display:flex;align-items:center;gap:6px}
  .bukettknapp .prikk{width:8px;height:8px;border-radius:50%}
  .bukettknapp[aria-pressed="false"]{opacity:.4}
  .bukettknapp.g .prikk{background:var(--gronn)} .bukettknapp.y .prikk{background:var(--gul)} .bukettknapp.r .prikk{background:var(--rod)} .bukettknapp.n .prikk{background:var(--skifer)}
  .antall-treff{font-size:12.5px;color:var(--dempet);margin-left:auto}

  .tabellramme{margin-top:16px;overflow-x:auto;border:1px solid var(--kant);background:#fff}
  table.tabell{width:100%;border-collapse:collapse;font-size:13px}
  table.tabell th{position:sticky;top:0;background:#fff;text-align:left;font-size:10.5px;font-weight:700;text-transform:uppercase;letter-spacing:.04em;color:var(--dempet);padding:9px 11px;border-bottom:2px solid var(--tekst);white-space:nowrap;cursor:pointer;user-select:none}
  table.tabell td{padding:8px 11px;border-bottom:1px solid var(--kant);white-space:nowrap}
  table.tabell tr:hover td{background:var(--flate)}
  .badge{display:inline-flex;align-items:center;gap:6px;font-weight:700;font-size:12px;padding:3px 9px;border-radius:99px}
  .badge .dot{width:7px;height:7px;border-radius:50%}
  .badge.g{background:var(--gronn-lys);color:var(--gronn)} .badge.g .dot{background:var(--gronn)}
  .badge.y{background:var(--gul-lys);color:var(--gul)} .badge.y .dot{background:var(--gul)}
  .badge.r{background:var(--rod-lys);color:var(--rod)} .badge.r .dot{background:var(--rod)}
  .badge.n{background:var(--flate2);color:var(--skifer)} .badge.n .dot{background:var(--skifer)}
  td.reg{font-weight:700;font-family:"IBM Plex Mono",monospace}

  .forhandlertabell{width:100%;border-collapse:collapse;margin-top:22px;background:#fff;border:1px solid var(--kant);font-size:13.5px}
  .forhandlertabell th{text-align:left;font-size:10.5px;font-weight:700;text-transform:uppercase;letter-spacing:.04em;color:var(--dempet);padding:10px 12px;border-bottom:2px solid var(--tekst)}
  .forhandlertabell td{padding:10px 12px;border-bottom:1px solid var(--kant)}
  .forhandlertabell tr.rad{cursor:pointer}
  .forhandlertabell tr.rad:hover td{background:var(--flate)}
  .forhandlertabell tr.totalrad td{font-weight:800;border-top:2px solid var(--tekst);border-bottom:none}
  .minisoyle{display:flex;height:8px;width:140px;border-radius:4px;overflow:hidden;background:var(--flate2)}
  .minisoyle span{height:100%}
  .minisoyle .g{background:var(--gronn)} .minisoyle .y{background:var(--gul)} .minisoyle .r{background:var(--rod)} .minisoyle .n{background:var(--skifer)}

  .tomvarsel{margin-top:26px;border:1px dashed var(--kant);padding:30px 22px;text-align:center;color:var(--dempet);font-size:14px;background:#fff}

  /* ============================================================
     Tannhjul + admin-panel (skjult input-side)
  ============================================================ */
  #tannhjul{position:fixed;right:16px;bottom:80px;z-index:41;width:38px;height:38px;border-radius:50%;border:1px solid var(--kant);background:#fff;color:var(--dempet);font-size:16px;cursor:pointer;opacity:.5}
  #tannhjul:hover{opacity:1}
  #adminBakteppe{position:fixed;inset:0;background:rgba(10,12,16,.5);z-index:49;display:flex;align-items:flex-end;justify-content:center}
  @media (min-width:760px){#adminBakteppe{align-items:center}}
  #adminPanel{position:relative;z-index:50;background:#fff;color:var(--tekst);width:100%;max-width:720px;max-height:88vh;overflow-y:auto;padding:24px 22px 32px;border:1px solid var(--kant)}
  #adminPanel h3{font-size:20px;font-weight:800}
  #adminPanel .lukk{position:absolute;right:14px;top:14px;border:0;background:none;font-size:22px;cursor:pointer;color:var(--dempet);padding:4px 8px}
  .admin-seksjon{margin-top:24px;padding-top:18px;border-top:1px solid var(--kant)}
  .admin-seksjon:first-of-type{margin-top:18px}
  .admin-seksjon h4{font-size:13px;font-weight:700;text-transform:uppercase;letter-spacing:.05em;color:var(--dempet)}
  .bilde-rad{display:flex;align-items:center;gap:12px;margin-top:10px;flex-wrap:wrap}
  .bilde-forhandsvisning{width:64px;height:40px;object-fit:cover;border:1px solid var(--kant);background:var(--flate)}
  .forhandlerslot{border:1px solid var(--kant);padding:14px 16px;margin-top:12px;position:relative}
  .forhandlerslot input[type="text"]{font:inherit;font-size:13.5px;font-weight:700;padding:8px 10px;border-radius:6px;border:1px solid var(--kant);width:100%;max-width:320px}
  textarea.limfelt{width:100%;min-height:76px;resize:vertical;border:1px solid var(--kant);background:var(--flate);color:var(--tekst);font:12.5px/1.4 "IBM Plex Mono",monospace;padding:9px;margin-top:8px}
  .forhandlerslot .fjernslot{position:absolute;right:10px;top:10px;border:0;background:none;color:var(--dempet);cursor:pointer;font-size:13px}
  .slot-status{font-size:12px;color:var(--dempet);margin-top:6px;min-height:15px}
  .slot-status.ok{color:var(--gronn)} .slot-status.feil{color:var(--rod)}
  input[type="file"]{font-size:13px;max-width:100%}
  .personvern{font-size:12px;color:var(--dempet);margin-top:16px;display:flex;gap:8px;align-items:flex-start}

  @media (prefers-reduced-motion:reduce){ *,*::before,*::after{transition-duration:.01ms!important} }

  /* ============================================================
     DEL C - rapportsidene (OFV-kontrollene fra Databricks) i samme
     BDO-stil som presentasjonen over.
  ============================================================ */
  .fri-side.bred{max-width:1440px}
  .fri-topp{gap:18px}
  .fri-topp .tittel2{white-space:nowrap}
  .fri-topp .selskap{font-size:13px;color:var(--dempet);white-space:nowrap}
  .rapportnav{display:flex;align-self:stretch;margin-left:18px;overflow-x:auto;scrollbar-width:none;flex:1;min-width:0}
  .rapportnav button{border:0;background:none;padding:0 14px;font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:.04em;color:var(--dempet);cursor:pointer;border-bottom:3px solid transparent;white-space:nowrap;display:flex;align-items:center;gap:7px}
  .rapportnav button:hover{color:var(--tekst)}
  .rapportnav button[aria-current="true"]{color:var(--tekst);border-bottom-color:var(--rod)}
  .navtall{font-family:"IBM Plex Mono",ui-monospace,monospace;font-size:11px;padding:1px 6px;border-radius:99px;background:var(--rod-lys);color:var(--rod)}
  .navtall.null{background:var(--gronn-lys);color:var(--gronn)}
  .knapp{font:inherit;padding:8px 14px;font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:.03em;cursor:pointer;border:1px solid var(--tekst);background:#fff;color:var(--tekst);white-space:nowrap}
  .knapp:hover{background:var(--flate)}
  .knapp.liten{padding:6px 11px;font-size:11.5px}
  .knapp.rolig{border-color:var(--kant);color:var(--dempet)}
  .knapp.rolig:hover{color:var(--tekst);border-color:var(--tekst)}
  .btn{font:inherit;font-size:12px;font-weight:700;padding:6px 11px;cursor:pointer;border:1px solid var(--kant);background:#fff;color:var(--tekst)}
  .btn:hover{border-color:var(--tekst)}
  #tilRapport{display:none}
  #bunn.scene #tilRapport{display:inline-block}

  .metarad{display:flex;flex-wrap:wrap;gap:8px;margin-top:16px}
  .metarad span{border:1px solid var(--kant);background:#fff;padding:5px 10px;font-size:12.5px;color:var(--dempet);max-width:100%;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
  .metarad b{color:var(--tekst);font-weight:700}
  details.regel{margin-top:14px;max-width:80ch}
  details.regel summary{cursor:pointer;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;color:var(--dempet)}
  details.regel p{margin:8px 0 0;font-size:14px;line-height:1.45;color:var(--dempet)}
  .kjoreinfo{display:flex;flex-wrap:wrap;gap:6px 26px;margin-top:16px;font-size:13px;color:var(--dempet)}
  .kjoreinfo b{color:var(--tekst)}

  button.kpi{font:inherit;text-align:left;cursor:pointer;color:inherit}
  button.kpi:hover{border-color:var(--tekst)}
  .kpi.valgt{outline:2px solid var(--tekst);outline-offset:-2px}
  .kpi .lab{display:flex;gap:6px;align-items:center}

  .oversiktgrid{display:grid;grid-template-columns:repeat(auto-fill,minmax(340px,1fr));gap:16px;margin-top:24px}
  .ovkort{border:1px solid var(--kant);border-top:3px solid var(--rod);background:#fff;padding:18px 20px;display:flex;flex-direction:column;gap:14px}
  .ovkort h3{font-size:18px;font-weight:800}
  .ovkort .metarad{margin-top:0}
  .minirad{display:grid;grid-template-columns:repeat(auto-fill,minmax(96px,1fr));gap:8px}
  .mini{background:var(--flate);padding:9px 11px}
  .mini .l{font-size:10.5px;font-weight:700;text-transform:uppercase;letter-spacing:.04em;color:var(--dempet);display:flex;gap:5px;align-items:center;line-height:1.25}
  .mini .v{font-size:21px;font-weight:800;margin-top:3px}
  .mini.g .v{color:var(--gronn)} .mini.y .v{color:var(--gul)} .mini.r .v{color:var(--rod)} .mini.n .v{color:var(--skifer)}
  .ovkort .knapp{align-self:flex-start}

  .panel{border:1px solid var(--kant);background:#fff;margin-top:22px}
  .panel-hode{padding:16px 18px 6px}
  .panel-hode h3{font-size:16px;font-weight:800}
  .panel-hode h3 .ant{font-weight:500;color:var(--dempet);font-family:"IBM Plex Mono",ui-monospace,monospace;font-size:13px;margin-left:6px}
  .panel-hode p{font-size:13.5px;color:var(--dempet);margin:4px 0 0;line-height:1.4;max-width:110ch}
  details.panel>summary{list-style:none;cursor:pointer}
  details.panel>summary::-webkit-details-marker{display:none}
  details.panel>summary .panel-hode{display:flex;gap:12px;align-items:flex-start;padding-bottom:16px}
  details.panel>summary .pil{color:var(--rod);font-size:12px;padding-top:3px;transition:transform .15s}
  details.panel[open]>summary .pil{transform:rotate(90deg)}
  details.panel[open]>summary .panel-hode{padding-bottom:6px}

  .verktoylinje{display:flex;flex-wrap:wrap;gap:10px;align-items:center;padding:12px 18px}
  .sokefelt{position:relative;flex:1 1 240px;max-width:360px}
  .sokefelt input{width:100%;font:inherit;font-size:13px;padding:8px 10px 8px 30px;border-radius:6px;border:1px solid var(--kant);background:#fff;color:var(--tekst);outline:none}
  .sokefelt svg{position:absolute;left:10px;top:50%;transform:translateY(-50%);color:var(--dempet)}
  .sokefelt input:focus,.filterfelt input:focus,.filterfelt select:focus{border-color:var(--tekst)}
  .verktoylinje .fyll{flex:1}
  .bukettknapp .n{font-family:"IBM Plex Mono",ui-monospace,monospace;font-weight:600}
  .kolmeny{position:relative}
  .kolmeny>summary{list-style:none}
  .kolmeny>summary::-webkit-details-marker{display:none}
  .kolmeny .meny{position:absolute;right:0;top:calc(100% + 6px);z-index:15;background:#fff;border:1px solid var(--tekst);padding:8px;min-width:230px;max-height:340px;overflow:auto;box-shadow:0 10px 24px rgba(0,0,0,.12)}
  .kolmeny label{display:flex;gap:8px;padding:4px 6px;cursor:pointer;font-size:13px}
  .kolmeny label:hover{background:var(--flate)}

  .tabellramme.rull{margin-top:0;border-left:0;border-right:0;border-bottom:0;max-height:72vh;overflow:auto}
  table.tabell thead tr.filterfelt th{top:35px;background:#fff;padding:6px 8px 8px;cursor:default;text-transform:none;letter-spacing:0;font-weight:400;border-bottom:1px solid var(--kant)}
  table.tabell thead tr.etiketter th{z-index:3}
  table.tabell thead tr.filterfelt th{z-index:2}
  table.tabell th .sortpil{opacity:.35;margin-left:4px;font-size:9px}
  table.tabell th.sortert .sortpil{opacity:1;color:var(--rod)}
  .filterfelt input,.filterfelt select{width:100%;min-width:90px;font:inherit;font-size:12px;padding:5px 6px;border-radius:4px;border:1px solid var(--kant);background:#fff;color:var(--tekst);outline:none}
  .filterfelt .spenn{display:flex;gap:4px}
  .filterfelt .spenn input{min-width:68px}
  .filterfelt input[type="date"]{min-width:122px}
  .filterfelt .aktiv{border-color:var(--rod);background:#FFF8F9}
  table.tabell td{max-width:380px;overflow:hidden;text-overflow:ellipsis}
  table.tabell td.t-num{text-align:right;font-family:"IBM Plex Mono",ui-monospace,monospace;font-size:12.5px}
  table.tabell td.t-date{font-variant-numeric:tabular-nums}
  table.tabell td.t-status{max-width:none}
  table.tabell td.tom{color:#B4BCC3}
  table.tabell td.reg{font-weight:700;font-family:"IBM Plex Mono",ui-monospace,monospace;font-size:12.5px}
  .tabellfot{display:flex;gap:12px;align-items:center;padding:10px 18px;color:var(--dempet);font-size:12.5px;border-top:1px solid var(--kant)}
  .tabellfot .fyll{flex:1}
  .ingentreff{padding:28px;text-align:center;color:var(--dempet)}

  .grafgrid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px;margin-top:6px}
  .grafgrid .panel{margin-top:0}
  .grafgrid .bred2{grid-column:1/-1}
  @media (max-width:980px){.grafgrid{grid-template-columns:1fr}.oversiktgrid{grid-template-columns:1fr}}
  .grafkropp{padding:8px 18px 6px}
  .grafkropp svg{display:block;width:100%;overflow:visible}
  .grafkropp svg text{font-family:"Work Sans","Helvetica Neue",Helvetica,Arial,sans-serif}
  .forklaring{display:flex;flex-wrap:wrap;gap:6px 16px;padding:2px 18px 4px;font-size:12px;color:var(--dempet)}
  .forklaring span{display:inline-flex;gap:6px;align-items:center}
  .forklaring i{width:10px;height:10px;display:inline-block}
  details.talltabell{padding:0 18px 14px}
  details.talltabell summary{cursor:pointer;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.05em;color:var(--dempet)}
  details.talltabell table{margin-top:8px;font-size:12.5px;border-collapse:collapse}
  details.talltabell td{padding:4px 10px 4px 0;border-bottom:1px solid var(--kant)}
  details.talltabell td.n{text-align:right;font-family:"IBM Plex Mono",ui-monospace,monospace}
  .tips{position:fixed;pointer-events:none;z-index:60;background:var(--tekst);color:#fff;padding:7px 10px;font-size:12.5px;line-height:1.35;max-width:280px;opacity:0;transition:opacity .08s}
  .tips b{font-weight:700}
  .tips.vis{opacity:1}
  .databricks-datasett{border:1px solid var(--kant);border-left:3px solid var(--rod);padding:12px 16px;margin-top:12px;font-size:13.5px;background:var(--flate)}
  .databricks-datasett .sone{font-size:10.5px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;color:var(--dempet)}
  .fotnote{max-width:1440px;margin:26px auto 0;padding:0 20px;color:var(--dempet);font-size:12px}

  @media print{
    #bunn,#tannhjul,.rapportnav,.verktoylinje,thead tr.filterfelt,.tabellfot .knapp,#scenehost{display:none!important}
    #frihost{display:block!important;padding:0}
    .fri-topp{position:static}
    .fri-side{display:block!important;break-before:page}
    .tabellramme.rull{max-height:none;overflow:visible}
    details.panel>*:not(summary){display:block}
  }
</style>
</head>
<body>

<div id="scenehost">
  <div class="topp"><img class="bdo-logo" src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAARwAAABuCAYAAAD1aayeAAAACXBIWXMAAAsSAAALEgHS3X78AAAOiElEQVR4nO2dT04jORvGi1Hvg8RykKBOQL79oE5LtSd9AtKbbCd9ggknaNjWhnCCgX2kJuIAHU4QIrEcpM4J+OToMV3QSVEVP3bZzvuTopY+zRcqLvvx+8+vd56fnxNBEPwnzfJOxYeczsb9nz7+oJ3//vyrlyTJocF33O493t2aPkia5d0kSdqm3+MQ9UKn6t/ZuD9t6iHSLFdj1m3q77/DFOOUzMZ94zmyDaRZvot10MG6VJ+PG/70eZIkD2qN6n9n4/5Dk8P4IUmSnsEP0rAm0z+k73FKmuXqz00wDteOBehnCOOGMVpAhPSn8QXgA7BcuhCZI+IjHeDzsr7TLF/oedrE+CsL59ZQcM72Hu+GjIdJs/wBAxQ690mSnM/G/ZGL35Fmufo7p4GO2bww+a89eB4nwKLXn1aDj6Lmqpo/IxdumG+Co6ytS8Z3eYJaTEPbwgO36kdIA7OGBSb/eYyWD9ylAbwKHzfWG4y9NffXN8HZha/ZpOLbQO0iPZuuVprlpu/RNyYQ6+BjP2mWqziMWiOhWKHWLPQ/2F9oAky6c5+eiYTyy3/AgrMFRfQ9QonndyWkNbIzXqE20DTL1XyeBebyqvl6qUIc7LH3SnBAjIKjuUS8hQ4sgXnjv5CPFp4RLOAgSLN8AGv974DH/qAg+iaZ7Be8ExxYOVcePIotTm2JDuIfsaIshAfLVqIxamHCvf0WUWhAif4UImqEjxZOgp0hZk4ZL28LacFKvPbR2oEYTiOLpWnU2H+DtbPx2PsqONvAt1BjEx5wotLoyM55AazWywgTHm/R1s5GYy+C0yxBxSU84wii06hoIzB8G3Ad1CYcYOxru7ciOM1ygLoMYTNaCGo2EtfBZhFbOUJVtHtba+xFcJpnIFaOMbUnvikFsWEeRVjHAnVJZ0mSfE6S5JN6hNm4v6M/+N/U5yuSLq4ylrXG/oPdZ2mUK3LwuY0Pu0K0hfJ2XzJME4Ozccq92XW0CN+iJv7UxTk2R2Kjq66vqxQ/Fv6bl/8WqewOrGibz6rG/meVoykxC87IRpUqXmIPL5EVIBx4JDjqTJNxESGCivrUs6vzQsuYjgPRObe4gGnHYXA8ZIRYoe1q51GVsReXqibqJWJBtlECzuAoNrdKTTy1aGbjvjrSsQtX4Ao7ty1atgPxKGewsWiV0HyZjfuHNo4UYN6qjTK1VOemxv7dcgURnA3B7tEhik5IvYBqo8xtTPhDxCJsCc+RLWsRVts3C199od6/i+4CBeH5ZCHOc/De2IvgGICqaFawcitqctSYwULUwmODE7R/oIGdm90+Q4nu59m4P3DdoQ/hhjZOiDMpHXsRHEPgs8Z8FMMKBeFJiVZiEbZrNSAnDNRv7jTZAwjvoGth/q4dexEcDtI+c0Ng4rfhVjBpsQ4CI+DK7Kqoxaax1rRF4GJ9IX7l2rEXweEggmOIcivIkz7BmTVGbIzZ+kO5UV3fmpwjfsS0dE5XnTAXwSEgfXk5YNKzRcfIysGiYWalOr7OF1g6zJjOb0ItgiN4hQXR+Who5TCtm6++uFEl9IjZq9+sHBEcf/B9IjrDgnm/0Xk1snUzmY373jeXI2dek7djL4JDgBQnEMF5zYC8026SsWIuPK8bhxVBypzlWr363SI4HExraOYSB3qNhZ12k+9i/f2rAN8vq4tBq1iXI4LDwfTlbM19THWwudO+B6xWVt1NcA3uIZAst/Zl7EVwDMHRfNOJGXMvYlNYO+1RzUbgrErlEK0bDSvm9OIBiOAYgF3Q9KVMAshcNAYWK8vKqSMiLMEJdjPBvGRUgbd0Z0YRnA3BbjkitF2I7T4pG7AWbSURQYCZ0X5iHsFFfqyxF8HZFKj1lDApL2K4WdI2OG/EOF1etRUo6+R+DLE51m8QwakL7hxSiv+dYNnMxbqpBWXiVyxhYJ3cD34zgUvLcKuWYh9zxz8aSOt1iUVgXp6n8Zxr0vh3KtQ8sSycWKxXSjtVtWHHLDjtNMs3/f8e4tOx0JF/4dNJ4YBgjVeVTBXjWtv7iDYU2tjHLDg2OrOZokzTnohNfZRpn2b5nFCCUMV6YQSMYyrkZM3XjsRw3HEhlo0xjEVMuZS/AtG8Z+aclRiOfW7QhV+ExhzGvd2lFhLxJs/YjqosCImStggOnwUWxjXuFJIzUjxCionE9t4ZYr8rgsOnhRejTPcu7p2e4r4nyUqZQVnEKlsiG0EziODY4wCfl10hzfIbWD1ydmozWCJx6MACEUFbgQSN3XKir0VNs3wod4p7CSWGE6EFRYlBiuA0Qwu3ADyw708SBEtQwgEiOM2ihOffNMvfvSJVEGJABMcPTnARv6saEWE9FNchwndJcTVFcPxBVbdORXRKcWEFsjKJ8h5XEHOW6tMmrR/g2hTL3zuFfw/J172+RblYyr3qSAp9JZRDldISpDkkLf4GLPTihHw1OSFIHXy6FgToCE2PJJgcNrHF5CgWm7hUNcEF8KqWZjAb99VL+GThMvgTyV6thGHhvNfIi3UEhdXiwhcYG+utCI4hyjzHFamp6k9M/GrvL01rAMYuWyooRFc2mhgO6d61JSI4JFSh12zcV27WGekrD3AjhPALRtuIKoLC6HAXk4XDEs+pCA6Z2bg/JN6NLS1IAfEUdxWXiWHlMMTRF2gn6EVwLICzUheEbz5gmrOBwxqHKoJDyWIRRbJpWMc9xMKxyJB000Ask9YUVhC9iuCwAsfBB/6JV+Ys45siOJZA8JER+N36bBUmPaO3dNU73Fl1OjG8O6rQi+DYhXG1iVSs1rwXvIRKQoLNghE4jsElZgnOcuxFcCyCtqKmbpXNyuZQYN0vXsdyYVk5rGd3Do7ZnJD+rgiOI4zjAdt8vgoFkCzRrWNxsm6c7AbcCYBlWd7o+iYRnDDYZreKVQB5U6eoD+etGEH/VohWDkSS9dwv4i2CI3hLmuWDhqwbDasV7CBAK2dIuKVBI4Ij+A3cSFbh42LDPtIs66oV0lEVBLr/Jn3dVdGyFMGxDyOdu40Nua+JO+xGix0pdNb5uNOADuQym/y/GnsRHIuwUqLbdqVJmuUj4tGAhaF1wTxeMvI9AZBm+Tlx7CdvL4AUwbELI+jGCFwGA8TmlPi85yYnwBE8Zlk5usGal/EcHBZmuVLJKrEWwbEEdjKGCb01VwRbEBtT60bDtHKO0L/aK9GB2FwSv3KyqrOiCI49RqQYRPTtMNXiww2lTLFJcKe78clvLBxmkzWvRMeC2CTranhiFZybJi0D7NSMYHHi+HfcEwveKoFA6gNxvDRqh2VmhgZk91aLTqNHH9SFjBbE5mxd3DG2nsZqQvRUC9Am/jh2rBGxHHzh8LecoZePE9C6YWhBaBI9D5hfqCwlWAL/Er9Wi86QLI7vApefuTFq7svmUUwWjgrstRsUmx52apbYJI6sjTluuHAiNmqc4D59tyQ2ioGNzB7mFrt/tXK7v6kxcZXBglUztTD+i/filrFYOF9d7xDJr126i92UVTNSxPZvusLitHYlTeGWiy4+NsapyNWGRX5VGaAZGLujn1r8szTLrxB7ogom3kMXVqWtA8G99547dMG5x49cGeeAf8wKzB3io++tsrU7a36rYSBS6npip910t9X/3zb+ddlqc4KG9taAa9WBhWBj4Z6iSHACl+faZEPAGuhZ3BQ1X6p4FyELzsW6LATUfEiuKXCNrYWjJnJ33STG+SXmORpX3LtqeAXR6SKDaGucPuJzmWa5DuYrkXso2WD1Ztgu3J3m4j1WtipDFJwFFszKdDEUnVmp2gRro/wGLCDQK900TNZrB5abDdSCdHpbqVr0sHRsio7mqDif0yy3/OdqcVXHqgwtaKzS3YclYqN25h+Bi83EQgBXL8h1YmMrNe2CiWux0cDS6JC6A4bI17oubCgWzgLBzZVmm8UUn2tsuAVr092was4tFNy5otbuaoM3lk5MV8OUUboeywjBwtHp7nVi07OU4nMN2y0oTXcXAp8his0CQUovLgrE9c9t0tVAvjPHPN0oE+i74KjdubMqnoFy+GtUSYYW4HzLDVlsriDSZa7n90D7JZduQE2i7ptPkuRzxAduLzD2G2dPfXWp1A/6X0k0vkPul9IkzApfJcyfS9LdIQfUS4PevqDGvtA8LOQsaZE5yiiMz/Xt/PfnX7eG7sjZ3uOdqyrVGNLdmkmVQikWAae7k7ISCJ+xfHzDBQu096Ct72DS4pGkuxMIzZCxW1Qh4HT3Au/7PNQGZHjHnQCFR7f1MOoltIogBAcxh388eBQTVFxl5Epokl/pblabDFfc45lHoVk063gjPD2PA/VzCI21sfdacAJPdy9gWdyalqfXJcB0t66kvbZ4nKNxIDy32ED1GbymLXY9T51sht4KDtLd5wHtzhMEbdWCuW1q4WAXHXmegZroccJYRWHJVAUu4tJlKXSG7DrcWOd6M3TdXcFLwYEr0POovebDipsT9G4w9WXBIM41XPO8rlHvTo+Lfp6HbWsI/x5F8Ul+bRj6LBTj8OuiIO5TzNfG3kFQWSpB2EYKp/f14cwyfuqN2mW8sCqxdfwThOiARaKtkkYazLGQJuqCIDhDBEcQBGeI4AiC4AwRHEEQnCGCIwiCM0RwBEFwBkNwpJBLEIRKmNThLJuZ7z3eRX/3tSAIHDa1cJbNzEVsBEGoQ10LZ9k8ee/xzrv2joIg+E8dwVl2qNt7vFsZs3naPw7tdLcgCA7Ze7zbqSo4aw9oPu0f76IdAvMSf0EQIuQ9wbmHVbOyTcTT/nFMzcwFQbBMmeAsG1fvPd791usFVk1MXekFQXDAKsEpTXc/7R/H0sxcEATHvE2Ll6a7n/aPY7i7WxCEhtAWTmm6+2n/OJa7uwVBaJA/0Ou0XSI2sdzdLQhCw+w8Pz+vfAJJdwuCwGRtHY6kuwVBsMErwZF0tyAINnkRHEl3C4Jgm2VaXNLdgiC44MPT/vEAt/xNZMQFQbBGkiT/B29UxSXyCOhjAAAAAElFTkSuQmCC"><span class="enhet">Digital Revisjon</span></div>

  <div class="scene-side" id="side-forside">
    <div class="helside" id="forsideHelside">
      <div class="duk"></div>
      <div class="innhold">
        <p class="eyebrow">Digital Revisjon &middot; Bilbransjen</p>
        <h1>Hver bil,<span class="l2">hele historikken.</span></h1>
        <p class="ingress2">Bokf&oslash;rte bilsalg kontrollert mot Statens vegvesen, via OFV. Hele populasjonen, ikke et utvalg.</p>
        <button type="button" class="stor-knapp" data-gaa="1">Start</button>
      </div>
      <div class="meta">Digital Revisjon, Bilbransjen</div>
    </div>
    
  </div>

  <div class="scene-side" id="side-foretter-for" hidden>
    <div class="tekstblokk">
      <p class="eyebrow">Fra utvalg til hele populasjonen &middot; f&oslash;r</p>
      <h1 class="tittel">Et utvalg,<span class="l2">sett over for h&aring;nd.</span></h1>
    </div>
    <ol class="sideliste" style="top:120px;width:520px">
      <li><span class="nr">01</span><span><b>Et utvalg p&aring; 20 biler,</b> sl&aring;tt opp &eacute;n og &eacute;n p&aring; vegvesen.no.</span></li>
      <li><span class="nr">02</span><span><b>Kun siste eier.</b> Var bilen solgt videre, m&aring;tte revisor lete manuelt.</span></li>
      <li><span class="nr">03</span><span><b>Resten av bilene:</b> ikke sett p&aring; enkeltvis.</span></li>
    </ol>
    <div class="fe-rad2">
      <span class="fe-merke2">Illustrasjon: hver figur er en bil. Bokf&oslash;rte bilsalg i perioden (anonymisert eksempel).</span>
      <div class="fe-rutenett2" id="feRutenettFor"></div>
      <div class="fe-tallboks" id="feTallFor"></div>
    </div>
  </div>

  <div class="scene-side" id="side-foretter-na" hidden>
    <div class="tekstblokk">
      <p class="eyebrow">Fra utvalg til hele populasjonen &middot; n&aring;</p>
      <h1 class="tittel">Hele populasjonen,<span class="l2">hele historikken.</span></h1>
    </div>
    <ol class="sideliste" style="top:120px;width:520px">
      <li><span class="nr">01</span><span><b>Hele populasjonen,</b> alle bokf&oslash;rte salg kj&oslash;res mot OFV i &eacute;n operasjon.</span></li>
      <li><span class="nr">02</span><span><b>Eierskiftet som treffer,</b> med selger og kj&oslash;per.</span></li>
      <li><span class="nr">03</span><span><b>Avvik i dager,</b> regnet ut automatisk for hver bil.</span></li>
    </ol>
    <div class="fe-rad2">
      <span class="fe-merke2">Illustrasjon: hver figur er en bil. Bokf&oslash;rte bilsalg i perioden (anonymisert eksempel).</span>
      <div class="fe-rutenett2" id="feRutenettNa"></div>
      <div class="fe-tallboks" id="feTallNa"></div>
    </div>
  </div>

  <div class="scene-side" id="side-datakilder-0" hidden>
    <div class="tekstblokk">
      <p class="eyebrow">Slik henter vi dataene</p>
      <h1 class="tittel">To kilder,<span class="l2">&eacute;n kontroll.</span></h1>
    </div>
    <ol class="sideliste" style="top:120px;width:520px">
      <li><span class="nr">01</span><span><b>Fra forhandleren:</b> regnummer eller chassisnummer, og bokf&oslash;rt dato.</span></li>
      <li><span class="nr">02</span><span><b>Utenfra:</b> hele transaksjonshistorikken fra OFV.</span></li>
    </ol>
    <div class="kart-rad" id="kartRad0">
      <div class="kilde3">
        <div class="boks"><div class="sone">Hos forhandleren</div><h3>Hovedboken <img class="boks-logo" id="hovedbokLogo0" hidden></h3><div class="sub">Bokf&oslash;rte bilsalg: regnr og bokf&oslash;rt dato</div><span class="antall antall-vent">Trykk Neste for &aring; hente</span></div>
        <div class="boks midt"><div class="sone">Utenfra</div><h3>OFV <img class="boks-logo" src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHZpZXdCb3g9IjAgMCAxNjIuNCA2Ni42Ij48ZyBmaWxsPSIjMjAyMDIwIj4KICA8cG9seWdvbiBwb2ludHM9Ijk3LjEsMjAuMyA5Ny4xLDE3LjggNzIuNCwxNy44IDcyLjQsMjkuNSA4OS43LDI5LjUgODkuNywyNy4xIDc0LjksMjcuMSA3NC45LDIwLjMgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI3Mi40LDU5IDc0LjksNTkgNzQuOSwzOC41IDg5LjcsMzguNSA4OS43LDM2IDcyLjQsMzYgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI5Ny4xLDguOSA2Mi45LDguOSA2Mi45LDU4LjkgNjUuMyw1OC45IDY1LjMsMTEuNCA5Ny4xLDExLjQgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSIxNTAuOSw3IDE0OSw1LjQgMTE4LjYsNDIuNyAxMDQuMiwzMSAxMDIuNiwzMi44IDExOSw0Ni4yICIvPgogIDxwb2x5Z29uIHBvaW50cz0iMTU2LjIsMTEuMyAxMjAuMiw1NS41IDk4LjUsMzcuOSA5NywzOS44IDEyMC41LDU4LjkgMTU4LjEsMTIuOCAiLz4KICA8cGF0aCBkPSJNMzAuMSw4LjljLTctMC4yLTEzLjUsMi40LTE4LjQsNy4zYy00LjksNC45LTcuNSwxMS41LTcuMywxOC40YzAuNCwxMy4xLDExLDIzLjcsMjQuMSwyNC4xYzAuMywwLDAuNiwwLDAuOCwwIGM2LjYsMCwxMi45LTIuNiwxNy42LTcuM2M0LjktNC45LDcuNS0xMS41LDcuMy0xOC40QzUzLjgsMTkuOSw0My4yLDkuMywzMC4xLDguOXogTTQ1LjIsNDkuN2MtNC40LDQuNC0xMC4zLDYuOC0xNi42LDYuNiBDMTYuOCw1NS45LDcuMiw0Ni40LDYuOCwzNC42QzYuNiwyOC4zLDksMjIuNCwxMy40LDE3LjljNC4yLTQuMiw5LjktNi42LDE1LjktNi42YzAuMiwwLDAuNSwwLDAuNywwYzExLjgsMC40LDIxLjQsOS45LDIxLjcsMjEuNyBDNTIsMzkuNCw0OS42LDQ1LjMsNDUuMiw0OS43eiIvPgogIDxwYXRoIGQ9Ik0yOC41LDE4LjJMMjguNSwxOC4yYy03LjksMC40LTE0LjQsNi45LTE0LjgsMTQuOGMtMC4yLDQuNCwxLjQsOC44LDQuNiwxMS45YzIuOSwyLjksNyw0LjYsMTEuMSw0LjYgYzAuMywwLDAuNSwwLDAuOCwwQzM4LDQ5LjEsNDQuNiw0Mi42LDQ1LDM0LjZjMC4yLTQuNC0xLjQtOC44LTQuNi0xMS45QzM3LjMsMTkuNiwzMi45LDE3LjksMjguNSwxOC4yeiBNNDIuNSwzNC41IEM0Mi4yLDQxLjIsMzYuNyw0Ni43LDMwLDQ3Yy0zLjcsMC4yLTcuNC0xLjItMTAtMy45Yy0yLjYtMi42LTQtNi4zLTMuOS0xMGMwLjMtNi43LDUuOC0xMi4yLDEyLjUtMTIuNWMwLjIsMCwwLjUsMCwwLjcsMCBjMy41LDAsNi45LDEuNCw5LjQsMy45QzQxLjMsMjcuMSw0Mi43LDMwLjgsNDIuNSwzNC41eiIvPgo8L2c+PC9zdmc+Cg==" alt="OFV"></h3><div class="sub">Juridisk eierskifte, fra Statens vegvesen</div><span class="antall antall-vent">Venter</span></div>
        <div class="boks"><div class="sone">Kontrollen</div><h3>Avvik i dager</h3><div class="sub">N&aelig;rmeste eierskifte mot bokf&oslash;rt dato</div><span class="antall antall-vent">Venter</span></div>
      </div>
    </div>
  </div>

  <div class="scene-side" id="side-datakilder-1" hidden>
    <div class="tekstblokk" style="width:1488px">
      <p class="eyebrow">Slik henter vi dataene &middot; 1 av 3</p>
      <h1 class="tittel">Hovedboken<span class="l2">Regnummer, chassisnummer og bokf&oslash;rt dato.</span></h1>
    </div>
    <div class="kart-rad" id="kartRad1">
      <div class="kilde3">
        <div class="boks"><div class="sone">Hos forhandleren</div><h3>Hovedboken <img class="boks-logo" id="hovedbokLogo1" hidden></h3><div class="sub">Bokf&oslash;rte bilsalg: regnr og bokf&oslash;rt dato</div><span class="antall antall-ferdig" id="antallA1">Hentet</span></div>
        <div class="boks midt"><div class="sone">Utenfra</div><h3>OFV <img class="boks-logo" src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHZpZXdCb3g9IjAgMCAxNjIuNCA2Ni42Ij48ZyBmaWxsPSIjMjAyMDIwIj4KICA8cG9seWdvbiBwb2ludHM9Ijk3LjEsMjAuMyA5Ny4xLDE3LjggNzIuNCwxNy44IDcyLjQsMjkuNSA4OS43LDI5LjUgODkuNywyNy4xIDc0LjksMjcuMSA3NC45LDIwLjMgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI3Mi40LDU5IDc0LjksNTkgNzQuOSwzOC41IDg5LjcsMzguNSA4OS43LDM2IDcyLjQsMzYgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI5Ny4xLDguOSA2Mi45LDguOSA2Mi45LDU4LjkgNjUuMyw1OC45IDY1LjMsMTEuNCA5Ny4xLDExLjQgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSIxNTAuOSw3IDE0OSw1LjQgMTE4LjYsNDIuNyAxMDQuMiwzMSAxMDIuNiwzMi44IDExOSw0Ni4yICIvPgogIDxwb2x5Z29uIHBvaW50cz0iMTU2LjIsMTEuMyAxMjAuMiw1NS41IDk4LjUsMzcuOSA5NywzOS44IDEyMC41LDU4LjkgMTU4LjEsMTIuOCAiLz4KICA8cGF0aCBkPSJNMzAuMSw4LjljLTctMC4yLTEzLjUsMi40LTE4LjQsNy4zYy00LjksNC45LTcuNSwxMS41LTcuMywxOC40YzAuNCwxMy4xLDExLDIzLjcsMjQuMSwyNC4xYzAuMywwLDAuNiwwLDAuOCwwIGM2LjYsMCwxMi45LTIuNiwxNy42LTcuM2M0LjktNC45LDcuNS0xMS41LDcuMy0xOC40QzUzLjgsMTkuOSw0My4yLDkuMywzMC4xLDguOXogTTQ1LjIsNDkuN2MtNC40LDQuNC0xMC4zLDYuOC0xNi42LDYuNiBDMTYuOCw1NS45LDcuMiw0Ni40LDYuOCwzNC42QzYuNiwyOC4zLDksMjIuNCwxMy40LDE3LjljNC4yLTQuMiw5LjktNi42LDE1LjktNi42YzAuMiwwLDAuNSwwLDAuNywwYzExLjgsMC40LDIxLjQsOS45LDIxLjcsMjEuNyBDNTIsMzkuNCw0OS42LDQ1LjMsNDUuMiw0OS43eiIvPgogIDxwYXRoIGQ9Ik0yOC41LDE4LjJMMjguNSwxOC4yYy03LjksMC40LTE0LjQsNi45LTE0LjgsMTQuOGMtMC4yLDQuNCwxLjQsOC44LDQuNiwxMS45YzIuOSwyLjksNyw0LjYsMTEuMSw0LjYgYzAuMywwLDAuNSwwLDAuOCwwQzM4LDQ5LjEsNDQuNiw0Mi42LDQ1LDM0LjZjMC4yLTQuNC0xLjQtOC44LTQuNi0xMS45QzM3LjMsMTkuNiwzMi45LDE3LjksMjguNSwxOC4yeiBNNDIuNSwzNC41IEM0Mi4yLDQxLjIsMzYuNyw0Ni43LDMwLDQ3Yy0zLjcsMC4yLTcuNC0xLjItMTAtMy45Yy0yLjYtMi42LTQtNi4zLTMuOS0xMGMwLjMtNi43LDUuOC0xMi4yLDEyLjUtMTIuNWMwLjIsMCwwLjUsMCwwLjcsMCBjMy41LDAsNi45LDEuNCw5LjQsMy45QzQxLjMsMjcuMSw0Mi43LDMwLjgsNDIuNSwzNC41eiIvPgo8L2c+PC9zdmc+Cg==" alt="OFV"></h3><div class="sub">Juridisk eierskifte, fra Statens vegvesen</div><span class="antall antall-vent">Venter</span></div>
        <div class="boks"><div class="sone">Kontrollen</div><h3>Avvik i dager</h3><div class="sub">N&aelig;rmeste eierskifte mot bokf&oslash;rt dato</div><span class="antall antall-vent">Venter</span></div>
      </div>
      <div class="bilkortrad2" id="bilkortrad1"></div>
    </div>
  </div>

  <div class="scene-side" id="side-datakilder-2" hidden>
    <div class="tekstblokk" style="width:1488px">
      <p class="eyebrow">Slik henter vi dataene &middot; 2 av 3</p>
      <h1 class="tittel">OFV<span class="l2">Eierskiftet som treffer bokf&oslash;rt dato.</span></h1>
    </div>
    <div class="kart-rad" id="kartRad2">
      <div class="kilde3">
        <div class="boks"><div class="sone">Hos forhandleren</div><h3>Hovedboken <img class="boks-logo" id="hovedbokLogo2" hidden></h3><div class="sub">Bokf&oslash;rte bilsalg: regnr og bokf&oslash;rt dato</div><span class="antall antall-ferdig">Hentet</span></div>
        <div class="boks midt"><div class="sone">Utenfra</div><h3>OFV <img class="boks-logo" src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHZpZXdCb3g9IjAgMCAxNjIuNCA2Ni42Ij48ZyBmaWxsPSIjMjAyMDIwIj4KICA8cG9seWdvbiBwb2ludHM9Ijk3LjEsMjAuMyA5Ny4xLDE3LjggNzIuNCwxNy44IDcyLjQsMjkuNSA4OS43LDI5LjUgODkuNywyNy4xIDc0LjksMjcuMSA3NC45LDIwLjMgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI3Mi40LDU5IDc0LjksNTkgNzQuOSwzOC41IDg5LjcsMzguNSA4OS43LDM2IDcyLjQsMzYgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI5Ny4xLDguOSA2Mi45LDguOSA2Mi45LDU4LjkgNjUuMyw1OC45IDY1LjMsMTEuNCA5Ny4xLDExLjQgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSIxNTAuOSw3IDE0OSw1LjQgMTE4LjYsNDIuNyAxMDQuMiwzMSAxMDIuNiwzMi44IDExOSw0Ni4yICIvPgogIDxwb2x5Z29uIHBvaW50cz0iMTU2LjIsMTEuMyAxMjAuMiw1NS41IDk4LjUsMzcuOSA5NywzOS44IDEyMC41LDU4LjkgMTU4LjEsMTIuOCAiLz4KICA8cGF0aCBkPSJNMzAuMSw4LjljLTctMC4yLTEzLjUsMi40LTE4LjQsNy4zYy00LjksNC45LTcuNSwxMS41LTcuMywxOC40YzAuNCwxMy4xLDExLDIzLjcsMjQuMSwyNC4xYzAuMywwLDAuNiwwLDAuOCwwIGM2LjYsMCwxMi45LTIuNiwxNy42LTcuM2M0LjktNC45LDcuNS0xMS41LDcuMy0xOC40QzUzLjgsMTkuOSw0My4yLDkuMywzMC4xLDguOXogTTQ1LjIsNDkuN2MtNC40LDQuNC0xMC4zLDYuOC0xNi42LDYuNiBDMTYuOCw1NS45LDcuMiw0Ni40LDYuOCwzNC42QzYuNiwyOC4zLDksMjIuNCwxMy40LDE3LjljNC4yLTQuMiw5LjktNi42LDE1LjktNi42YzAuMiwwLDAuNSwwLDAuNywwYzExLjgsMC40LDIxLjQsOS45LDIxLjcsMjEuNyBDNTIsMzkuNCw0OS42LDQ1LjMsNDUuMiw0OS43eiIvPgogIDxwYXRoIGQ9Ik0yOC41LDE4LjJMMjguNSwxOC4yYy03LjksMC40LTE0LjQsNi45LTE0LjgsMTQuOGMtMC4yLDQuNCwxLjQsOC44LDQuNiwxMS45YzIuOSwyLjksNyw0LjYsMTEuMSw0LjYgYzAuMywwLDAuNSwwLDAuOCwwQzM4LDQ5LjEsNDQuNiw0Mi42LDQ1LDM0LjZjMC4yLTQuNC0xLjQtOC44LTQuNi0xMS45QzM3LjMsMTkuNiwzMi45LDE3LjksMjguNSwxOC4yeiBNNDIuNSwzNC41IEM0Mi4yLDQxLjIsMzYuNyw0Ni43LDMwLDQ3Yy0zLjcsMC4yLTcuNC0xLjItMTAtMy45Yy0yLjYtMi42LTQtNi4zLTMuOS0xMGMwLjMtNi43LDUuOC0xMi4yLDEyLjUtMTIuNWMwLjIsMCwwLjUsMCwwLjcsMCBjMy41LDAsNi45LDEuNCw5LjQsMy45QzQxLjMsMjcuMSw0Mi43LDMwLjgsNDIuNSwzNC41eiIvPgo8L2c+PC9zdmc+Cg==" alt="OFV"></h3><div class="sub">Juridisk eierskifte, fra Statens vegvesen</div><span class="antall antall-ferdig" id="antallB2">Hentet</span></div>
        <div class="boks"><div class="sone">Kontrollen</div><h3>Avvik i dager</h3><div class="sub">N&aelig;rmeste eierskifte mot bokf&oslash;rt dato</div><span class="antall antall-vent">Venter</span></div>
      </div>
      <div class="bilkortrad2" id="bilkortrad2"></div>
    </div>
  </div>

  <div class="scene-side" id="side-datakilder-3" hidden>
    <div class="tekstblokk" style="width:1488px">
      <p class="eyebrow">Slik henter vi dataene &middot; 3 av 3</p>
      <h1 class="tittel">Kontrollen<span class="l2">N&aelig;rmeste eierskifte, ikke siste.</span></h1>
      <p class="ingress">Stemmer bokf&oslash;rt salgsdato med det juridiske eierskiftet?</p>
    </div>
    <div class="kart-rad" id="kartRad3">
      <div class="kilde3">
        <div class="boks"><div class="sone">Hos forhandleren</div><h3>Hovedboken <img class="boks-logo" id="hovedbokLogo3" hidden></h3><div class="sub">Bokf&oslash;rte bilsalg: regnr og bokf&oslash;rt dato</div><span class="antall antall-ferdig">Hentet</span></div>
        <div class="boks midt"><div class="sone">Utenfra</div><h3>OFV <img class="boks-logo" src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHZpZXdCb3g9IjAgMCAxNjIuNCA2Ni42Ij48ZyBmaWxsPSIjMjAyMDIwIj4KICA8cG9seWdvbiBwb2ludHM9Ijk3LjEsMjAuMyA5Ny4xLDE3LjggNzIuNCwxNy44IDcyLjQsMjkuNSA4OS43LDI5LjUgODkuNywyNy4xIDc0LjksMjcuMSA3NC45LDIwLjMgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI3Mi40LDU5IDc0LjksNTkgNzQuOSwzOC41IDg5LjcsMzguNSA4OS43LDM2IDcyLjQsMzYgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI5Ny4xLDguOSA2Mi45LDguOSA2Mi45LDU4LjkgNjUuMyw1OC45IDY1LjMsMTEuNCA5Ny4xLDExLjQgIi8+CiAgPHBvbHlnb24gcG9pbnRzPSIxNTAuOSw3IDE0OSw1LjQgMTE4LjYsNDIuNyAxMDQuMiwzMSAxMDIuNiwzMi44IDExOSw0Ni4yICIvPgogIDxwb2x5Z29uIHBvaW50cz0iMTU2LjIsMTEuMyAxMjAuMiw1NS41IDk4LjUsMzcuOSA5NywzOS44IDEyMC41LDU4LjkgMTU4LjEsMTIuOCAiLz4KICA8cGF0aCBkPSJNMzAuMSw4LjljLTctMC4yLTEzLjUsMi40LTE4LjQsNy4zYy00LjksNC45LTcuNSwxMS41LTcuMywxOC40YzAuNCwxMy4xLDExLDIzLjcsMjQuMSwyNC4xYzAuMywwLDAuNiwwLDAuOCwwIGM2LjYsMCwxMi45LTIuNiwxNy42LTcuM2M0LjktNC45LDcuNS0xMS41LDcuMy0xOC40QzUzLjgsMTkuOSw0My4yLDkuMywzMC4xLDguOXogTTQ1LjIsNDkuN2MtNC40LDQuNC0xMC4zLDYuOC0xNi42LDYuNiBDMTYuOCw1NS45LDcuMiw0Ni40LDYuOCwzNC42QzYuNiwyOC4zLDksMjIuNCwxMy40LDE3LjljNC4yLTQuMiw5LjktNi42LDE1LjktNi42YzAuMiwwLDAuNSwwLDAuNywwYzExLjgsMC40LDIxLjQsOS45LDIxLjcsMjEuNyBDNTIsMzkuNCw0OS42LDQ1LjMsNDUuMiw0OS43eiIvPgogIDxwYXRoIGQ9Ik0yOC41LDE4LjJMMjguNSwxOC4yYy03LjksMC40LTE0LjQsNi45LTE0LjgsMTQuOGMtMC4yLDQuNCwxLjQsOC44LDQuNiwxMS45YzIuOSwyLjksNyw0LjYsMTEuMSw0LjYgYzAuMywwLDAuNSwwLDAuOCwwQzM4LDQ5LjEsNDQuNiw0Mi42LDQ1LDM0LjZjMC4yLTQuNC0xLjQtOC44LTQuNi0xMS45QzM3LjMsMTkuNiwzMi45LDE3LjksMjguNSwxOC4yeiBNNDIuNSwzNC41IEM0Mi4yLDQxLjIsMzYuNyw0Ni43LDMwLDQ3Yy0zLjcsMC4yLTcuNC0xLjItMTAtMy45Yy0yLjYtMi42LTQtNi4zLTMuOS0xMGMwLjMtNi43LDUuOC0xMi4yLDEyLjUtMTIuNWMwLjIsMCwwLjUsMCwwLjcsMCBjMy41LDAsNi45LDEuNCw5LjQsMy45QzQxLjMsMjcuMSw0Mi43LDMwLjgsNDIuNSwzNC41eiIvPgo8L2c+PC9zdmc+Cg==" alt="OFV"></h3><div class="sub">Juridisk eierskifte, fra Statens vegvesen</div><span class="antall antall-ferdig">Hentet</span></div>
        <div class="boks"><div class="sone">Kontrollen</div><h3>Avvik i dager</h3><div class="sub">N&aelig;rmeste eierskifte mot bokf&oslash;rt dato</div><span class="antall antall-vent" id="antallC3">Venter</span>
          <div class="tellere2" id="tellereEksempel"></div>
        </div>
      </div>
      <div class="bilkortrad2" id="bilkortrad3"></div>
    </div>
  </div>

  <div class="scene-side" id="side-sporsmal" hidden>
    <div class="helside" id="sporsmalHelside">
      <div class="duk"></div>
      <div class="innhold" style="top:320px">
        <h1>Sp&oslash;rsm&aring;l?</h1>
        <p class="ingress2">Ta kontakt om noe i tabellen ser feil ut, eller om du vil ha hjelp til &aring; tolke et avvik.</p>
      </div>
    </div>
    
  </div>

</div>

<div id="frihost" hidden>
  <div class="fri-topp">
    <img class="bdo-logo" src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAARwAAABuCAYAAAD1aayeAAAACXBIWXMAAAsSAAALEgHS3X78AAAOiElEQVR4nO2dT04jORvGi1Hvg8RykKBOQL79oE5LtSd9AtKbbCd9ggknaNjWhnCCgX2kJuIAHU4QIrEcpM4J+OToMV3QSVEVP3bZzvuTopY+zRcqLvvx+8+vd56fnxNBEPwnzfJOxYeczsb9nz7+oJ3//vyrlyTJocF33O493t2aPkia5d0kSdqm3+MQ9UKn6t/ZuD9t6iHSLFdj1m3q77/DFOOUzMZ94zmyDaRZvot10MG6VJ+PG/70eZIkD2qN6n9n4/5Dk8P4IUmSnsEP0rAm0z+k73FKmuXqz00wDteOBehnCOOGMVpAhPSn8QXgA7BcuhCZI+IjHeDzsr7TLF/oedrE+CsL59ZQcM72Hu+GjIdJs/wBAxQ690mSnM/G/ZGL35Fmufo7p4GO2bww+a89eB4nwKLXn1aDj6Lmqpo/IxdumG+Co6ytS8Z3eYJaTEPbwgO36kdIA7OGBSb/eYyWD9ylAbwKHzfWG4y9NffXN8HZha/ZpOLbQO0iPZuuVprlpu/RNyYQ6+BjP2mWqziMWiOhWKHWLPQ/2F9oAky6c5+eiYTyy3/AgrMFRfQ9QonndyWkNbIzXqE20DTL1XyeBebyqvl6qUIc7LH3SnBAjIKjuUS8hQ4sgXnjv5CPFp4RLOAgSLN8AGv974DH/qAg+iaZ7Be8ExxYOVcePIotTm2JDuIfsaIshAfLVqIxamHCvf0WUWhAif4UImqEjxZOgp0hZk4ZL28LacFKvPbR2oEYTiOLpWnU2H+DtbPx2PsqONvAt1BjEx5wotLoyM55AazWywgTHm/R1s5GYy+C0yxBxSU84wii06hoIzB8G3Ad1CYcYOxru7ciOM1ygLoMYTNaCGo2EtfBZhFbOUJVtHtba+xFcJpnIFaOMbUnvikFsWEeRVjHAnVJZ0mSfE6S5JN6hNm4v6M/+N/U5yuSLq4ylrXG/oPdZ2mUK3LwuY0Pu0K0hfJ2XzJME4Ozccq92XW0CN+iJv7UxTk2R2Kjq66vqxQ/Fv6bl/8WqewOrGibz6rG/meVoykxC87IRpUqXmIPL5EVIBx4JDjqTJNxESGCivrUs6vzQsuYjgPRObe4gGnHYXA8ZIRYoe1q51GVsReXqibqJWJBtlECzuAoNrdKTTy1aGbjvjrSsQtX4Ao7ty1atgPxKGewsWiV0HyZjfuHNo4UYN6qjTK1VOemxv7dcgURnA3B7tEhik5IvYBqo8xtTPhDxCJsCc+RLWsRVts3C199od6/i+4CBeH5ZCHOc/De2IvgGICqaFawcitqctSYwULUwmODE7R/oIGdm90+Q4nu59m4P3DdoQ/hhjZOiDMpHXsRHEPgs8Z8FMMKBeFJiVZiEbZrNSAnDNRv7jTZAwjvoGth/q4dexEcDtI+c0Ng4rfhVjBpsQ4CI+DK7Kqoxaax1rRF4GJ9IX7l2rEXweEggmOIcivIkz7BmTVGbIzZ+kO5UV3fmpwjfsS0dE5XnTAXwSEgfXk5YNKzRcfIysGiYWalOr7OF1g6zJjOb0ItgiN4hQXR+Who5TCtm6++uFEl9IjZq9+sHBEcf/B9IjrDgnm/0Xk1snUzmY373jeXI2dek7djL4JDgBQnEMF5zYC8026SsWIuPK8bhxVBypzlWr363SI4HExraOYSB3qNhZ12k+9i/f2rAN8vq4tBq1iXI4LDwfTlbM19THWwudO+B6xWVt1NcA3uIZAst/Zl7EVwDMHRfNOJGXMvYlNYO+1RzUbgrErlEK0bDSvm9OIBiOAYgF3Q9KVMAshcNAYWK8vKqSMiLMEJdjPBvGRUgbd0Z0YRnA3BbjkitF2I7T4pG7AWbSURQYCZ0X5iHsFFfqyxF8HZFKj1lDApL2K4WdI2OG/EOF1etRUo6+R+DLE51m8QwakL7hxSiv+dYNnMxbqpBWXiVyxhYJ3cD34zgUvLcKuWYh9zxz8aSOt1iUVgXp6n8Zxr0vh3KtQ8sSycWKxXSjtVtWHHLDjtNMs3/f8e4tOx0JF/4dNJ4YBgjVeVTBXjWtv7iDYU2tjHLDg2OrOZokzTnohNfZRpn2b5nFCCUMV6YQSMYyrkZM3XjsRw3HEhlo0xjEVMuZS/AtG8Z+aclRiOfW7QhV+ExhzGvd2lFhLxJs/YjqosCImStggOnwUWxjXuFJIzUjxCionE9t4ZYr8rgsOnhRejTPcu7p2e4r4nyUqZQVnEKlsiG0EziODY4wCfl10hzfIbWD1ydmozWCJx6MACEUFbgQSN3XKir0VNs3wod4p7CSWGE6EFRYlBiuA0Qwu3ADyw708SBEtQwgEiOM2ihOffNMvfvSJVEGJABMcPTnARv6saEWE9FNchwndJcTVFcPxBVbdORXRKcWEFsjKJ8h5XEHOW6tMmrR/g2hTL3zuFfw/J172+RblYyr3qSAp9JZRDldISpDkkLf4GLPTihHw1OSFIHXy6FgToCE2PJJgcNrHF5CgWm7hUNcEF8KqWZjAb99VL+GThMvgTyV6thGHhvNfIi3UEhdXiwhcYG+utCI4hyjzHFamp6k9M/GrvL01rAMYuWyooRFc2mhgO6d61JSI4JFSh12zcV27WGekrD3AjhPALRtuIKoLC6HAXk4XDEs+pCA6Z2bg/JN6NLS1IAfEUdxWXiWHlMMTRF2gn6EVwLICzUheEbz5gmrOBwxqHKoJDyWIRRbJpWMc9xMKxyJB000Ask9YUVhC9iuCwAsfBB/6JV+Ys45siOJZA8JER+N36bBUmPaO3dNU73Fl1OjG8O6rQi+DYhXG1iVSs1rwXvIRKQoLNghE4jsElZgnOcuxFcCyCtqKmbpXNyuZQYN0vXsdyYVk5rGd3Do7ZnJD+rgiOI4zjAdt8vgoFkCzRrWNxsm6c7AbcCYBlWd7o+iYRnDDYZreKVQB5U6eoD+etGEH/VohWDkSS9dwv4i2CI3hLmuWDhqwbDasV7CBAK2dIuKVBI4Ij+A3cSFbh42LDPtIs66oV0lEVBLr/Jn3dVdGyFMGxDyOdu40Nua+JO+xGix0pdNb5uNOADuQym/y/GnsRHIuwUqLbdqVJmuUj4tGAhaF1wTxeMvI9AZBm+Tlx7CdvL4AUwbELI+jGCFwGA8TmlPi85yYnwBE8Zlk5usGal/EcHBZmuVLJKrEWwbEEdjKGCb01VwRbEBtT60bDtHKO0L/aK9GB2FwSv3KyqrOiCI49RqQYRPTtMNXiww2lTLFJcKe78clvLBxmkzWvRMeC2CTranhiFZybJi0D7NSMYHHi+HfcEwveKoFA6gNxvDRqh2VmhgZk91aLTqNHH9SFjBbE5mxd3DG2nsZqQvRUC9Am/jh2rBGxHHzh8LecoZePE9C6YWhBaBI9D5hfqCwlWAL/Er9Wi86QLI7vApefuTFq7svmUUwWjgrstRsUmx52apbYJI6sjTluuHAiNmqc4D59tyQ2ioGNzB7mFrt/tXK7v6kxcZXBglUztTD+i/filrFYOF9d7xDJr126i92UVTNSxPZvusLitHYlTeGWiy4+NsapyNWGRX5VGaAZGLujn1r8szTLrxB7ogom3kMXVqWtA8G99547dMG5x49cGeeAf8wKzB3io++tsrU7a36rYSBS6npip910t9X/3zb+ddlqc4KG9taAa9WBhWBj4Z6iSHACl+faZEPAGuhZ3BQ1X6p4FyELzsW6LATUfEiuKXCNrYWjJnJ33STG+SXmORpX3LtqeAXR6SKDaGucPuJzmWa5DuYrkXso2WD1Ztgu3J3m4j1WtipDFJwFFszKdDEUnVmp2gRro/wGLCDQK900TNZrB5abDdSCdHpbqVr0sHRsio7mqDif0yy3/OdqcVXHqgwtaKzS3YclYqN25h+Bi83EQgBXL8h1YmMrNe2CiWux0cDS6JC6A4bI17oubCgWzgLBzZVmm8UUn2tsuAVr092was4tFNy5otbuaoM3lk5MV8OUUboeywjBwtHp7nVi07OU4nMN2y0oTXcXAp8his0CQUovLgrE9c9t0tVAvjPHPN0oE+i74KjdubMqnoFy+GtUSYYW4HzLDVlsriDSZa7n90D7JZduQE2i7ptPkuRzxAduLzD2G2dPfXWp1A/6X0k0vkPul9IkzApfJcyfS9LdIQfUS4PevqDGvtA8LOQsaZE5yiiMz/Xt/PfnX7eG7sjZ3uOdqyrVGNLdmkmVQikWAae7k7ISCJ+xfHzDBQu096Ct72DS4pGkuxMIzZCxW1Qh4HT3Au/7PNQGZHjHnQCFR7f1MOoltIogBAcxh388eBQTVFxl5Epokl/pblabDFfc45lHoVk063gjPD2PA/VzCI21sfdacAJPdy9gWdyalqfXJcB0t66kvbZ4nKNxIDy32ED1GbymLXY9T51sht4KDtLd5wHtzhMEbdWCuW1q4WAXHXmegZroccJYRWHJVAUu4tJlKXSG7DrcWOd6M3TdXcFLwYEr0POovebDipsT9G4w9WXBIM41XPO8rlHvTo+Lfp6HbWsI/x5F8Ul+bRj6LBTj8OuiIO5TzNfG3kFQWSpB2EYKp/f14cwyfuqN2mW8sCqxdfwThOiARaKtkkYazLGQJuqCIDhDBEcQBGeI4AiC4AwRHEEQnCGCIwiCM0RwBEFwBkNwpJBLEIRKmNThLJuZ7z3eRX/3tSAIHDa1cJbNzEVsBEGoQ10LZ9k8ee/xzrv2joIg+E8dwVl2qNt7vFsZs3naPw7tdLcgCA7Ze7zbqSo4aw9oPu0f76IdAvMSf0EQIuQ9wbmHVbOyTcTT/nFMzcwFQbBMmeAsG1fvPd791usFVk1MXekFQXDAKsEpTXc/7R/H0sxcEATHvE2Ll6a7n/aPY7i7WxCEhtAWTmm6+2n/OJa7uwVBaJA/0Ou0XSI2sdzdLQhCw+w8Pz+vfAJJdwuCwGRtHY6kuwVBsMErwZF0tyAINnkRHEl3C4Jgm2VaXNLdgiC44MPT/vEAt/xNZMQFQbBGkiT/B29UxSXyCOhjAAAAAElFTkSuQmCC" alt="BDO">
    <span class="tittel2">OFV-kontroller</span>
    <span class="selskap" id="toppSelskap"></span>
    <nav class="rapportnav" id="rapportnav" aria-label="Rapportsider"></nav>
    <button type="button" class="knapp liten rolig" id="skrivUt" title="Skriv ut / lagre som PDF">Skriv ut</button>
  </div>
  <div id="frisider"></div>
  <p class="fotnote" id="fotnote"></p>
</div>

<div id="bunn">
  <div class="merknad" id="merknad">OFV-kontroller &middot; interne data behandles kun lokalt i nettleseren.</div>
  <nav class="prikknav" id="prikknav" aria-label="Sider"></nav>
  <div class="styr">
    <button type="button" id="tilRapport">Til rapporten</button>
    <button type="button" id="tilbake">Tilbake</button>
    <button type="button" id="frem" class="frem">Neste</button>
  </div>
</div>

<button type="button" id="tannhjul" title="Rediger innhold" aria-haspopup="dialog">&#9881;</button>

<div id="adminBakteppe" hidden>
  <div id="adminPanel" role="dialog" aria-modal="true" aria-labelledby="adminTittel">
    <button type="button" class="lukk" id="adminLukk" aria-label="Lukk">&times;</button>
    <h3 id="adminTittel">Innhold og data</h3>
    <p style="font-size:13px;color:var(--dempet);margin-top:4px">Dette panelet er ikke synlig for andre som &aring;pner lenken &mdash; du finner det igjen via tannhjulet.</p>

    <div class="admin-seksjon">
      <h4>Bilde p&aring; forsiden</h4>
      <div class="bilde-rad">
        <img class="bilde-forhandsvisning" id="forhandsvisningForside" hidden>
        <input type="file" id="bildeForside" accept="image/*">
        <button type="button" class="btn small ghost" id="fjernBildeForside">Fjern</button>
      </div>
    </div>

    <div class="admin-seksjon">
      <h4>Bilde p&aring; sp&oslash;rsm&aring;l-siden</h4>
      <div class="bilde-rad">
        <img class="bilde-forhandsvisning" id="forhandsvisningSporsmal" hidden>
        <input type="file" id="bildeSporsmal" accept="image/*">
        <button type="button" class="btn small ghost" id="fjernBildeSporsmal">Fjern</button>
      </div>
    </div>

    <div class="admin-seksjon">
      <h4>Logo i Hovedboken-boksen (Slik henter vi dataene)</h4>
      <p style="font-size:13px;color:var(--dempet);margin-top:4px">Valgfritt &mdash; f.eks. forhandlerens egen logo. Vises ved siden av "Hovedboken" p&aring; alle fire "Slik henter vi dataene"-sidene.</p>
      <div class="bilde-rad">
        <img class="bilde-forhandsvisning" id="forhandsvisningHovedbokLogo" hidden>
        <input type="file" id="bildeHovedbokLogo" accept="image/*">
        <button type="button" class="btn small ghost" id="fjernBildeHovedbokLogo">Fjern</button>
      </div>
    </div>

    <div class="admin-seksjon">
      <h4>Datasett fra Kontroll solgte biler (opptil 20 forhandlere)</h4>
      <p style="font-size:13px;color:var(--dempet);margin-top:4px">Ett datasett per forhandler/juridisk enhet. Kopier tabellen (inkl. overskriftsrad) i Excel og lim inn, eller last opp arbeidsboken. Innlimte datasett vises sammen med Databricks-kj&oslash;ringen p&aring; sidene Per forhandler, Solgte biler og Analyse.</p>
      <div id="databricksDatasett"></div>
      <div id="forhandlersloter"></div>
      <button type="button" class="btn small ghost" id="leggTilForhandler" style="margin-top:12px">+ Legg til forhandler</button>
      <p class="personvern">&#128274; <span>Alt skjer i din egen nettleser &mdash; ingenting lastes opp til noen server.</span></p>
    </div>
  </div>
</div>

<div class="tips" id="tips" role="tooltip"></div>

__XLSX__
<script id="rapport-data" type="application/json">__DATA__</script>
<script>
(function(){
  "use strict";

  var $ = function(id){ return document.getElementById(id); };
  var REDUSERT = window.matchMedia && window.matchMedia("(prefers-reduced-motion: reduce)").matches;
  var D = JSON.parse($("rapport-data").textContent);

  var nf = new Intl.NumberFormat("nb-NO");
  var pf = new Intl.NumberFormat("nb-NO", { style: "percent", maximumFractionDigits: 0 });
  var coll = new Intl.Collator("nb", { numeric: true, sensitivity: "base" });
  var SIDE = 200;   // rader per "side" i tabellene

  /* ==========================================================
     Felles hjelpere
  ========================================================== */
  function el(tag, attrs){
    var e = document.createElement(tag);
    if (attrs) for (var k in attrs){
      var v = attrs[k];
      if (v == null || v === false) continue;
      if (k === "text") e.textContent = v;
      else if (k === "cls") e.className = v;
      else if (k.slice(0, 2) === "on") e.addEventListener(k.slice(2), v);
      else e.setAttribute(k, v === true ? "" : v);
    }
    for (var i = 2; i < arguments.length; i++) leggTil(e, arguments[i]);
    return e;
  }
  function leggTil(e, c){
    if (c == null) return;
    if (Array.isArray(c)) c.forEach(function(x){ leggTil(e, x); });
    else e.appendChild(typeof c === "string" ? document.createTextNode(c) : c);
  }
  var SVGNS = "http://www.w3.org/2000/svg";
  function s(tag, attrs, text){
    var e = document.createElementNS(SVGNS, tag);
    for (var k in attrs || {}){
      var v = attrs[k];
      if ((k === "fill" || k === "stroke") && String(v).indexOf("var(") === 0) e.style.setProperty(k, v);
      else e.setAttribute(k, v);
    }
    if (text != null) e.textContent = text;
    return e;
  }
  function esc(s){
    return String(s == null ? "" : s).replace(/[&<>"]/g, function(c){ return {"&":"&amp;","<":"&lt;",">":"&gt;","\"":"&quot;"}[c]; });
  }
  function fmtDato(iso){ return iso ? iso.slice(8, 10) + "." + iso.slice(5, 7) + "." + iso.slice(0, 4) : ""; }
  function vis(col, v){
    if (v == null || v === "") return "";
    if (col.type === "date") return fmtDato(v);
    if (col.type === "num") return typeof v === "number" ? nf.format(v) : String(v);
    return String(v);
  }
  /* Tone-koder som i presentasjonen: g = OK, y = varsel, r = avvik, n = annet */
  var TONE_NAVN = {g:"OK", y:"Varsel", r:"Avvik", n:"Annet"};
  var TONE_FARGE = {g:"var(--gronn)", y:"var(--gul)", r:"var(--rod)", n:"var(--skifer)"};
  function toneAv(tekst){
    tekst = String(tekst || "");
    if (/^OK/i.test(tekst)) return "g";
    if (/^Avvik/i.test(tekst)) return "r";
    if (/^Varsel/i.test(tekst)) return "y";
    return "n";
  }
  function radTone(col, rad){ return col.toneKey ? (rad[col.toneKey] || "n") : toneAv(rad[col.key]); }

  /* --------------------------- lagring lokalt --------------------------- */

  function lagre(nokkel, verdi){ try { localStorage.setItem(nokkel, verdi); } catch (e) {} }
  function last(nokkel){ try { return localStorage.getItem(nokkel); } catch (e) { return null; } }
  function slett(nokkel){ try { localStorage.removeItem(nokkel); } catch (e) {} }

  /* Innlimte datasett lagres per selskap, sa en rapport for et annet
     selskap pa samme maskin aldri viser disse dataene. */
  var NOKKEL_SLOTER = "ofv-rapport-sloter-v1:" + D.selskap;
  var NOKKEL_BILDE_FORSIDE = "ksb-bilde-forside-v1";
  var NOKKEL_BILDE_SPORSMAL = "ksb-bilde-sporsmal-v1";
  var NOKKEL_LOGO_HOVEDBOK = "ksb-logo-hovedbok-v1";
  var MAKS_FORHANDLERE = 20;

  /* Avvik i dager -> tone (samme grenser som i Databricks og presentasjonen) */
  function avvikTall(v){
    if (v == null) return null;
    if (typeof v === "number") return Math.abs(v);
    var t = String(v).replace(",", ".").trim();
    if (t === "" || !/^-?\d+(\.\d+)?$/.test(t)) return null;
    return Math.abs(parseFloat(t));
  }
  function fargeklasse(v){
    if (v == null) return "n";
    if (v <= D.grenser.gronnMaks) return "g";
    if (v <= D.grenser.gulMaks) return "y";
    return "r";
  }

  var tips = $("tips");
  function visTips(ev, linjer){
    tips.textContent = "";
    linjer.forEach(function(l, i){
      if (i) tips.appendChild(el("br"));
      if (i === 0) tips.appendChild(el("b", {text: l})); else tips.appendChild(document.createTextNode(l));
    });
    var x = ev.clientX + 14, y = ev.clientY + 14;
    tips.classList.add("vis");
    var r = tips.getBoundingClientRect();
    if (x + r.width > window.innerWidth - 8) x = ev.clientX - r.width - 14;
    if (y + r.height > window.innerHeight - 8) y = ev.clientY - r.height - 14;
    tips.style.left = x + "px"; tips.style.top = y + "px";
  }
  function skjulTips(){ tips.classList.remove("vis"); }

  /* ==========================================================
     FASTE EKSEMPELDATA - 20 anonymiserte biler, alltid de samme,
     uavhengig av hva som limes inn i input-siden. Brukes kun i
     illustrasjonene "Før og etter" og "Datakilder".
  ========================================================== */
  /* Samme 14 biler (og samme transaksjonshistorikk) som i illustrasjonen
     du delte - regnr og selskapsnavn er anonymisert (AB12345 osv.,
     "Forhandler" i stedet for firmanavn), datoer og eierskifter er
     uendret. "ny" = kun forstegangsregistrering (ingen kjoper/selger). */
  var EKSEMPEL_BILER = [
    {regnr:"AB12345", modell:"bZ4X", bokf:"2025-04-07", hist:[
      {dato:"2025-04-07", fra:"Forhandler", til:"Privat"},
      {dato:"2025-02-19", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12346", modell:"Corolla Cross", bokf:"2025-05-08", ny:true, hist:[
      {dato:"2025-03-31", forstegang:true}]},
    {regnr:"AB12347", modell:"RAV4", bokf:"2025-12-19", hist:[
      {dato:"2025-12-06", fra:"Forhandler", til:"Privat"},
      {dato:"2025-05-22", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12348", modell:"UX", bokf:"2025-02-18", hist:[
      {dato:"2025-10-30", fra:"Privat", til:"Privat"},
      {dato:"2025-09-15", fra:"Privat", til:"Forhandler"},
      {dato:"2025-02-18", fra:"Forhandler", til:"Privat"},
      {dato:"2025-01-09", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12349", modell:"Yaris Cross", bokf:"2025-11-19", ny:true, hist:[
      {dato:"2025-11-19", forstegang:true}]},
    {regnr:"AB12350", modell:"Yaris", bokf:"2025-09-01", hist:[
      {dato:"2025-08-30", fra:"Forhandler", til:"Privat"},
      {dato:"2025-05-07", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12351", modell:"RAV4", bokf:"2025-06-05", hist:[
      {dato:"2026-01-14", fra:"Privat", til:"Privat"},
      {dato:"2025-11-03", fra:"Privat", til:"Forhandler"},
      {dato:"2025-08-22", fra:"Privat", til:"Privat"},
      {dato:"2025-06-04", fra:"Forhandler", til:"Privat"},
      {dato:"2025-05-12", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12352", modell:"RAV4", bokf:"2025-01-24", hist:[
      {dato:"2025-01-24", fra:"Forhandler", til:"Privat"},
      {dato:"2025-01-24", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12353", modell:"C-HR", bokf:"2025-03-20", ny:true, hist:[
      {dato:"2025-03-05", forstegang:true}]},
    {regnr:"AB12354", modell:"RAV4", bokf:"2025-10-30", hist:[
      {dato:"2026-08-12", fra:"Forhandler", til:"Privat"},
      {dato:"2026-06-18", fra:"Privat", til:"Forhandler"},
      {dato:"2025-10-28", fra:"Forhandler", til:"Privat"},
      {dato:"2025-09-26", fra:"Privat", til:"Forhandler"},
      {dato:"2024-10-22", fra:"Forhandler", til:"Privat"},
      {dato:"2024-08-21", fra:"Privat", til:"Forhandler"},
      {dato:"2022-12-08", fra:"Forhandler", til:"Privat"},
      {dato:"2022-10-07", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12355", modell:"RAV4", bokf:"2025-12-11", hist:[
      {dato:"2025-12-11", fra:"Forhandler", til:"Privat"},
      {dato:"2025-10-22", fra:"Privat", til:"Forhandler"}]},
    {regnr:"AB12356", modell:"UX", bokf:"2025-09-23", hist:[
      {dato:"2025-09-22", fra:"Forhandler", til:"Privat"},
      {dato:"2025-09-22", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12357", modell:"Yaris", bokf:"2025-05-27", hist:[
      {dato:"2025-12-08", fra:"Forhandler", til:"Privat"},
      {dato:"2025-11-19", fra:"Privat", til:"Forhandler"},
      {dato:"2025-05-26", fra:"Forhandler", til:"Privat"},
      {dato:"2025-04-11", fra:"Privat", til:"Forhandler"},
      {dato:"2022-03-03", fra:"Forhandler", til:"Privat"},
      {dato:"2022-02-14", fra:"Forhandler", til:"Forhandler"}]},
    {regnr:"AB12358", modell:"bZ4X", bokf:"2025-09-09", hist:[
      {dato:"2026-02-02", fra:"Privat", til:"Privat"},
      {dato:"2025-12-20", fra:"Privat", til:"Privat"},
      {dato:"2025-09-05", fra:"Forhandler", til:"Privat"},
      {dato:"2025-07-30", fra:"Forhandler", til:"Forhandler"}]}
  ];

  function dagerMellom(iso1, iso2){
    function tallDato(iso){ var p = iso.split("-").map(Number); return Date.UTC(p[0], p[1] - 1, p[2]); }
    return Math.round(Math.abs(tallDato(iso1) - tallDato(iso2)) / 86400000);
  }
  function nbDato(iso){ var p = iso.split("-"); return p[2] + "." + p[1] + "." + p[0]; }

  /* Samme kontrollregel som ellers i verktoyet: naermeste eierskifte
     (eller forstegangsregistrering) der forhandleren er selger. */
  EKSEMPEL_BILER.forEach(function(b){
    var best = -1, bestD = Infinity;
    b.hist.forEach(function(h, i){
      if (h.forstegang || h.fra === "Forhandler"){
        var d = dagerMellom(b.bokf, h.dato);
        if (d < bestD){ bestD = d; best = i; }
      }
    });
    b.treffIndex = best;
    b.avvik = bestD;
    b.farge = fargeklasse(bestD);
  });

  /* --------------------------- før/etter: renders per side (før eller nå) --------------------------- */

  var ANTALL_RUTENETT = 160;
  var ANTALL_UTVALG = 20;

  function lagUtvalgIndekser(){
    var valgt = {};
    var seed = 7;
    function rnd(){ seed = (seed * 16807) % 2147483647; return (seed - 1) / 2147483646; }
    while (Object.keys(valgt).length < ANTALL_UTVALG){
      valgt[Math.floor(rnd() * ANTALL_RUTENETT)] = true;
    }
    return valgt;
  }
  var UTVALG_INDEKSER = lagUtvalgIndekser();

  function spillAvForEtter(erEtter){

    var rutenettId = erEtter ? "feRutenettNa" : "feRutenettFor";
    var tallboksId = erEtter ? "feTallNa" : "feTallFor";

    var el = $(rutenettId);
    el.classList.remove("spill");
    el.innerHTML = "";

    for (var i = 0; i < ANTALL_RUTENETT; i++){
      var span = document.createElement("span");
      var utvalgt = erEtter ? true : !!UTVALG_INDEKSER[i];
      span.className = "bilag2" + (utvalgt ? " utvalg" : "");
      span.style.setProperty("--d", (REDUSERT ? 0 : (i % 40) * 12) + "ms");
      el.appendChild(span);
    }

    $(tallboksId).innerHTML = erEtter
      ? '<div class="lab">Slik gjør vi det nå</div>' +
        '<div class="stor">Alle</div>' +
        '<div class="under">bokførte salg, hele populasjonen, kjørt mot OFV i én operasjon</div>' +
        '<div class="skille"></div>' +
        '<div class="stor2">Hele historikken</div>' +
        '<div class="under2">Hvert eierskifte, med selger og kjøper</div>'
      : '<div class="lab">Slik har det vært</div>' +
        '<div class="stor">' + ANTALL_UTVALG + '</div>' +
        '<div class="under">biler slått opp én og én på vegvesen.no</div>' +
        '<div class="skille"></div>' +
        '<div class="stor2">Kun siste eier</div>' +
        '<div class="under2">Var bilen solgt videre, måtte revisor lete manuelt</div>';

    void el.offsetWidth;
    requestAnimationFrame(function(){ el.classList.add("spill"); });
  }

  /* --------------------------- datakilder: styrt av Neste/Tilbake (trinn 0-3) --------------------------- */

  function tegnKortRad(containerId, trinn){

    var container = $(containerId);
    if (!container) return;

    container.classList.remove("spill");

    var perKortMs = REDUSERT ? 0 : (trinn >= 3 ? 90 : 45);
    var visHistorikk = trinn >= 2;
    var visFarge = trinn >= 3;

    container.innerHTML = EKSEMPEL_BILER.map(function(b, i){

      var kortFarge = visFarge ? b.farge : "";
      var linjerHtml = "";

      if (visHistorikk){
        linjerHtml = '<div class="histliste2">' + b.hist.map(function(h, j){

          var erTreff = visFarge && j === b.treffIndex;
          var erForbi = visFarge && !erTreff;
          var tekst = h.forstegang ? "Førstegangsreg." : (h.fra + " &rarr; " + h.til);
          var cls = "hist-linje" + (erTreff ? " treff " + b.farge : "") + (erForbi ? " forbi" : "");
          var badge = erTreff ? '<span class="a num">' + b.avvik + 'd</span>' : "";

          return '<div class="' + cls + '"><span class="p">' + nbDato(h.dato) + " " + tekst + '</span>' + badge + '</div>';

        }).join("") + '</div>';
      }

      return '<div class="bilkort2 ' + kortFarge + '" style="--d:' + (i * perKortMs) + 'ms">' +
        '<div class="hode"><span class="regnr">' + b.regnr + '</span><span class="modell">' + b.modell + '</span></div>' +
        '<div class="bokf">Bokført <b>' + nbDato(b.bokf) + '</b></div>' +
        linjerHtml +
      '</div>';

    }).join("");

    void container.offsetWidth;
    requestAnimationFrame(function(){ container.classList.add("spill"); });
  }

  function tegnTellereEksempel(g, y, r){
    $("tellereEksempel").innerHTML =
      '<span><i class="sw g"></i><b class="num" id="tellEksG">' + g + '</b> 0–2 d</span>' +
      '<span><i class="sw y"></i><b class="num" id="tellEksY">' + y + '</b> 3–14 d</span>' +
      '<span><i class="sw r"></i><b class="num" id="tellEksR">' + r + '</b> 15+ d</span>';
  }

  /* ------- animert kontroll: én bil om gangen, én historikklinje om gangen ------- */

  var EKS_TIMERE = [];
  var EKS_LINJE_MS = 300;
  function eksSenere(fn, ms){ EKS_TIMERE.push(setTimeout(fn, ms)); }
  function eksStoppTimere(){ EKS_TIMERE.forEach(clearTimeout); EKS_TIMERE = []; }

  function byggKortEksempel(containerId){
    var container = $(containerId);
    if (!container) return [];
    container.classList.remove("spill");
    var perKortMs = REDUSERT ? 0 : 45;
    container.innerHTML = EKSEMPEL_BILER.map(function(b, i){
      var linjerHtml = '<div class="histliste2">' + b.hist.map(function(h){
        var tekst = h.forstegang ? "Førstegangsreg." : (h.fra + " &rarr; " + h.til);
        return '<div class="hist-linje"><span class="p">' + nbDato(h.dato) + " " + tekst + '</span><span class="a num"></span></div>';
      }).join("") + '</div>';
      return '<div class="bilkort2" data-i="' + i + '" style="--d:' + (i * perKortMs) + 'ms">' +
        '<div class="hode"><span class="regnr">' + b.regnr + '</span><span class="modell">' + b.modell + '</span></div>' +
        '<div class="bokf">Bokført <b>' + nbDato(b.bokf) + '</b></div>' +
        linjerHtml +
      '</div>';
    }).join("");
    void container.offsetWidth;
    requestAnimationFrame(function(){ container.classList.add("spill"); });
    return Array.prototype.slice.call(container.querySelectorAll(".bilkort2"));
  }

  function nullstillKontrollEksempel(kort){
    kort.forEach(function(k){
      k.classList.remove("aktiv", "g", "y", "r");
      Array.prototype.forEach.call(k.querySelectorAll(".hist-linje"), function(li){
        li.classList.remove("leser", "forbi", "treff", "g", "y", "r");
        li.querySelector(".a").textContent = "";
      });
    });
  }

  function settTreffEksempel(i, k){
    var b = EKSEMPEL_BILER[i];
    var lis = k.querySelectorAll(".hist-linje");
    Array.prototype.forEach.call(lis, function(li, j){
      li.classList.remove("leser");
      if (j < b.treffIndex) li.classList.add("forbi");
      if (j === b.treffIndex){ li.classList.add("treff", b.farge); li.querySelector(".a").textContent = b.avvik + "d"; }
    });
    k.classList.remove("aktiv");
    k.classList.add(b.farge);
    var id = b.farge === "g" ? "tellEksG" : b.farge === "y" ? "tellEksY" : "tellEksR";
    var el = $(id);
    if (el) el.textContent = String(parseInt(el.textContent, 10) + 1);
  }

  function kjorKontrollEksempel(kort){
    nullstillKontrollEksempel(kort);
    tegnTellereEksempel(0, 0, 0);
    eksStoppTimere();

    if (REDUSERT){
      EKSEMPEL_BILER.forEach(function(b, i){ settTreffEksempel(i, kort[i]); });
      $("antallC3").textContent = "Kjørt";
      $("antallC3").className = "antall antall-ferdig";
      return;
    }

    $("antallC3").textContent = "Kjører …";
    $("antallC3").className = "antall antall-vent";

    var t = 0;
    EKSEMPEL_BILER.forEach(function(b, i){
      var k = kort[i], lis = k.querySelectorAll(".hist-linje");
      for (var j = 0; j <= b.treffIndex; j++){
        (function(j){
          eksSenere(function(){
            kort.forEach(function(x){ x.classList.remove("aktiv"); });
            k.classList.add("aktiv");
            Array.prototype.forEach.call(lis, function(li){ li.classList.remove("leser"); });
            if (j > 0){
              var forrige = b.hist[j - 1];
              lis[j - 1].classList.add("forbi");
              if (forrige.forstegang || forrige.fra === "Forhandler"){
                lis[j - 1].querySelector(".a").textContent = dagerMellom(b.bokf, forrige.dato) + "d";
              }
            }
            lis[j].classList.add("leser");
          }, t);
        })(j);
        t += EKS_LINJE_MS;
      }
      (function(i, k){ eksSenere(function(){ settTreffEksempel(i, k); }, t); })(i, k);
      t += 120;
    });
    eksSenere(function(){ $("antallC3").textContent = "Kjørt"; $("antallC3").className = "antall antall-ferdig"; }, t);
  }

  function antallLinjerTekst(){
    var totalLinjer = 0, forstegang = 0;
    EKSEMPEL_BILER.forEach(function(b){
      totalLinjer += b.hist.length;
      b.hist.forEach(function(h){ if (h.forstegang) forstegang++; });
    });
    return totalLinjer + " linjer fra OFV: " + (totalLinjer - forstegang) + " eierskifter og " + forstegang + " førstegangsregistreringer";
  }

  function spillAvDatakilder(trinn){

    if (trinn === 1){
      tegnKortRad("bilkortrad1", 1);
      $("antallA1").textContent = EKSEMPEL_BILER.length + " biler";
    }

    if (trinn === 2){
      tegnKortRad("bilkortrad2", 2);
      $("antallB2").textContent = antallLinjerTekst();
    }

    if (trinn === 3){
      var kort3 = byggKortEksempel("bilkortrad3");
      kjorKontrollEksempel(kort3);
    }
  }

  /* ==========================================================
     SIDER - én liste driver Tilbake/Neste, prikknav, rapportnav og
     hvilken host (scenehost = fast 1600x900, frihost = fri/scrollbar)
     som vises. Presentasjonen forst, deretter rapporten, til slutt
     Sporsmal.
  ========================================================== */
  var SIDER = [
    {id:"forside", vert:"scene"},
    {id:"foretter-for", vert:"scene"},
    {id:"foretter-na", vert:"scene"},
    {id:"datakilder-0", vert:"scene"},
    {id:"datakilder-1", vert:"scene"},
    {id:"datakilder-2", vert:"scene"},
    {id:"datakilder-3", vert:"scene"},
    {id:"oversikt", vert:"fri", nav:"Oversikt"},
    {id:"oversiktforhandler", vert:"fri", nav:"Per forhandler"},
    {id:"solgte", vert:"fri", nav:"Solgte biler"}
  ];
  D.faner.forEach(function(f){ if (f.id !== "solgte") SIDER.push({id:f.id, vert:"fri", nav:f.tittel}); });
  SIDER.push({id:"analyse", vert:"fri", nav:"Analyse"});
  SIDER.push({id:"sporsmal", vert:"scene"});
  var naa = 0;
  function sideIndeks(id){ for (var i = 0; i < SIDER.length; i++) if (SIDER[i].id === id) return i; return -1; }

  function gaa(i){
    if (i < 0 || i >= SIDER.length) return;
    eksStoppTimere();
    naa = i;
    tegnSide();
  }

  function tegnSide(){

    var side = SIDER[naa];

    $("scenehost").hidden = side.vert !== "scene";
    $("frihost").hidden = side.vert !== "fri";
    $("bunn").classList.toggle("scene", side.vert === "scene" && naa < sideIndeks("oversikt"));

    SIDER.forEach(function(s, i){
      var e = $("side-" + s.id);
      if (e) e.hidden = (i !== naa);
    });

    $("tilbake").disabled = naa === 0;
    $("frem").disabled = naa === SIDER.length - 1;
    $("frem").textContent = naa === SIDER.length - 1 ? "Ferdig" : "Neste";

    Array.prototype.forEach.call($("prikknav").children, function(b, i){
      b.classList.toggle("passert", i < naa);
      b.setAttribute("aria-current", i === naa ? "true" : "false");
    });
    Array.prototype.forEach.call($("rapportnav").children, function(b){
      b.setAttribute("aria-current", b.getAttribute("data-side") === side.id ? "true" : "false");
    });

    if (side.vert === "scene") skalerScene();
    else { visFriSide(side.id); window.scrollTo(0, 0); }

    if (side.id === "foretter-for") spillAvForEtter(false);
    if (side.id === "foretter-na") spillAvForEtter(true);
    if (side.id.indexOf("datakilder-") === 0){
      var trinn = parseInt(side.id.split("-")[1], 10);
      spillAvDatakilder(trinn);
    }

    if (location.hash !== "#rediger" && location.hash !== "#" + side.id){
      try { history.replaceState(null, "", "#" + side.id); } catch (e) {}
    }
  }

  SIDER.forEach(function(s, i){
    var b = document.createElement("button");
    b.type = "button";
    b.setAttribute("aria-label", "Side " + (i + 1) + (s.nav ? ": " + s.nav : ""));
    b.addEventListener("click", function(){ gaa(i); });
    $("prikknav").appendChild(b);
    if (s.nav){
      var nb = el("button", {type:"button", "data-side":s.id, onclick:function(){ gaa(i); }}, s.nav);
      $("rapportnav").appendChild(nb);
    }
  });

  $("tilbake").addEventListener("click", function(){ gaa(naa - 1); });
  $("frem").addEventListener("click", function(){ gaa(naa + 1); });
  $("tilRapport").addEventListener("click", function(){ gaa(sideIndeks("oversikt")); });
  $("skrivUt").addEventListener("click", function(){ window.print(); });

  document.addEventListener("keydown", function(e){
    if (!$("adminBakteppe").hidden) return;
    var t = e.target && e.target.tagName;
    if (t === "INPUT" || t === "SELECT" || t === "TEXTAREA") return;
    if (e.key === "ArrowRight") gaa(naa + 1);
    if (e.key === "ArrowLeft") gaa(naa - 1);
  });

  Array.prototype.forEach.call(document.querySelectorAll("[data-gaa]"), function(b){
    b.addEventListener("click", function(){ gaa(parseInt(b.getAttribute("data-gaa"), 10)); });
  });

  /* --------------------------- skalering av den faste scenen --------------------------- */

  function skalerScene(){
    var bunnHoyde = 64;
    var tilgjengeligHoyde = Math.max(200, window.innerHeight - bunnHoyde);
    var s = Math.min(window.innerWidth / 1600, tilgjengeligHoyde / 900);
    var x = (window.innerWidth - 1600 * s) / 2;
    var y = Math.max(0, (tilgjengeligHoyde - 900 * s) / 2);
    $("scenehost").style.transform = "translate(" + x + "px," + y + "px) scale(" + s + ")";
  }
  window.addEventListener("resize", function(){ if (SIDER[naa].vert === "scene") skalerScene(); });

  /* ==========================================================
     INNLIMING: kolonner vi kjenner igjen fra en kopiert/eksportert
     versjon av "Kontroll solgte biler" (VBA-arket, Excel-arbeidsboken
     fra Databricks, eller CSV-eksporten fra denne rapporten).
  ========================================================== */
  var FELT = [
    {key:"kilde",        etiketter:["kilde"]},
    {key:"apitreff",     etiketter:["apitreff","api treff"]},
    {key:"regnr",        etiketter:["regnr","registreringsnummer"]},
    {key:"chassis",      etiketter:["chassisnummer","chassisnr","vin"]},
    {key:"modell",       etiketter:["modell"]},
    {key:"forstegang",   etiketter:["forstegangsregistrert","forstegangsregistrering","forstegangsreg"]},
    {key:"bokfort",      etiketter:["bokfortdato","bokfort"]},
    {key:"transaksjon",  etiketter:["transaksjonsdato","transaksjondato"]},
    {key:"regtype",      etiketter:["registreringstype","regtype"]},
    {key:"avvik",        etiketter:["dageravvik","avvik"]},
    {key:"selvhandel",   etiketter:["selvhandel"]},
    {key:"selger",       etiketter:["selger"]},
    {key:"kjoper",       etiketter:["kjoper","kjper"]}
  ];

  function normaliser(tekst){
    return String(tekst == null ? "" : tekst)
      .toLowerCase()
      .replace(/æ/g,"ae").replace(/ø/g,"o").replace(/å/g,"a")
      .replace(/[^a-z0-9]/g,"");
  }

  function finnFeltForOverskrift(overskrift){
    var n = normaliser(overskrift);
    if (!n) return null;
    for (var i = 0; i < FELT.length; i++){
      for (var j = 0; j < FELT[i].etiketter.length; j++){
        if (n.indexOf(FELT[i].etiketter[j]) >= 0) return FELT[i].key;
      }
    }
    return null;
  }

  function radTilObjekter(rader){

    if (!rader.length) return [];

    /* Overskriftsraden er ikke alltid forste rad (Excel-arket fra
       Databricks har tittel og KPI-bokser over tabellen) - let i de
       forste 20 radene etter raden med flest kjente kolonnenavn. */
    var hodeRad = -1, besteTreff = 0;
    for (var h = 0; h < Math.min(20, rader.length); h++){
      var treff = rader[h].map(finnFeltForOverskrift).filter(Boolean).length;
      if (treff > besteTreff){ besteTreff = treff; hodeRad = h; }
    }

    var kolonneTilFelt, startRad;
    if (besteTreff >= 3){
      kolonneTilFelt = rader[hodeRad].map(finnFeltForOverskrift);
      /* Samme felt to ganger (f.eks. "Bokfort dato" og "Bokfort inn") - forste vinner */
      var sett = {};
      kolonneTilFelt = kolonneTilFelt.map(function(f){ if (!f || sett[f]) return null; sett[f] = 1; return f; });
      startRad = hodeRad + 1;
    } else {
      kolonneTilFelt = FELT.filter(function(f){ return f.key !== "selvhandel"; }).map(function(f){ return f.key; });
      startRad = 0;
    }

    var resultat = [];

    for (var r = startRad; r < rader.length; r++){
      var rad = rader[r];
      var tom = rad.every(function(c){ return String(c || "").trim() === ""; });
      if (tom) continue;

      var obj = {};
      for (var c = 0; c < kolonneTilFelt.length; c++){
        if (kolonneTilFelt[c]) obj[kolonneTilFelt[c]] = String(rad[c] == null ? "" : rad[c]).trim();
      }
      if (!obj.regnr && !obj.chassis) continue;
      resultat.push(obj);
    }

    return resultat;
  }

  function tsvTilRader(tekst){
    return tekst.replace(/\r/g,"").split("\n").filter(function(l){ return l.length > 0; }).map(function(l){ return l.split("\t"); });
  }

  /* Tekstdato fra Excel/innliming -> ISO (aaaa-mm-dd), ellers null */
  function tolkDato(t){
    t = String(t == null ? "" : t).trim();
    if (!t || t === "-" || t === "–") return null;
    var m = t.match(/^(\d{4})-(\d{1,2})-(\d{1,2})/);
    if (m) return isoAv(+m[1], +m[2], +m[3]);
    m = t.match(/^(\d{1,2})[.\/-](\d{1,2})[.\/-](\d{2,4})$/);
    if (m){
      var a = +m[1], b = +m[2], y = +m[3];
      if (y < 100) y += 2000;
      /* Norsk dd.mm.aaaa; amerikansk m/d/aa bare nar dag > 12 avslorer det */
      if (t.indexOf("/") >= 0 && b > 12) return isoAv(y, a, b);
      return isoAv(y, b, a);
    }
    if (/^\d{5}(\.\d+)?$/.test(t)){
      var d = new Date(Date.UTC(1899, 11, 30) + Math.floor(+t) * 86400000);
      return isoAv(d.getUTCFullYear(), d.getUTCMonth() + 1, d.getUTCDate());
    }
    return null;
  }
  function isoAv(y, m, d){
    if (m < 1 || m > 12 || d < 1 || d > 31) return null;
    return y + "-" + (m < 10 ? "0" : "") + m + "-" + (d < 10 ? "0" : "") + d;
  }

  /* Innlimt rad -> samme felt som Databricks-radene i Solgte biler */
  function vurderingFor(avvik, selvhandel, kontrollert){
    if (!kontrollert) return ["Ikke kontrollert", "n"];
    if (selvhandel === "Ja") return ["Avvik - selvhandel", "r"];
    var k = fargeklasse(avvik);
    if (k === "g") return ["OK (0-" + D.grenser.gronnMaks + " dager)", "g"];
    if (k === "y") return ["Varsel (" + (D.grenser.gronnMaks + 1) + "-" + D.grenser.gulMaks + " dager)", "y"];
    return ["Avvik (" + (D.grenser.gulMaks + 1) + "+ dager)", "r"];
  }
  function fraInnlimt(o, forhandler){
    var v = avvikTall(o.avvik);
    var selvhandel = /^ja$/i.test(o.selvhandel || "") ? "Ja" : (o.selvhandel ? "Nei" : "");
    var vt = vurderingFor(v, selvhandel, v != null);
    return {
      Vurdering: vt[0], _tone: vt[1], Forhandler: forhandler, DagerAvvik: v,
      RegnrInput: o.regnr || "", Chassisnummer: o.chassis || "", Modell: o.modell || "",
      BokfortDato: tolkDato(o.bokfort), KontrollTransaksjonDato: tolkDato(o.transaksjon),
      RegistreringsType: o.regtype || "", Forstegangsregistrert: tolkDato(o.forstegang),
      Selger: o.selger || "", Kjoper: o.kjoper || "", Selvhandel: selvhandel,
      Kilde: o.kilde || "", ApiTreff: o.apitreff || "", Kontrollert: v != null ? "Ja" : "Nei",
      _innlimt: true
    };
  }

  /* ==========================================================
     ADMIN-PANEL: opptil 20 forhandlerdatasett (navn + innliming)
  ========================================================== */

  var SLOTER = []; /* {id, navn, rader} */
  var slotTeller = 0;

  function nySlotId(){ slotTeller++; return "slot" + slotTeller; }

  function byggSlotHtml(slot){
    return (
      '<div class="forhandlerslot" data-id="' + slot.id + '">' +
        '<button type="button" class="fjernslot" data-fjern="' + slot.id + '">Fjern</button>' +
        '<input type="text" class="slot-navn" data-navn="' + slot.id + '" placeholder="Forhandlernavn" value="' + esc(slot.navn) + '">' +
        '<textarea class="limfelt" data-lim="' + slot.id + '" placeholder="Lim inn tabellen fra Kontroll solgte biler her (Ctrl+V) ..."></textarea>' +
        '<div class="bilde-rad" style="margin-top:6px">' +
          '<input type="file" accept=".xlsx,.xls" data-fil="' + slot.id + '">' +
        '</div>' +
        '<div class="slot-status' + (slot.rader.length ? " ok" : "") + '" data-status="' + slot.id + '">' +
          (slot.rader.length ? slot.rader.length + " biler lagret." : "") + '</div>' +
      '</div>'
    );
  }

  function tegnSloter(){
    $("forhandlersloter").innerHTML = SLOTER.map(byggSlotHtml).join("");
    $("leggTilForhandler").disabled = SLOTER.length >= MAKS_FORHANDLERE;
    koblSlotHendelser();
  }

  function leggTilSlot(navn){
    if (SLOTER.length >= MAKS_FORHANDLERE) return;
    var slot = {id: nySlotId(), navn: navn || ("Forhandler " + (SLOTER.length + 1)), rader: []};
    SLOTER.push(slot);
    tegnSloter();
    return slot;
  }

  function fjernSlot(id){
    SLOTER = SLOTER.filter(function(s){ return s.id !== id; });
    tegnSloter();
    lagreSloter();
    tegnAlt();
  }

  function finnSlot(id){
    for (var i = 0; i < SLOTER.length; i++) if (SLOTER[i].id === id) return SLOTER[i];
    return null;
  }

  function settSlotStatus(id, tekst, klasse){
    var e = document.querySelector('[data-status="' + id + '"]');
    if (!e) return;
    e.textContent = tekst || "";
    e.className = "slot-status" + (klasse ? " " + klasse : "");
  }

  function tolkSlotTekst(id){
    var ta = document.querySelector('[data-lim="' + id + '"]');
    var slot = finnSlot(id);
    if (!slot || !ta) return;

    var tekst = ta.value;
    if (!tekst.trim()){ settSlotStatus(id, "Lim inn tabellen først.", "feil"); return; }

    var rader = radTilObjekter(tsvTilRader(tekst));
    if (!rader.length){ settSlotStatus(id, "Fant ingen rader med regnr eller chassisnummer.", "feil"); return; }

    slot.rader = rader;
    settSlotStatus(id, rader.length + " biler tolket.", "ok");
    lagreSloter();
    tegnAlt();
  }

  function koblSlotHendelser(){

    Array.prototype.forEach.call(document.querySelectorAll("[data-fjern]"), function(b){
      b.onclick = function(){ fjernSlot(b.getAttribute("data-fjern")); };
    });

    Array.prototype.forEach.call(document.querySelectorAll("[data-navn]"), function(inp){
      inp.oninput = function(){
        var slot = finnSlot(inp.getAttribute("data-navn"));
        if (slot){ slot.navn = inp.value.trim() || slot.navn; lagreSloter(); tegnAlt(); }
      };
    });

    Array.prototype.forEach.call(document.querySelectorAll("[data-lim]"), function(ta){
      var id = ta.getAttribute("data-lim");
      ta.addEventListener("paste", function(){ setTimeout(function(){ tolkSlotTekst(id); }, 30); });
      ta.addEventListener("blur", function(){ if (ta.value.trim()) tolkSlotTekst(id); });
    });

    Array.prototype.forEach.call(document.querySelectorAll("[data-fil]"), function(inp){
      var id = inp.getAttribute("data-fil");
      inp.addEventListener("change", function(e){
        var fil = e.target.files && e.target.files[0];
        if (!fil) return;
        if (typeof XLSX === "undefined"){ settSlotStatus(id, "Excel-leseren er ikke tilgjengelig - kopier tabellen og lim inn i stedet.", "feil"); return; }
        settSlotStatus(id, "Leser " + fil.name + " ...", "");
        var leser = new FileReader();
        leser.onload = function(ev){
          try {
            var data = new Uint8Array(ev.target.result);
            var wb = XLSX.read(data, {type:"array"});
            var navn = wb.SheetNames.filter(function(n){ return normaliser(n).indexOf("kontrollsolgtebiler") >= 0; })[0] || wb.SheetNames[0];
            var aoa = XLSX.utils.sheet_to_json(wb.Sheets[navn], {header:1, raw:false, defval:""});
            var rader = radTilObjekter(aoa);
            var slot = finnSlot(id);
            if (!rader.length){ settSlotStatus(id, 'Fant ingen biler i fanen "' + navn + '".', "feil"); return; }
            slot.rader = rader;
            settSlotStatus(id, rader.length + ' biler lest fra fanen "' + navn + '".', "ok");
            lagreSloter();
            tegnAlt();
          } catch (err) {
            settSlotStatus(id, "Klarte ikke å lese filen: " + err.message, "feil");
          }
        };
        leser.readAsArrayBuffer(fil);
      });
    });
  }

  $("leggTilForhandler").addEventListener("click", function(){ leggTilSlot(); });

  function lagreSloter(){
    lagre(NOKKEL_SLOTER, JSON.stringify(SLOTER.map(function(s){ return {id:s.id, navn:s.navn, rader:s.rader}; })));
  }

  /* ==========================================================
     Samlet datasett for Solgte biler: Databricks-kjoringen forst,
     deretter alle innlimte forhandlere.
  ========================================================== */

  var SOLGTE = null;   // fanen "solgte" fra Databricks (alltid med, evt. uten rader)
  D.faner.forEach(function(f){ if (f.id === "solgte") SOLGTE = f; });
  var SOLGTE_DB = SOLGTE.tabeller[0].rader;

  function datasett(){
    var ut = [];
    if (SOLGTE_DB.length) ut.push({navn: D.selskap, kilde: "Databricks", rader: SOLGTE_DB});
    SLOTER.forEach(function(slot){
      if (slot.rader.length) ut.push({navn: slot.navn, kilde: "Innlimt", rader: slot.rader.map(function(r){ return fraInnlimt(r, slot.navn); })});
    });
    return ut;
  }
  function alleSolgte(){
    var ut = [];
    datasett().forEach(function(d){ ut = ut.concat(d.rader); });
    return ut;
  }
  function telling(rader){
    var t = {g:0, y:0, r:0, n:0, total:rader.length};
    rader.forEach(function(r){ t[r._tone || "n"]++; });
    return t;
  }

  /* Kalles nar innlimte data endres */
  function tegnAlt(){
    var rader = alleSolgte();
    if (solgteTabell) solgteTabell.settRader(rader);
    if (innlimtTeller) innlimtTeller.textContent = String(SLOTER.filter(function(s){ return s.rader.length; }).length);
    var bt = $("navtall-solgte");
    if (bt) settNavtall(bt, telling(rader).r);
    tegnDatabricksDatasett();
    var side = SIDER[naa];
    if (side.vert === "fri" && side.id !== "solgte") visFriSide(side.id);
  }

  function tegnDatabricksDatasett(){
    var e = $("databricksDatasett");
    e.textContent = "";
    if (!SOLGTE_DB.length) return;
    e.appendChild(el("div", {cls:"databricks-datasett"},
      el("div", {cls:"sone", text:"Fra Databricks-kjøringen (låst)"}),
      el("b", {text: D.selskap}), " – " + nf.format(SOLGTE_DB.length) + " biler"));
  }

  /* ==========================================================
     Filtrerbar tabell (sortering, sok, kolonnefiltre, statusknapper,
     kolonnevalg og CSV-eksport)
  ========================================================== */
  function Tabell(spec, valg){
    this.spec = spec;
    this.valg = valg || {};
    this.cols = spec.kolonner;
    this.statusCol = this.cols.filter(function(c){ return c.type === "status"; })[0] || null;
    this.state = {q:"", toner:{g:true, y:true, r:true, n:true}, f:{}, sort:null, skjult:{}, limit:SIDE};
    this.settRader(spec.rader, true);
    this.root = this.bygg();
  }

  Tabell.prototype.settRader = function(rader, forste){
    var self = this;
    this.rows = rader;
    this.cols.forEach(function(c){
      if (c.type === "text" || c.type === "status"){
        var sett = {};
        rader.forEach(function(r){ sett[r[c.key] == null ? "" : String(r[c.key])] = 1; });
        c._verdier = Object.keys(sett).sort(coll.compare);
        c._velg = c._verdier.length > 1 && c._verdier.length <= 30;
      }
    });
    if (!forste){ this.byggHode(); this.tegn(); }
  };

  Tabell.prototype.bygg = function(){
    var self = this;
    var kort = el("div");

    var sok = el("input", {type:"search", placeholder:"Søk i alle kolonner …", "aria-label":"Søk i tabellen",
      oninput:function(){ self.state.q = this.value.trim().toLowerCase(); self.state.limit = SIDE; self.tegn(); }});
    this.sokInput = sok;
    var linje = el("div", {cls:"verktoylinje"}, el("label", {cls:"sokefelt"}, sokIkon(), sok));
    if (this.statusCol){
      this.knapper = el("div", {cls:"bukettbrytere", role:"group", "aria-label":"Vis status"});
      linje.appendChild(this.knapper);
    }
    linje.appendChild(el("span", {cls:"fyll"}));

    var meny = el("div", {cls:"meny"});
    this.cols.forEach(function(c){
      var cb = el("input", {type:"checkbox", checked:true, onchange:function(){
        if (this.checked) delete self.state.skjult[c.key]; else self.state.skjult[c.key] = 1;
        self.byggHode(); self.tegn();
      }});
      meny.appendChild(el("label", null, cb, c.label));
    });
    linje.appendChild(el("details", {cls:"kolmeny"}, el("summary", {cls:"knapp liten rolig"}, "Kolonner"), meny));
    linje.appendChild(el("button", {cls:"knapp liten rolig", type:"button", text:"Nullstill filtre", onclick:function(){ self.nullstill(); }}));
    linje.appendChild(el("button", {cls:"knapp liten", type:"button", text:"Last ned CSV", title:"Laster ned radene som vises nå (etter filtrering)", onclick:function(){ self.csv(); }}));
    kort.appendChild(linje);

    this.thead = el("thead");
    this.tbody = el("tbody");
    kort.appendChild(el("div", {cls:"tabellramme rull"}, el("table", {cls:"tabell"}, this.thead, this.tbody)));
    this.info = el("span");
    this.mer = el("button", {cls:"knapp liten rolig", type:"button", text:"Vis flere", onclick:function(){ self.state.limit += SIDE * 5; self.tegn(); }});
    this.alle = el("button", {cls:"knapp liten rolig", type:"button", text:"Vis alle", onclick:function(){ self.state.limit = Infinity; self.tegn(); }});
    kort.appendChild(el("div", {cls:"tabellfot"}, this.info, el("span", {cls:"fyll"}), this.mer, this.alle));
    this.byggHode();
    this.tegn();
    return kort;
  };

  function sokIkon(){
    var g = s("svg", {width:14, height:14, viewBox:"0 0 24 24", fill:"none", stroke:"currentColor", "stroke-width":2.2, "stroke-linecap":"round"});
    g.appendChild(s("circle", {cx:11, cy:11, r:7}));
    g.appendChild(s("line", {x1:16.5, y1:16.5, x2:21, y2:21}));
    return g;
  }

  Tabell.prototype.synlige = function(){
    var sk = this.state.skjult;
    return this.cols.filter(function(c){ return !sk[c.key]; });
  };

  Tabell.prototype.byggHode = function(){
    var self = this;
    this.thead.textContent = "";
    var tr1 = el("tr", {cls:"etiketter"}), tr2 = el("tr", {cls:"filterfelt"});
    this.synlige().forEach(function(c){
      var sortert = self.state.sort && self.state.sort.key === c.key;
      var pil = sortert ? (self.state.sort.dir > 0 ? "▲" : "▼") : "⇅";
      tr1.appendChild(el("th", {cls: sortert ? "sortert" : null, title:"Klikk for å sortere",
        "aria-sort": sortert ? (self.state.sort.dir > 0 ? "ascending" : "descending") : null,
        onclick:function(){ self.sorter(c.key); }}, c.label, el("span", {cls:"sortpil", text:pil})));
      tr2.appendChild(el("th", null, self.filterFelt(c)));
    });
    this.thead.appendChild(tr1);
    this.thead.appendChild(tr2);
  };

  Tabell.prototype.filterFelt = function(c){
    var self = this, f = this.state.f[c.key] || {};
    function sett(k, v){
      var cur = self.state.f[c.key] || {};
      cur[k] = v;
      self.state.f[c.key] = cur;
      self.state.limit = SIDE;
      self.tegn();
    }
    function aktiv(inp, v){ inp.classList.toggle("aktiv", v != null && v !== ""); }
    if (c.type === "num" || c.type === "date"){
      var typ = c.type === "num" ? "number" : "date";
      var a = el("input", {type:typ, placeholder:"min", title: c.type === "date" ? "Fra og med" : "Minimum", value: f.min || null,
        oninput:function(){ aktiv(this, this.value); sett("min", this.value); }});
      var b = el("input", {type:typ, placeholder:"maks", title: c.type === "date" ? "Til og med" : "Maksimum", value: f.max || null,
        oninput:function(){ aktiv(this, this.value); sett("max", this.value); }});
      aktiv(a, f.min); aktiv(b, f.max);
      return el("div", {cls:"spenn"}, a, b);
    }
    if (c._velg || (f.eq && c._verdier)){
      var sel = el("select", {"aria-label":"Filtrer " + c.label, onchange:function(){ aktiv(this, this.value); sett("eq", this.value); }},
        el("option", {value:"", text:"Alle"}));
      c._verdier.forEach(function(v){
        sel.appendChild(el("option", {value:"=" + v, text: v === "" ? "(tom)" : v, selected: f.eq === "=" + v}));
      });
      aktiv(sel, f.eq);
      return sel;
    }
    var inp = el("input", {type:"text", placeholder:"Filtrer …", value: f.like || null, "aria-label":"Filtrer " + c.label,
      oninput:function(){ aktiv(this, this.value); sett("like", this.value.trim().toLowerCase()); }});
    aktiv(inp, f.like);
    return inp;
  };

  Tabell.prototype.settFilter = function(key, f){
    if (f) this.state.f[key] = f; else delete this.state.f[key];
    this.state.limit = SIDE;
    this.byggHode();
    this.tegn();
  };

  Tabell.prototype.sorter = function(key){
    var st = this.state.sort;
    if (!st || st.key !== key) this.state.sort = {key:key, dir:1};
    else if (st.dir === 1) st.dir = -1;
    else this.state.sort = null;
    this.byggHode();
    this.tegn();
  };

  Tabell.prototype.nullstill = function(){
    this.state.q = ""; this.state.toner = {g:true, y:true, r:true, n:true}; this.state.f = {}; this.state.sort = null; this.state.limit = SIDE;
    this.sokInput.value = "";
    this.byggHode();
    this.tegn();
  };

  /* Vis bare en tone (eller alle igjen hvis den allerede var eneste) */
  Tabell.prototype.visBareTone = function(tone){
    var t = this.state.toner, bare = t[tone] && Object.keys(t).every(function(k){ return k === tone || !t[k]; });
    this.state.toner = bare ? {g:true, y:true, r:true, n:true} : {g:false, y:false, r:false, n:false};
    if (!bare) this.state.toner[tone] = true;
    this.state.limit = SIDE;
    this.tegn();
  };

  Tabell.prototype.passerUtenTone = function(r){
    var cols = this.cols, f = this.state.f;
    for (var i = 0; i < cols.length; i++){
      var c = cols[i], ff = f[c.key];
      if (!ff) continue;
      var v = r[c.key];
      if (ff.eq){ if ((v == null ? "" : String(v)) !== ff.eq.slice(1)) return false; }
      if (ff.like){ if (vis(c, v).toLowerCase().indexOf(ff.like) < 0) return false; }
      if (ff.min){
        if (v == null || v === "") return false;
        if (c.type === "num" ? v < parseFloat(ff.min) : v < ff.min) return false;
      }
      if (ff.max){
        if (v == null || v === "") return false;
        if (c.type === "num" ? v > parseFloat(ff.max) : v > ff.max) return false;
      }
    }
    if (this.state.q){
      var syn = this.synlige(), hay = "";
      for (var j = 0; j < syn.length; j++) hay += " " + vis(syn[j], r[syn[j].key]);
      hay = hay.toLowerCase();
      var ord = this.state.q.split(/\s+/);
      for (var k = 0; k < ord.length; k++) if (hay.indexOf(ord[k]) < 0) return false;
    }
    return true;
  };

  Tabell.prototype.filtrert = function(){
    var self = this, sc = this.statusCol, toner = this.state.toner;
    var base = this.rows.filter(function(r){ return self.passerUtenTone(r); });
    this._base = base;
    this._toneTelling = {g:0, y:0, r:0, n:0};
    if (sc) base.forEach(function(r){ self._toneTelling[radTone(sc, r)]++; });
    var ut = !sc ? base : base.filter(function(r){ return toner[radTone(sc, r)]; });
    var st = this.state.sort;
    if (st){
      var col = this.cols.filter(function(c){ return c.key === st.key; })[0];
      ut = ut.slice().sort(function(a, b){
        var x = a[st.key], y = b[st.key];
        var xe = x == null || x === "", ye = y == null || y === "";
        if (xe && ye) return 0;
        if (xe) return 1;
        if (ye) return -1;
        var r = col.type === "num" ? x - y : coll.compare(String(x), String(y));
        return r * st.dir;
      });
    }
    return ut;
  };

  Tabell.prototype.tegnKnapper = function(){
    if (!this.knapper) return;
    var self = this, t = this._toneTelling;
    this.knapper.textContent = "";
    ["g", "y", "r", "n"].forEach(function(tone){
      if (!t[tone] && self.state.toner[tone]) return;
      self.knapper.appendChild(el("button", {cls:"bukettknapp " + tone, type:"button", "aria-pressed": String(!!self.state.toner[tone]),
        title:"Vis/skjul " + TONE_NAVN[tone].toLowerCase(),
        onclick:function(){ self.state.toner[tone] = !self.state.toner[tone]; self.state.limit = SIDE; self.tegn(); }},
        el("span", {cls:"prikk"}), TONE_NAVN[tone], el("span", {cls:"n", text: nf.format(t[tone])})));
    });
  };

  Tabell.prototype.tegn = function(){
    var rader = this.filtrert(), syn = this.synlige();
    this._sist = rader;
    this.tegnKnapper();
    this.tbody.textContent = "";
    var frag = document.createDocumentFragment();
    var n = Math.min(rader.length, this.state.limit);
    for (var i = 0; i < n; i++){
      var r = rader[i], tr = document.createElement("tr");
      for (var j = 0; j < syn.length; j++){
        var c = syn[j], v = r[c.key], txt = vis(c, v);
        var td = document.createElement("td");
        td.className = "t-" + c.type + (txt === "" ? " tom" : "") + (c.key === "RegnrInput" || c.key === "RegNo" ? " reg" : "");
        if (txt !== "" && (c.type === "status" || c.toneKey)){
          var tone = radTone(c, r);
          td.appendChild(el("span", {cls:"badge " + tone}, el("span", {cls:"dot"}), txt));
          td.title = txt;
        } else {
          td.textContent = txt === "" ? "–" : txt;
          if (txt.length > 40) td.title = txt;
        }
        tr.appendChild(td);
      }
      frag.appendChild(tr);
    }
    if (!rader.length){
      frag.appendChild(el("tr", null, el("td", {colspan: syn.length, cls:"ingentreff", text:"Ingen rader matcher filtrene."})));
    }
    this.tbody.appendChild(frag);
    var tot = this.rows.length;
    this.info.textContent = "Viser " + nf.format(n) + (n < rader.length ? " av " + nf.format(rader.length) + " treff" : "") +
      (rader.length !== tot ? " (filtrert fra " + nf.format(tot) + " rader)" : n === tot ? " rader" : "");
    this.mer.style.display = n < rader.length ? "" : "none";
    this.alle.style.display = n < rader.length ? "" : "none";
    if (this.valg.vedEndring) this.valg.vedEndring(this);
  };

  Tabell.prototype.csv = function(){
    var syn = this.synlige(), rader = this._sist || [];
    function q(x){ x = String(x == null ? "" : x); return /[";\n\r]/.test(x) ? '"' + x.replace(/"/g, '""') + '"' : x; }
    var linjer = [syn.map(function(c){ return q(c.label); }).join(";")];
    rader.forEach(function(r){ linjer.push(syn.map(function(c){ return q(vis(c, r[c.key])); }).join(";")); });
    var blob = new Blob(["﻿" + linjer.join("\r\n")], {type:"text/csv;charset=utf-8"});
    var a = el("a", {href: URL.createObjectURL(blob), download: (D.selskap + " - " + this.spec.tittel).replace(/[\\/:*?"<>|]/g, "_") + ".csv"});
    document.body.appendChild(a); a.click(); a.remove();
    setTimeout(function(){ URL.revokeObjectURL(a.href); }, 1000);
  };

  /* Tabellpanel med tittel - historikk er sammenlagt og tegnes forst nar det apnes */
  function tabellPanel(spec, lukket, valg){
    var hode = el("div", {cls:"panel-hode"},
      lukket ? el("span", {cls:"pil", text:"▶"}) : null,
      el("div", null,
        el("h3", null, spec.tittel, el("span", {cls:"ant", text:"(" + nf.format(spec.rader.length) + ")"})),
        spec.beskrivelse ? el("p", {text: spec.beskrivelse}) : null));
    if (!lukket){
      var t = new Tabell(spec, valg);
      var p = el("div", {cls:"panel", id:"t-" + spec.id}, hode, t.root);
      p._tabell = t; p._ant = hode.querySelector(".ant");
      return p;
    }
    var d = el("details", {cls:"panel", id:"t-" + spec.id}, el("summary", null, hode));
    d.addEventListener("toggle", function(){
      if (d.open && !d._tabell){ d._tabell = new Tabell(spec, valg); d.appendChild(d._tabell.root); }
    });
    return d;
  }

  /* ==========================================================
     Grafer (ren SVG, BDO-farger)
  ========================================================== */
  function pent(maks){
    var steg = Math.max(1, Math.ceil(maks / 4));
    var p = Math.pow(10, Math.floor(Math.log10(steg))), m = steg / p;
    return (m <= 1 ? 1 : m <= 2 ? 2 : m <= 5 ? 5 : 10) * p * 4;
  }
  function tekstBredde(t, px){ return String(t).length * (px || 12) * 0.56; }

  function hbar(g, svgEl, B){
    var data = g.data, sum = data.reduce(function(a, d){ return a + d.value; }, 0);
    var maks = Math.max.apply(null, data.map(function(d){ return d.value; }).concat([1]));
    var lw = Math.min(B * 0.42, Math.max.apply(null, data.map(function(d){ return tekstBredde(d.label, 12.5); })) + 12);
    var bw = Math.max(40, B - lw - 64), rh = 28, H = data.length * rh + 4;
    svgEl.setAttribute("viewBox", "0 0 " + B + " " + H); svgEl.setAttribute("height", H);
    data.forEach(function(d, i){
      var y = i * rh, w = Math.max(d.value > 0 ? 3 : 0, d.value / maks * bw);
      var lab = d.label;
      while (tekstBredde(lab, 12.5) > lw - 10 && lab.length > 4) lab = lab.slice(0, -2);
      if (lab !== d.label) lab = lab.replace(/.$/, "…");
      svgEl.appendChild(s("text", {x: lw - 10, y: y + rh / 2 + 4, "text-anchor":"end", "font-size":12.5, fill:"var(--tekst)"}, lab));
      svgEl.appendChild(s("rect", {x: lw, y: y + 8, width: w, height: rh - 16, fill: d.tone ? TONE_FARGE[d.tone] : d.muted ? "var(--flate2)" : "var(--skifer)"}));
      svgEl.appendChild(s("text", {x: lw + w + 8, y: y + rh / 2 + 4, "font-size":12, fill:"var(--dempet)", "font-weight":600}, nf.format(d.value)));
      var hit = s("rect", {x:0, y:y, width:B, height:rh, fill:"transparent"});
      hit.addEventListener("mousemove", function(ev){ visTips(ev, [d.label, nf.format(d.value) + " (" + pf.format(sum ? d.value / sum : 0) + ")"]); });
      hit.addEventListener("mouseleave", skjulTips);
      svgEl.appendChild(hit);
    });
  }

  function soyler(g, svgEl, B){
    var data = g.data, H = 230, top = 14, bunn = 26, venstre = 36;
    var maks = pent(Math.max.apply(null, data.map(function(d){ return d.value; }).concat([1])));
    var ih = H - top - bunn, iw = B - venstre - 4, steg = iw / data.length;
    var bw = Math.max(4, Math.min(48, steg * 0.62));
    svgEl.setAttribute("viewBox", "0 0 " + B + " " + H); svgEl.setAttribute("height", H);
    for (var k = 0; k <= 4; k++){
      var v = maks * k / 4, y = top + ih - ih * k / 4;
      svgEl.appendChild(s("line", {x1: venstre, x2: B, y1: y, y2: y, stroke: k ? "var(--kant)" : "var(--tekst)", "stroke-width":1}));
      svgEl.appendChild(s("text", {x: venstre - 6, y: y + 4, "text-anchor":"end", "font-size":11, fill:"var(--dempet)"}, nf.format(v)));
    }
    var hvert = Math.ceil(data.length / Math.max(1, Math.floor(iw / 46)));
    data.forEach(function(d, i){
      var x = venstre + i * steg + (steg - bw) / 2, h = d.value / maks * ih;
      if (d.value > 0) svgEl.appendChild(s("rect", {x:x, y: top + ih - h, width: bw, height: h, fill: d.tone ? TONE_FARGE[d.tone] : "var(--skifer)"}));
      if (i % hvert === 0 || data.length <= 12)
        svgEl.appendChild(s("text", {x: x + bw / 2, y: H - 8, "text-anchor":"middle", "font-size":11, fill:"var(--dempet)"}, d.label));
      var hit = s("rect", {x: venstre + i * steg, y: top, width: steg, height: ih, fill:"transparent"});
      hit.addEventListener("mousemove", function(ev){ visTips(ev, [d.label, nf.format(d.value)]); });
      hit.addEventListener("mouseleave", skjulTips);
      svgEl.appendChild(hit);
    });
    var mi = 0; data.forEach(function(d, i){ if (d.value > data[mi].value) mi = i; });
    if (data[mi] && data[mi].value > 0){
      var xm = venstre + mi * steg + steg / 2, ym = top + ih - data[mi].value / maks * ih - 6;
      svgEl.insertBefore(s("text", {x: xm, y: Math.max(10, ym), "text-anchor":"middle", "font-size":11.5, "font-weight":700, fill:"var(--tekst)"}, nf.format(data[mi].value)), svgEl.firstChild);
    }
  }

  function stabel(g, svgEl, B){
    var rh = 56, H = g.rows.length * rh;
    svgEl.setAttribute("viewBox", "0 0 " + B + " " + H); svgEl.setAttribute("height", H);
    g.rows.forEach(function(row, i){
      var y = i * rh, sum = row.segments.reduce(function(a, d){ return a + d.value; }, 0), x = 0;
      svgEl.appendChild(s("text", {x:0, y: y + 14, "font-size":12.5, "font-weight":700, fill:"var(--tekst)"}, row.label));
      svgEl.appendChild(s("text", {x:B, y: y + 14, "text-anchor":"end", "font-size":12, fill:"var(--dempet)"}, nf.format(sum) + " biler"));
      if (!sum) return;
      row.segments.forEach(function(seg){
        if (!seg.value) return;
        var w = seg.value / sum * B;
        var r = s("rect", {x:x, y: y + 22, width: Math.max(1, w - 2), height: 20, fill: TONE_FARGE[seg.tone]});
        r.addEventListener("mousemove", function(ev){ visTips(ev, [row.label + " – " + seg.label, nf.format(seg.value) + " av " + nf.format(sum) + " (" + pf.format(seg.value / sum) + ")"]); });
        r.addEventListener("mouseleave", skjulTips);
        svgEl.appendChild(r);
        var lbl = pf.format(seg.value / sum);
        if (w > tekstBredde(lbl, 11.5) + 12)
          svgEl.appendChild(s("text", {x: x + 7, y: y + 36, "font-size":11.5, "font-weight":700, fill:"#ffffff", "pointer-events":"none"}, lbl));
        x += w;
      });
    });
  }

  var TEGNERE = {hbar: hbar, columns: soyler, stack: stabel};
  var grafPaneler = [];

  function grafPanel(g){
    var svgEl = s("svg", {role:"img", "aria-label": g.tittel});
    var kropp = el("div", {cls:"grafkropp"}, svgEl);
    var p = el("div", {cls:"panel" + (g.bred ? " bred2" : "")},
      el("div", {cls:"panel-hode"}, el("h3", {text: g.tittel}), g.undertittel ? el("p", {text: g.undertittel}) : null));
    var leg = null;
    if (g.type === "stack"){
      var sett = {}, rekke = [];
      g.rows.forEach(function(r){ r.segments.forEach(function(sg){ var k = sg.label + "|" + sg.tone; if (sg.value && !sett[k]){ sett[k] = 1; rekke.push([sg.label, sg.tone]); } }); });
      leg = rekke;
    } else if (g.statusLegend){
      leg = [["OK", "g"], ["Varsel", "y"], ["Avvik", "r"]];
    }
    if (leg) p.appendChild(el("div", {cls:"forklaring"}, leg.map(function(l){
      return el("span", null, el("i", {style:"background:" + TONE_FARGE[l[1]]}), l[0]);
    })));
    p.appendChild(kropp);
    var tb = el("tbody");
    if (g.type === "stack") g.rows.forEach(function(r){ r.segments.forEach(function(sg){
      tb.appendChild(el("tr", null, el("td", {text: r.label}), el("td", {text: sg.label}), el("td", {cls:"n", text: nf.format(sg.value)})));
    }); });
    else g.data.forEach(function(d){ tb.appendChild(el("tr", null, el("td", {text: d.label}), el("td", {cls:"n", text: nf.format(d.value)}))); });
    p.appendChild(el("details", {cls:"talltabell"}, el("summary", {text:"Vis tallene"}), el("table", null, tb)));
    p._tegn = function(){
      if (!kropp.clientWidth) return;
      var B = Math.max(260, kropp.clientWidth - 36);
      svgEl.textContent = "";
      TEGNERE[g.type](g, svgEl, B);
    };
    grafPaneler.push(p);
    return p;
  }
  function tegnSynligeGrafer(){ grafPaneler.forEach(function(p){ if (p.offsetParent) p._tegn(); }); }
  var rt;
  window.addEventListener("resize", function(){ clearTimeout(rt); rt = setTimeout(tegnSynligeGrafer, 120); });

  /* ==========================================================
     Rapportsidene
  ========================================================== */
  var friSider = {};

  function sidehode(eyebrow, tittel, sub){
    return [el("p", {cls:"eyebrow", text: eyebrow}), el("h2", {text: tittel}), sub ? el("p", {cls:"sub", text: sub}) : null];
  }
  function metaRad(meta){
    return el("div", {cls:"metarad"}, meta.map(function(m){
      return el("span", {title: m.label + ": " + m.verdi}, m.label + ": ", el("b", {text: String(m.verdi)}));
    }));
  }
  function kpiRad(kpier, vedKlikk){
    return el("div", {cls:"kpirad"}, kpier.map(function(k){
      var klikk = !!(k.tone && vedKlikk);
      var e = el(klikk ? "button" : "div", {cls:"kpi " + (k.tone || "") + (k.valgt ? " valgt" : ""), type: klikk ? "button" : null,
        title: klikk ? "Vis bare disse radene i tabellen under" : null,
        onclick: klikk ? function(){ vedKlikk(k.tone); } : null},
        el("div", {cls:"lab", text: k.label}),
        el("div", {cls:"tall num", text: nf.format(k.verdi)}),
        k.pst != null ? el("div", {cls:"pst", text: k.pst}) : null);
      return e;
    }));
  }
  function settNavtall(e, n){ e.textContent = nf.format(n); e.className = "navtall" + (n ? "" : " null"); }

  function antallAvvik(f){
    var n = 0;
    f.tabeller.forEach(function(t){
      if (/historikk$/.test(t.id)) return;
      var sc = t.kolonner.filter(function(c){ return c.type === "status"; })[0];
      if (sc) t.rader.forEach(function(r){ if (radTone(sc, r) === "r") n++; });
    });
    return n;
  }

  function visFriSide(id){
    Object.keys(friSider).forEach(function(k){ friSider[k].hidden = k !== id; });
    var p = friSider[id];
    if (!p){
      p = friSider[id] = el("div", {cls:"fri-side bred", id:"side-" + id});
      $("frisider").appendChild(p);
      p._bygget = false;
    }
    p.hidden = false;
    /* Sider med data som kan endres (innliming) tegnes pa nytt hver gang */
    if (!p._bygget || id === "oversikt" || id === "oversiktforhandler" || id === "analyse"){
      p.textContent = "";
      if (id === "analyse") grafPaneler = [];
      BYGGERE[id](p);
      p._bygget = true;
    }
    requestAnimationFrame(tegnSynligeGrafer);
  }

  var BYGGERE = {};
  var solgteTabell = null, solgtePanel = null, innlimtTeller = null;

  /* --- Oversikt --- */
  BYGGERE.oversikt = function(p){
    leggTil(p, sidehode("Oversikt", "OFV-kontrollene",
      "Samlet resultat for kontrollene som er kjørt i Databricks for " + D.selskap + ". Klikk deg inn på en kontroll for detaljer, filtrering og eksport, eller se Analyse for nøkkeltall og grafer."));
    p.appendChild(el("div", {cls:"kjoreinfo"},
      el("span", null, "Generert: ", el("b", {text: D.generert})),
      el("span", null, "Tidsbruk: ", el("b", {text: nf.format(D.tidsbruk) + " s"})),
      el("span", null, "OFV-oppslag: ", el("b", {text: nf.format(D.api.ofvKjoretoy) + " kjøretøy"}),
        " (" + nf.format(D.api.ofvBuffer) + " gjenbrukt, " + nf.format(D.api.ofvHttp) + " HTTP-kall)"),
      el("span", null, "SVV-kall: ", el("b", {text: nf.format(D.api.svvHttp)}))));
    var grid = el("div", {cls:"oversiktgrid"});
    D.oversikt.forEach(function(o){
      var kpier = o.kpier;
      if (o.id === "solgte") kpier = solgteKpier(alleSolgte(), false);
      grid.appendChild(el("div", {cls:"ovkort"},
        el("h3", {text: o.tittel}),
        metaRad(o.meta.filter(function(m){ return m.label !== "Kildefil"; })),
        el("div", {cls:"minirad"}, kpier.map(function(k){
          return el("div", {cls:"mini " + (k.tone || "")}, el("div", {cls:"l", text: k.label}), el("div", {cls:"v num", text: nf.format(k.verdi)}));
        })),
        el("button", {cls:"knapp liten", type:"button", text:"Åpne kontrollen →", onclick:function(){ gaa(sideIndeks(o.id)); }})));
    });
    p.appendChild(grid);
    p.appendChild(tabellPanel(D.input, true));
  };

  /* --- Per forhandler (fra presentasjonen) --- */
  BYGGERE.oversiktforhandler = function(p){
    leggTil(p, sidehode("Oversikt", "Per forhandler",
      "Antall biler og fordeling av dagers avvik for hvert datasett i Kontroll solgte biler – Databricks-kjøringen og datasett limt inn via tannhjulet. Klikk en rad for å filtrere Solgte biler til akkurat den forhandleren."));
    var sett = datasett();
    if (!sett.length){
      p.appendChild(el("div", {cls:"tomvarsel", text:"Ingen data lastet inn ennå. Bruk tannhjulet nederst til høyre for å laste inn ett eller flere datasett."}));
      return;
    }
    function soyle(t){
      var tot = t.total || 1;
      return el("div", {cls:"minisoyle"}, ["g", "y", "r", "n"].map(function(k){
        return el("span", {cls:k, style:"width:" + (t[k] / tot * 100) + "%"});
      }));
    }
    var kropp = el("tbody");
    var sum = {g:0, y:0, r:0, n:0, total:0};
    sett.forEach(function(d){
      var t = telling(d.rader);
      ["g", "y", "r", "n", "total"].forEach(function(k){ sum[k] += t[k]; });
      kropp.appendChild(el("tr", {cls:"rad", title:"Vis " + d.navn + " i Solgte biler", onclick:function(){
        gaa(sideIndeks("solgte"));
        if (solgteTabell) solgteTabell.settFilter("Forhandler", {eq: "=" + d.navn});
      }},
        el("td", null, el("b", {text: d.navn}), el("span", {style:"color:var(--dempet);font-size:12px;margin-left:8px", text: d.kilde})),
        el("td", {cls:"num", text: nf.format(t.total)}), el("td", null, soyle(t)),
        el("td", {cls:"num", text: nf.format(t.g)}), el("td", {cls:"num", text: nf.format(t.y)}),
        el("td", {cls:"num", text: nf.format(t.r)}), el("td", {cls:"num", text: nf.format(t.n)})));
    });
    kropp.appendChild(el("tr", {cls:"totalrad"}, el("td", {text:"Totalt"}), el("td", {cls:"num", text: nf.format(sum.total)}), el("td", null, soyle(sum)),
      el("td", {cls:"num", text: nf.format(sum.g)}), el("td", {cls:"num", text: nf.format(sum.y)}),
      el("td", {cls:"num", text: nf.format(sum.r)}), el("td", {cls:"num", text: nf.format(sum.n)})));
    var g = D.grenser;
    p.appendChild(el("table", {cls:"forhandlertabell"},
      el("thead", null, el("tr", null, ["Forhandler", "Antall biler", "Fordeling", "0–" + g.gronnMaks + " d", (g.gronnMaks + 1) + "–" + g.gulMaks + " d", (g.gulMaks + 1) + "+ d / selvh.", "Ikke kontr."].map(function(h){ return el("th", {text:h}); }))),
      kropp));
  };

  /* --- Kontrollsider (Solgte biler, Varekjop, Demobil) --- */
  function solgteKpier(rader, pst){
    var t = telling(rader), g = D.grenser;
    function p(x){ return pst ? (t.total ? Math.round(x / t.total * 100) + " %" : "0 %") : null; }
    return [
      {label:"Biler totalt", verdi: t.total, tone:""},
      {label: "0–" + g.gronnMaks + " dager", verdi: t.g, tone:"g", pst: p(t.g)},
      {label: (g.gronnMaks + 1) + "–" + g.gulMaks + " dager", verdi: t.y, tone:"y", pst: p(t.y)},
      {label: (g.gulMaks + 1) + "+ dager / selvhandel", verdi: t.r, tone:"r", pst: p(t.r)},
      {label:"Ikke kontrollert", verdi: t.n, tone:"n", pst: p(t.n)}
    ];
  }

  function byggKontrollside(f){
    return function(p){
      var erSolgte = f.id === "solgte";
      var eyebrow = erSolgte ? "Resultatet" : "Kontroll";
      var meta = f.meta.slice();
      if (erSolgte){
        var innlimt = SLOTER.filter(function(s){ return s.rader.length; }).length;
        meta.push({label:"Innlimte datasett", verdi: String(innlimt)});
      }
      leggTil(p, sidehode(eyebrow, f.fullTittel, erSolgte ?
        "Databricks-kjøringen og eventuelle innlimte datasett, samlet. Klikk en kolonneoverskrift for å sortere, bruk feltene under overskriftene for å filtrere." : null));
      var mr = metaRad(meta);
      p.appendChild(mr);
      if (erSolgte) innlimtTeller = mr.lastChild.querySelector("b");
      p.appendChild(el("details", {cls:"regel", open:true}, el("summary", {text:"Kontrollregel"}), el("p", {text: f.regel})));

      var kpiHolder = el("div");
      p.appendChild(kpiHolder);
      var paneler = [];
      var forsteValg = {vedEndring: function(t){
        var kpier = erSolgte ? solgteKpier(t._base, true) : f.kpier;
        var toner = t.state.toner, bare = Object.keys(toner).filter(function(k){ return toner[k]; });
        kpier.forEach(function(k){ k.valgt = bare.length === 1 && k.tone === bare[0]; });
        kpiHolder.textContent = "";
        kpiHolder.appendChild(kpiRad(kpier, function(tone){
          t.visBareTone(tone);
          paneler[0].scrollIntoView({behavior: REDUSERT ? "auto" : "smooth", block:"start"});
        }));
        if (paneler[0] && paneler[0]._ant) paneler[0]._ant.textContent = "(" + nf.format(t.rows.length) + ")";
      }};
      f.tabeller.forEach(function(t, i){
        var spec = t;
        if (erSolgte && i === 0) spec = Object.assign({}, t, {rader: alleSolgte()});
        var panel = tabellPanel(spec, /historikk$/.test(t.id), i === 0 ? forsteValg : null);
        paneler.push(panel);
        p.appendChild(panel);
      });
      if (erSolgte){
        solgtePanel = paneler[0];
        solgteTabell = paneler[0]._tabell;
        if (!SOLGTE_DB.length && !alleSolgte().length)
          paneler[0].insertBefore(el("div", {cls:"tomvarsel", style:"margin:0 18px 16px", text:"Kontroll solgte biler ble ikke kjørt i Databricks for dette selskapet, og ingen datasett er limt inn ennå. Bruk tannhjulet nederst til høyre for å lime inn data."}), paneler[0].children[1]);
      }
    };
  }
  D.faner.forEach(function(f){ BYGGERE[f.id] = byggKontrollside(f); });

  /* --- Analyse --- */
  BYGGERE.analyse = function(p){
    leggTil(p, sidehode("Analyse", "Nøkkeltall og grafer",
      "Fordelinger på tvers av kontrollene. Hold musen over en graf for eksakte tall, eller åpne «Vis tallene» under hver graf."));
    var solgte = alleSolgte(), ts = telling(solgte);
    var avvik = ts.r;
    var biler = solgte.length;
    D.faner.forEach(function(f){
      if (f.id === "solgte") return;
      avvik += antallAvvik(f);
      biler += f.antallBiler;
    });
    p.appendChild(kpiRad([
      {label:"Biler kontrollert", verdi: biler},
      {label:"Unike kjøretøy i OFV", verdi: D.analyse.unikeKjoretoy},
      {label:"Unike eierskifter", verdi: D.analyse.unikeEierskifter},
      {label:"Avvik totalt", verdi: avvik, tone:"r"}
    ], null));

    var grafer = D.analyse.grafer.map(function(g){ return JSON.parse(JSON.stringify(g)); });
    var status = grafer.filter(function(g){ return g.type === "stack"; })[0];
    if (!status){
      status = {type:"stack", tittel:"Status per kontroll", undertittel:"Andel biler per utfall - hold musen over for antall", rows:[], bred:true};
      grafer.unshift(status);
    }
    var nye = [];
    if (solgte.length){
      status.rows.unshift({label:"Solgte biler", segments:[
        {label:"OK", value: ts.g, tone:"g"}, {label:"Varsel", value: ts.y, tone:"y"},
        {label:"Avvik", value: ts.r, tone:"r"}, {label:"Ikke kontrollert", value: ts.n, tone:"n"}]});
      var dager = solgte.filter(function(r){ return r.Kontrollert === "Ja" && typeof r.DagerAvvik === "number"; }).map(function(r){ return r.DagerAvvik; });
      if (dager.length){
        var g = D.grenser;
        var bokser = [["0-" + g.gronnMaks, 0, g.gronnMaks, "g"], [(g.gronnMaks + 1) + "-7", g.gronnMaks + 1, 7, "y"], ["8-" + g.gulMaks, 8, g.gulMaks, "y"],
          [(g.gulMaks + 1) + "-30", g.gulMaks + 1, 30, "r"], ["31-90", 31, 90, "r"], ["91-365", 91, 365, "r"], ["> 1 år", 366, 1e9, "r"]];
        nye.push({type:"columns", tittel:"Solgte biler: dager avvik", undertittel:"Antall kontrollerte biler per avviksintervall (dager), alle datasett", statusLegend:true,
          data: bokser.map(function(b){ return {label:b[0], tone:b[3], value: dager.filter(function(d){ return d >= b[1] && d <= b[2]; }).length}; })});
      }
      var per = datasett();
      if (per.length > 1){
        nye.push({type:"stack", tittel:"Solgte biler per forhandler", undertittel:"Andel biler per utfall for hvert datasett", bred:true,
          rows: per.map(function(d){ var t = telling(d.rader); return {label:d.navn, segments:[
            {label:"OK", value:t.g, tone:"g"}, {label:"Varsel", value:t.y, tone:"y"}, {label:"Avvik", value:t.r, tone:"r"}, {label:"Ikke kontrollert", value:t.n, tone:"n"}]}; })});
      }
      var kilder = {};
      solgte.forEach(function(r){ var k = r.Kilde || "Ukjent"; kilder[k] = (kilder[k] || 0) + 1; });
      nye.push({type:"hbar", tittel:"Solgte biler: kilde for kontrolldato", undertittel:"OFV-eierskifte, SVV-førstegangsregistrering eller ingen treff",
        data: Object.keys(kilder).sort(function(a, b){ return kilder[b] - kilder[a]; }).map(function(k){ return {label:k, value:kilder[k]}; })});
    }
    var idx = grafer.indexOf(status) + 1;
    grafer.splice.apply(grafer, [idx, 0].concat(nye));
    if (!status.rows.length) grafer.splice(grafer.indexOf(status), 1);

    var grid = el("div", {cls:"grafgrid"});
    grafer.forEach(function(g){ grid.appendChild(grafPanel(g)); });
    p.appendChild(grid);
  };

  /* --- Toppmeny-tall (avvik per kontroll) --- */
  Array.prototype.forEach.call($("rapportnav").children, function(b){
    var id = b.getAttribute("data-side");
    var f = D.faner.filter(function(x){ return x.id === id; })[0];
    if (!f) return;
    var t = el("span", {cls:"navtall", id:"navtall-" + id, title:"Antall avvik"});
    settNavtall(t, id === "solgte" ? telling(alleSolgte()).r : antallAvvik(f));
    b.appendChild(t);
  });
  $("toppSelskap").textContent = D.selskap + " · " + D.generert;
  $("fotnote").textContent = "Kilder: OFV Transactions API og Statens vegvesen (reserve). Rapporten er generert av 02_OFV_API_Kontroller i Databricks " + D.generert + ".";

  /* --------------------------- bilder (forside / spørsmål) --------------------------- */

  function settBakgrunnsbilde(elId, dataUrl){
    var el = $(elId);
    if (dataUrl){ el.style.backgroundImage = "url(" + dataUrl + ")"; }
    else { el.style.backgroundImage = ""; }
  }

  function settOppBildeInput(inputId, forhandsvisId, fjernId, nokkel, helsideId){
    var lagretBilde = last(nokkel);
    if (lagretBilde){
      settBakgrunnsbilde(helsideId, lagretBilde);
      $(forhandsvisId).src = lagretBilde;
      $(forhandsvisId).hidden = false;
    }
    $(inputId).addEventListener("change", function(e){
      var fil = e.target.files && e.target.files[0];
      if (!fil) return;
      var leser = new FileReader();
      leser.onload = function(ev){
        var url = ev.target.result;
        lagre(nokkel, url);
        settBakgrunnsbilde(helsideId, url);
        $(forhandsvisId).src = url;
        $(forhandsvisId).hidden = false;
      };
      leser.readAsDataURL(fil);
    });
    $(fjernId).addEventListener("click", function(){
      slett(nokkel);
      settBakgrunnsbilde(helsideId, null);
      $(inputId).value = "";
      $(forhandsvisId).hidden = true;
    });
  }

  settOppBildeInput("bildeForside", "forhandsvisningForside", "fjernBildeForside", NOKKEL_BILDE_FORSIDE, "forsideHelside");
  settOppBildeInput("bildeSporsmal", "forhandsvisningSporsmal", "fjernBildeSporsmal", NOKKEL_BILDE_SPORSMAL, "sporsmalHelside");

  function settHovedbokLogo(dataUrl){
    for (var i = 0; i < 4; i++){
      var el = $("hovedbokLogo" + i);
      if (!el) continue;
      if (dataUrl){ el.src = dataUrl; el.hidden = false; }
      else { el.removeAttribute("src"); el.hidden = true; }
    }
  }

  (function settOppHovedbokLogoInput(){
    var lagretLogo = last(NOKKEL_LOGO_HOVEDBOK);
    if (lagretLogo){
      settHovedbokLogo(lagretLogo);
      $("forhandsvisningHovedbokLogo").src = lagretLogo;
      $("forhandsvisningHovedbokLogo").hidden = false;
    }
    $("bildeHovedbokLogo").addEventListener("change", function(e){
      var fil = e.target.files && e.target.files[0];
      if (!fil) return;
      var leser = new FileReader();
      leser.onload = function(ev){
        var url = ev.target.result;
        lagre(NOKKEL_LOGO_HOVEDBOK, url);
        settHovedbokLogo(url);
        $("forhandsvisningHovedbokLogo").src = url;
        $("forhandsvisningHovedbokLogo").hidden = false;
      };
      leser.readAsDataURL(fil);
    });
    $("fjernBildeHovedbokLogo").addEventListener("click", function(){
      slett(NOKKEL_LOGO_HOVEDBOK);
      settHovedbokLogo(null);
      $("bildeHovedbokLogo").value = "";
      $("forhandsvisningHovedbokLogo").hidden = true;
    });
  })();

  /* --------------------------- admin-panel (tannhjul) --------------------------- */

  function apneAdmin(){ $("adminBakteppe").hidden = false; document.body.style.overflow = "hidden"; }
  function lukkAdmin(){ $("adminBakteppe").hidden = true; document.body.style.overflow = ""; }

  $("tannhjul").addEventListener("click", apneAdmin);
  $("adminLukk").addEventListener("click", lukkAdmin);
  $("adminBakteppe").addEventListener("click", function(e){ if (e.target === $("adminBakteppe")) lukkAdmin(); });
  document.addEventListener("keydown", function(e){ if (e.key === "Escape" && !$("adminBakteppe").hidden) lukkAdmin(); });
  if (location.hash === "#rediger") apneAdmin();

  /* --------------------------- gjenopprett siste gang --------------------------- */

  (function gjenopprett(){
    var lagretSloter = last(NOKKEL_SLOTER);
    if (lagretSloter){
      try {
        var data = JSON.parse(lagretSloter);
        if (data && data.length){
          SLOTER = data;
          slotTeller = SLOTER.length;
          tegnSloter();
        }
      } catch (e) {}
    }
    if (!SLOTER.length) leggTilSlot();
    tegnAlt();
  })();

  window.addEventListener("beforeprint", function(){
    SIDER.forEach(function(s){ if (s.vert === "fri" && !(friSider[s.id] && friSider[s.id]._bygget)) visFriSide(s.id); });
    Object.keys(friSider).forEach(function(k){ friSider[k].hidden = false; });
    $("frihost").hidden = false;
  });
  window.addEventListener("afterprint", function(){ tegnSide(); });

  var startSide = sideIndeks(location.hash.slice(1));
  gaa(startSide >= 0 ? startSide : 0);

})();
</script>
</body>
</html>
"""


skriv_html_rapport(_bundles, OUTPUT_HTML)
print(f"HTML-rapport lagret: {OUTPUT_HTML}")

In [ ]:
# ==================================================================
# CELLE 6: OPPSUMMERING
# ==================================================================

print("=" * 70)
print("OFV-KONTROLLER - OPPSUMMERING")
print("=" * 70)
print(f"Selskap: {SELSKAP}")
print(f"Tidsbruk: {_elapsed_s:.1f} sekunder")
print(f"OFV: {API_TELLER['ofv_kjoretoy']} kjoretoy slatt opp, "
      f"{API_TELLER['ofv_buffer']} gjenbrukt fra buffer, "
      f"{API_TELLER['ofv_http']} HTTP-kall. SVV: {API_TELLER['svv_http']} HTTP-kall.\n")

for _navn, _kjort in KJOR_KONTROLL.items():
    if not _kjort:
        print(f"--- {_navn}: HOPPET OVER (inputfil mangler) ---\n")

for bundle in _bundles:
    print(f"--- {bundle['kontroll']} ---")
    print(f"  Kildefil: {bundle['kildefil']}")
    print(f"  Orgnr: {bundle['orgnr'] or '(ikke oppgitt)'}")
    if bundle["dato_fra"] and bundle["dato_til"]:
        print(f"  Periode: {bundle['dato_fra']:%d.%m.%Y} - {bundle['dato_til']:%d.%m.%Y}")
    if bundle["kontroll"] in TIDSBRUK_PER_KONTROLL:
        print(f"  Tidsbruk: {TIDSBRUK_PER_KONTROLL[bundle['kontroll']]:.1f} s")
    for key, value in bundle["stats"].items():
        print(f"  {key}: {value}")
    print()

print(f"Excel-arbeidsbok: {OUTPUT_XLSX}")
print(f"HTML-rapport: {OUTPUT_HTML}")